# MODEL 1: LOGISTIC REGRESSION

In [ ]:
# Moving on from feature engineering to machine learning.

import pandas as pd
from pathlib import Path

In [ ]:
PROCESSED_DIR = Path("../data/processed")

FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df = pd.read_csv(FEATURE_FILE)

print(model_df.shape)
model_df.head()

In [ ]:
# Recreate the 42-feature list

engineered_features = [
    col for col in model_df.columns
    if (
        col.startswith("home_avg_")
        or col.startswith("away_avg_")
        or col.startswith("home_points_")
        or col.startswith("away_points_")
        or col.startswith("home_win_rate_")
        or col.startswith("away_win_rate_")
        or col.startswith("home_goal_difference_")
        or col.startswith("away_goal_difference_")
        or col.startswith("home_xg_difference_")
        or col.startswith("away_xg_difference_")
        or col in ["home_days_rest", "away_days_rest"]
    )
]

print("Number of model features:", len(engineered_features))

In [ ]:
# Recreate the clean modeling dataset

model_ready_df = model_df.dropna(
    subset = engineered_features + ["target"]
).copy()

print("Model-ready matches:", len(model_ready_df))

In [ ]:
# Rebuild the chronological train/test split

train_df = model_ready_df[
    model_ready_df["season"].isin(["2023_24", "2024_25"])
].copy()

test_df = model_ready_df[
    model_ready_df["season"] == "2025_26"
].copy()

print("Training matches:", len(train_df))
print("Testing matches:", len(test_df))

In [ ]:
# X: What the model knows before the game (features)
# y: what actually happened (home win, draw, away win)

# 42 pre-match features from the 612 training matches.
# Info the model learns from.
X_train = train_df[engineered_features]

# Actual result of those 612 matches (0 home, 1 draw, 2 away)
# What the model learns to predict.
y_train = train_df["target"]

# Same 42 features, but for the 357 matches from the test set (season 2025/26)
# Model hasn't trained on this.
X_test = test_df[engineered_features]

# What actually happened in the 357 matches.
# Hide this from the model and later compare its predictions against it.
y_test = test_df["target"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
# Training our first baseline model.
# Beginning with a multinomial log. reg.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# Use feature scaling since all of the features are on different scaled.
# Possession: 30-70 % normally
# Passes: 300-700+
# etc.

scaler = StandardScaler()

# Same information from X_train, but standardized for the model to understand.
# fit_transform is only used on X_train because the scaler learns its averages only from the training seasons
# We don't want information from our 2025/26 future test set to influence preprocessing.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train the model

log_model = LogisticRegression(
    max_iter = 2000
)

log_model.fit(X_train_scaled, y_train)

In [ ]:
# Make predictions on the unseen 2025/26 matches:
# Training on 2023/24 + 2024/25
# Hasn't seen outcomes from 2025/26
# Asking the model to predict the 357 matches from the testing set using only information available before each match.
y_pred = log_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)

print("Test accuracy:", accuracy)

In [ ]:
# Check to see what the model is actually predicting

from sklearn.metrics import classification_report, confusion_matrix

print(
    classification_report(
        y_test,
        y_pred,
        target_names = ["Home Win", "Draw", "Away Win"]
    )
)

In [ ]:
# Create a confusion matrix

confusion_matrix(
    y_test,
    y_pred
)

# For 99 draws, it's only getting 7 correct.

In [ ]:
# Move on from hard predictions to probabilities.

y_proba = log_model.predict_proba(X_test_scaled)

y_proba[:5]

In [ ]:
# Make the table readable.

probability_df = test_df[
    [
        "date",
        "home_team",
        "away_team",
        "winner"
    ]
].copy()

probability_df["home_probability"] = y_proba[:, 0]
probability_df["draw_probability"] = y_proba[:, 1]
probability_df["away_probability"] = y_proba[:, 2]

probability_df[
    [
        "home_team",
        "away_team",
        "home_probability",
        "draw_probability",
        "away_probability",
        "winner"
    ]
].head(10)

In [ ]:
# Evaluate the probabilites
# Accuracy only checks whether the highest probability was correct.
# This project is alsoo about whether the probabiltiies themselves are sensible
# A standard metric to use is log loss, the lower the better and penalizes a model when it's very confident and wrong.

from sklearn.metrics import log_loss

logloss = log_loss(
    y_test,
    y_proba
)

print("Log loss:", logloss)

In [ ]:
# Estable a simple historical-frequency baseline
# Training data tells us how often home wins and away wins occurec historically.
# A simple model could ignore all 42 features and predict those same probabilities for every match.

class_probs = (
    y_train
    .value_counts(normalize = True)
    .sort_index()
)

print(class_probs)

In [ ]:
import numpy as np

baseline_proba = np.tile(
    class_probs.values,
    (len(y_test), 1)
)

baseline_logloss = log_loss(
    y_test,
    baseline_proba
)

print("Historical frequency baseline log loss:", baseline_logloss)
print("Our logistic regression log loss:", logloss)

In [ ]:
# Let's compare our baseline model to Bet365, and call it a benchmark

test_df[
    [
        "home_odds",
        "draw_odds",
        "away_odds"
    ]
].head()

In [ ]:
# Convert Bet365 to raw implied probabilities.

test_df["market_home_raw"] = 1 / test_df["home_odds"]
test_df["market_draw_raw"] = 1 / test_df["draw_odds"]
test_df["market_away_raw"] = 1 / test_df["away_odds"]

# Calculate total implied probability

test_df["market_total"] = (
    test_df["market_home_raw"]
    + test_df["market_draw_raw"]
    + test_df["market_away_raw"]
)

# Remove bookmaker margin (including this would add up to more than 100%)

test_df["market_home_prob"] = (
    test_df["market_home_raw"] / test_df["market_total"]
)

test_df["market_draw_prob"] = (
    test_df["market_draw_raw"] / test_df["market_total"]
)

test_df["market_away_prob"] = (
    test_df["market_away_raw"] / test_df["market_total"]
)

In [ ]:
# Inspect

test_df[
    [
        "home_team",
        "away_team",
        "home_odds",
        "draw_odds",
        "away_odds",
        "market_home_prob",
        "market_draw_prob",
        "market_away_prob"
    ]
].head(10)

In [ ]:
# Calculate Bet365 log loss

market_proba = test_df[
    [
        "market_home_prob",
        "market_draw_prob",
        "market_away_prob"
    ]
].values

In [ ]:
market_logloss = log_loss(
    y_test,
    market_proba
)

print("Historical baseline:", baseline_logloss)
print("Logistic regression:", logloss)
print("Bet365 market:", market_logloss)

# Bet365 market model had better performance and less logloss compared to the baseline model
# At this point we move on to model #2 to reduce the gap in logloss.

# MODEL 2: RANDOM FOREST

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train the RF model using the same train/test split

rf_model = RandomForestClassifier(
    n_estimators = 500,
    random_state = 42,
    n_jobs = -1
)

rf_model.fit(X_train, y_train)

# No need to use scaled X_train/X_test sets

In [ ]:
# Make predictions and probabilities for RF model.

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)

In [ ]:
# Evaluate the model

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_logloss = log_loss(y_test, rf_proba)

print("Random Forest accuracy:", rf_accuracy)
print("Random Forest log loss:", rf_logloss)

In [ ]:
# Check what the model is predicting

print(
    classification_report(
        y_test,
        rf_pred,
        target_names = ["Home Win", "Draw", "Away Win"]
    )
)

In [ ]:
confusion_matrix(
    y_test,
    rf_pred
)

# This model has learned well to distinguish home strength vs away strength
# But doesn't understand what makes a match likely to draw/

# MODEL 3: GRADIENT BOOSTING

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
# Train the GB model on the same training data

gb_model = GradientBoostingClassifier(
    random_state = 42
)

gb_model.fit(X_train, y_train)

In [ ]:
# Generate predictions and probabilities for the GB model

gb_pred = gb_model.predict(X_test)
gb_proba = gb_model.predict_proba(X_test)

In [ ]:
# Evaluate the GB model

gb_accuracy = accuracy_score(
    y_test,
    gb_pred
)

gb_logloss = log_loss(
    y_test,
    gb_proba
)

print("Gradient Boosting accuracy:", gb_accuracy)
print("Gradient Boosting log loss:", gb_logloss)

# Gradient Boosting is a more advanced model, but still performed worse than the other models.

In [ ]:
# Inspect RF feature importance
# Check what the best model is actually using.
# Used to expose weak or redundant features

feature_importance = pd.DataFrame({
    "feature": engineered_features,
    "importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending = False
)

feature_importance.head(15)

# Ranking the 42 features by how much the RF model relied on them

# MODEL TRAINING PART 2

In [ ]:
# Reload the updated file and add the six new features to the original 42

import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

model_df = pd.read_csv(
    PROCESSED_DIR / "model_features.csv"
)

print(model_df.shape)

In [ ]:
# Rebuild the feature list

engineered_features = [
    col for col in model_df.columns
    if (
        col.startswith("home_avg_")
        or col.startswith("away_avg_")
        or col.startswith("home_points_")
        or col.startswith("away_points_")
        or col.startswith("home_win_rate_")
        or col.startswith("away_win_rate_")
        or col.startswith("home_goal_difference_")
        or col.startswith("away_goal_difference_")
        or col.startswith("home_xg_difference_")
        or col.startswith("away_xg_difference_")
        or col in ["home_days_rest", "away_days_rest"]
    )
]

differential_features = [
    "xg_form_difference",
    "points_form_difference",
    "goal_difference_form_difference",
    "xg_difference_form_difference",
    "shots_on_target_form_difference",
    "win_rate_difference"
]

engineered_features_v2 = engineered_features + differential_features

print("V1 features:", len(engineered_features))
print("V2 features:", len(engineered_features_v2))

In [ ]:
# Rebuild the V2 modeling dataset

model_ready_df_v2 = model_df.dropna(
    subset = engineered_features_v2 + ["target"]
).copy()

print("Complete V2 matches:", len(model_ready_df_v2))

print(
    model_ready_df_v2["season"]
    .value_counts()
    .sort_index()
)

In [ ]:
# Build same chronological split as V1

train_df_v2 = model_ready_df_v2[
    model_ready_df_v2["season"].isin(["2023_24", "2024_25"])
].copy()

test_df_v2 = model_ready_df_v2[
    model_ready_df_v2["season"] == "2025_26"
].copy()

X_train_v2 = train_df_v2[engineered_features_v2]
y_train_v2 = train_df_v2["target"]

X_test_v2 = test_df_v2[engineered_features_v2]
y_test_v2 = test_df_v2["target"]

print("X_train:", X_train_v2.shape)
print("y_train:", y_train_v2.shape)
print("X_test:", X_test_v2.shape)
print("y_test:", y_test_v2.shape)

In [ ]:
# Training V2 Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix

rf_model_v2 = RandomForestClassifier(
    n_estimators = 500,
    random_state = 42,
    n_jobs = -1
)

rf_model_v2.fit(X_train_v2, y_train_v2)

rf_pred_v2 = rf_model_v2.predict(X_test_v2)
rf_proba_v2 = rf_model_v2.predict_proba(X_test_v2)

rf_accuracy_v2 = accuracy_score(y_test_v2, rf_pred_v2)
rf_logloss_v2 = log_loss(y_test_v2, rf_proba_v2)

print("Random Forest V2 Accuracy:", rf_accuracy_v2)
print("Random Forest V2 Log Loss:", rf_logloss_v2)

print("\nClassification Report:")
print(classification_report(y_test_v2, rf_pred_v2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_v2, rf_pred_v2))

In [ ]:
# There is a class imbalance with not many predictions going towards draws.
# Train a class-weighted RF model

rf_model_weighted = RandomForestClassifier(
    n_estimators = 500,
    random_state = 42,
    n_jobs = -1,

    # Gives more important to the less common classes during training 
    # In this case, draws.
    class_weight = "balanced"
)

rf_model_weighted.fit(X_train_v2, y_train_v2)

rf_pred_weighted = rf_model_weighted.predict(X_test_v2)
rf_proba_weighted = rf_model_weighted.predict_proba(X_test_v2)

rf_accuracy_weighted = accuracy_score(
    y_test_v2,
    rf_pred_weighted
)

rf_logloss_weighted = log_loss(
    y_test_v2,
    rf_proba_weighted
)

print("Weighted RF Accuracy:", rf_accuracy_weighted)
print("Weighted RF Log Loss:", rf_logloss_weighted)

print("\nClassification Report:")
print(classification_report(y_test_v2, rf_pred_weighted))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_v2, rf_pred_weighted))

# MODEL TRAINING PART 3

In [ ]:
model_df = pd.read_csv(
    PROCESSED_DIR / "model_features.csv"
)

print(model_df.shape)

In [ ]:
# Adding ELO features to existing V2 list

elo_features = [
    "home_elo",
    "away_elo",
    "elo_difference"
]

engineered_features_v3 = (
    engineered_features_v2
    + elo_features
)

print("V2 features:", len(engineered_features_v2))
print("V3 features:", len(engineered_features_v3))

In [ ]:
# Create V3 using the same train/test splits

model_ready_df_v3 = model_df.dropna(
    subset=engineered_features_v3 + ["target"]
).copy()

train_df_v3 = model_ready_df_v3[
    model_ready_df_v3["season"].isin(["2023_24", "2024_25"])
].copy()

test_df_v3 = model_ready_df_v3[
    model_ready_df_v3["season"] == "2025_26"
].copy()

X_train_v3 = train_df_v3[engineered_features_v3]
y_train_v3 = train_df_v3["target"]

X_test_v3 = test_df_v3[engineered_features_v3]
y_test_v3 = test_df_v3["target"]

print("Complete V3 matches:", len(model_ready_df_v3))
print("X_train:", X_train_v3.shape)
print("y_train:", y_train_v3.shape)
print("X_test:", X_test_v3.shape)
print("y_test:", y_test_v3.shape)

In [ ]:
# Train RF V3 with same settings as V1 and V2

rf_model_v3 = RandomForestClassifier(
    n_estimators = 500,
    random_state = 42,
    n_jobs = -1
)

rf_model_v3.fit(X_train_v3, y_train_v3)

rf_pred_v3 = rf_model_v3.predict(X_test_v3)
rf_proba_v3 = rf_model_v3.predict_proba(X_test_v3)

rf_accuracy_v3 = accuracy_score(
    y_test_v3,
    rf_pred_v3
)

rf_logloss_v3 = log_loss(
    y_test_v3,
    rf_proba_v3
)

print("Random Forest V3 Accuracy:", rf_accuracy_v3)
print("Random Forest V3 Log Loss:", rf_logloss_v3)

print("\nClassification Report:")
print(classification_report(y_test_v3, rf_pred_v3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_v3, rf_pred_v3))

In [ ]:
# Check if RF is actually using ELO

feature_importance_v3 = pd.DataFrame({
    "feature": engineered_features_v3,
    "importance": rf_model_v3.feature_importances_
}).sort_values(
    "importance",
    ascending = False
)

feature_importance_v3.head(20)

# MODEL TRAINING PART 4

In [ ]:
model_df = pd.read_csv(
    PROCESSED_DIR / "model_features.csv"
)

print(model_df.shape)

In [ ]:
# Create V4 feature list

elo_features_v4 = [
    "home_elo_v4",
    "away_elo_v4",
    "elo_difference_v4"
]

engineered_features_v4 = (
    engineered_features_v2
    + elo_features_v4
)

print("V2 features:", len(engineered_features_v2))
print("V4 features:", len(engineered_features_v4))

In [ ]:
# Rebuild V4 modeling dataset with same chronological train/test split

model_ready_df_v4 = model_df.dropna(
    subset=engineered_features_v4 + ["target"]
).copy()

train_df_v4 = model_ready_df_v4[
    model_ready_df_v4["season"].isin(["2023_24", "2024_25"])
].copy()

test_df_v4 = model_ready_df_v4[
    model_ready_df_v4["season"] == "2025_26"
].copy()

X_train_v4 = train_df_v4[engineered_features_v4]
y_train_v4 = train_df_v4["target"]

X_test_v4 = test_df_v4[engineered_features_v4]
y_test_v4 = test_df_v4["target"]

print("Complete V4 matches:", len(model_ready_df_v4))
print("X_train:", X_train_v4.shape)
print("y_train:", y_train_v4.shape)
print("X_test:", X_test_v4.shape)
print("y_test:", y_test_v4.shape)

In [ ]:
# Train V4 with same RF settings

rf_model_v4 = RandomForestClassifier(
    n_estimators = 500,
    random_state = 42,
    n_jobs = -1
)

rf_model_v4.fit(X_train_v4, y_train_v4)

rf_pred_v4 = rf_model_v4.predict(X_test_v4)
rf_proba_v4 = rf_model_v4.predict_proba(X_test_v4)

rf_accuracy_v4 = accuracy_score(
    y_test_v4,
    rf_pred_v4
)

rf_logloss_v4 = log_loss(
    y_test_v4,
    rf_proba_v4
)

print("Random Forest V4 Accuracy:", rf_accuracy_v4)
print("Random Forest V4 Log Loss:", rf_logloss_v4)

print("\nClassificatin Report:")
print(classification_report(y_test_v4, rf_pred_v4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_v4, rf_pred_v4))

# MODEL TRAINING PART 5

In [ ]:
model_df = pd.read_csv(
    PROCESSED_DIR / "model_features.csv"
)

print(model_df.shape)

In [ ]:
# V4b feature list

elo_features_v4b = [
    "home_elo_v4b",
    "away_elo_v4b",
    "elo_difference_v4b"
]

engineered_features_v4b = (
    engineered_features_v2
    + elo_features_v4b
)

print("V2 features:", len(engineered_features_v2))
print("V4b features:", len(engineered_features_v4b))

In [ ]:
# Build the same chronological train/test split for V4b

model_ready_df_v4b = model_df.dropna(
    subset=engineered_features_v4b + ["target"]
).copy()

train_df_v4b = model_ready_df_v4b[
    model_ready_df_v4b["season"].isin(["2023_24", "2024_25"])
].copy()

test_df_v4b = model_ready_df_v4b[
    model_ready_df_v4b["season"] == "2025_26"
].copy()

X_train_v4b = train_df_v4b[engineered_features_v4b]
y_train_v4b = train_df_v4b["target"]

X_test_v4b = test_df_v4b[engineered_features_v4b]
y_test_v4b = test_df_v4b["target"]

print("Complete V4b matches:", len(model_ready_df_v4b))
print("X_train:", X_train_v4b.shape)
print("y_train:", y_train_v4b.shape)
print("X_test:", X_test_v4b.shape)
print("y_test:", y_test_v4b.shape)

In [ ]:
# Train V4b using same RF settings

rf_model_v4b = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf_model_v4b.fit(X_train_v4b, y_train_v4b)

rf_pred_v4b = rf_model_v4b.predict(X_test_v4b)
rf_proba_v4b = rf_model_v4b.predict_proba(X_test_v4b)

rf_accuracy_v4b = accuracy_score(
    y_test_v4b,
    rf_pred_v4b
)

rf_logloss_v4b = log_loss(
    y_test_v4b,
    rf_proba_v4b
)

print("Random Forest V4b Accuracy:", rf_accuracy_v4b)
print("Random Forest V4b Log Loss:", rf_logloss_v4b)

print("\nClassification Report:")
print(classification_report(y_test_v4b, rf_pred_v4b))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_v4b, rf_pred_v4b))

# MODEL TRAINING PART 6

In [ ]:
# Same process as before, using different configuration

model_df = pd.read_csv(
    PROCESSED_DIR / "model_features.csv"
)

print(model_df.shape)

In [ ]:
elo_features_v4c = [
    "home_elo_v4c",
    "away_elo_v4c",
    "elo_difference_v4c"
]

engineered_features_v4c = (
    engineered_features_v2
    + elo_features_v4c
)

print("V2 features:", len(engineered_features_v2))
print("V4c features:", len(engineered_features_v4c))

In [ ]:
# Build the same chronological train/test split for V4c

model_ready_df_v4c = model_df.dropna(
    subset=engineered_features_v4c + ["target"]
).copy()

train_df_v4c = model_ready_df_v4c[
    model_ready_df_v4c["season"].isin(["2023_24", "2024_25"])
].copy()

test_df_v4c = model_ready_df_v4c[
    model_ready_df_v4c["season"] == "2025_26"
].copy()

X_train_v4c = train_df_v4c[engineered_features_v4c]
y_train_v4c = train_df_v4c["target"]

X_test_v4c = test_df_v4c[engineered_features_v4c]
y_test_v4c = test_df_v4c["target"]

print("Complete V4c matches:", len(model_ready_df_v4c))
print("X_train:", X_train_v4c.shape)
print("y_train:", y_train_v4c.shape)
print("X_test:", X_test_v4c.shape)
print("y_test:", y_test_v4c.shape)

In [ ]:
rf_model_v4c = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf_model_v4c.fit(X_train_v4c, y_train_v4c)

rf_pred_v4c = rf_model_v4c.predict(X_test_v4c)
rf_proba_v4c = rf_model_v4c.predict_proba(X_test_v4c)

rf_accuracy_v4c = accuracy_score(
    y_test_v4c,
    rf_pred_v4c
)

rf_logloss_v4c = log_loss(
    y_test_v4c,
    rf_proba_v4c
)

print("Random Forest V4c Accuracy:", rf_accuracy_v4c)
print("Random Forest V4c Log Loss:", rf_logloss_v4c)

print("\nClassification Report:")
print(classification_report(y_test_v4c, rf_pred_v4c))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_v4c, rf_pred_v4c))

In [ ]:
# When the model says something has a 60% chance of happening, does it actually happen roughly 60% of the time?
# Looking to see if our model is good, overconfident, or underconfident

from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

class_names = {
    0: "Home Win",
    1: "Draw",
    2: "Away Win"
}

plt.figure(figsize=(8, 6))

for class_idx, class_name in class_names.items():

    # Convert the actual result into yes/no for this class
    y_binary = (y_test_v4b.to_numpy() == class_idx).astype(int)

    prob_true, prob_pred = calibration_curve(
        y_binary,
        rf_proba_v4b[:, class_idx],
        n_bins=6,
        strategy="quantile"
    )

    plt.plot(
        prob_pred,
        prob_true,
        marker="o",
        label=class_name
    )

# Perfectly calibrated reference line
plt.plot(
    [0, 1],
    [0, 1],
    "--",
    label="Perfect Calibration"
)

plt.xlabel("Model Predicted Probability")
plt.ylabel("Actual Frequency")
plt.title("Random Forest V4b — Probability Calibration")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# Perform a Brier scoring test to measure probability alongside log loss
from sklearn.preprocessing import label_binarize
import numpy as np

y_test_onehot_v4b = label_binarize(
    y_test_v4b,
    classes = [0, 1, 2]
)

brier_v4b = np.mean(
    np.sum(
        (rf_proba_v4b - y_test_onehot_v4b) ** 2,
        axis = 1
    )
)

print("V4b Multiclass Brier Score:", brier_v4b)

In [ ]:
print(market_proba.shape)

In [ ]:
# Calculate Brier score for Bet365 fair probabilities.

brier_bet365 = np.mean(
    np.sum(
        (market_proba - y_test_onehot_v4b) ** 2,
        axis = 1
    )
)

print("Bet365 Multiclass Brier Score:", brier_bet365)


In [ ]:
model_ready_df_v4b.groupby("season").size()

In [ ]:
# Create a train/test split for calibration
# Train on 2023_24
# Calibrate using 24/25
# Test on 25/26

train_cal_df = model_ready_df_v4b[
    model_ready_df_v4b["season"] == "2023_24"
].copy()

calibration_df = model_ready_df_v4b[
    model_ready_df_v4b["season"] == "2024_25"
].copy()

final_test_df = model_ready_df_v4b[
    model_ready_df_v4b["season"] == "2025_26"
].copy()


X_train_cal = train_cal_df[engineered_features_v4b]
y_train_cal = train_cal_df["target"]

X_calibration = calibration_df[engineered_features_v4b]
y_calibration = calibration_df["target"]

X_final_test = final_test_df[engineered_features_v4b]
y_final_test = final_test_df["target"]


print("RF training:", X_train_cal.shape)
print("Calibration:", X_calibration.shape)
print("Final test:", X_final_test.shape)

In [ ]:
# Train a new RF model only on 2023/24
# Can't be an existing model because it has already seen 2024/25 info during training

rf_cal_base = RandomForestClassifier(
    n_estimators = 500,
    random_state = 42,
    n_jobs = -1
)

rf_cal_base.fit(
    X_train_cal,
    y_train_cal
)

# Get uncalibrated probabilites on the 2024/25 calibration season
calibration_proba = rf_cal_base.predict_proba(
    X_calibration
)

print("Calibration probability shape:", calibration_proba.shape)
print("First 5 probability predictions:")
print(calibration_proba[:5])

In [ ]:
# Perform sigmoid calibration
# Sigmoid calibration is less prone to overfitting with limited calibration data (347 matches)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

In [ ]:
calibrated_rf = CalibratedClassifierCV(
    FrozenEstimator(rf_cal_base),
    method = "sigmoid"
)

calibrated_rf.fit(
    X_calibration,
    y_calibration
)

print("Sigmoid calibration fitted successfully.")

In [ ]:
# Compare calibrated to uncalibrated models

# Calibrated probabilities on the untouched 2025/26 season
calibrated_proba = calibrated_rf.predict_proba(
    X_final_test
)

calibrated_logloss = log_loss(
    y_final_test,
    calibrated_proba
)

# Converting results to one-hot format for Brier scoring
y_final_onehot = label_binarize(
    y_final_test,
    classes = [0, 1, 2]
)

# Calculating multiclass Brier score
calibrated_brier = np.mean(
    np.sum(
        (calibrated_proba - y_final_onehot) **2,
        axis = 1
    )
)

print("Sigmoid Calibrated RF Log Loss:", calibrated_logloss)
print("Sigmoid Calibrated RF Brier Score:", calibrated_brier)

In [ ]:
# Graph the calibrated probabilities using calibration chart to see what the improved scores are telling us

from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

class_names = {
    0: "Home Win",
    1: "Draw",
    2: "Away Win"
}

plt.figure(figsize = (8, 6))

for class_idx, class_name in class_names.items():
    y_binary = (y_final_test.to_numpy() == class_idx).astype(int)

    prob_true, prob_pred = calibration_curve(
        y_binary,
        calibrated_proba[:, class_idx],
        n_bins = 6,
        strategy = "quantile"
    )

    plt.plot(
        prob_pred,
        prob_true,
        marker = "o",
        label = class_name
    )

plt.plot(
    [0, 1],
    [0, 1],
    "--",
    label = "Perfect Calibration"
)

plt.xlabel("Model Predicted Probability")
plt.ylabel("Actual Frequency")
plt.title("Sigmoid Calibrated Rf - Probability Calibration")
plt.legend()
plt.grid(alpha = 0.3)

plt.show()

# Sigmoid calibration has compressed the probabilities towards the middle.

In [ ]:
# Try calibrating again, but with 24/25 being split chronologically

season_2324 = model_ready_df_v4b[
    model_ready_df_v4b["season"] == "2023_24"
].sort_values("date").copy()

season_2425 = model_ready_df_v4b[
    model_ready_df_v4b["season"] == "2024_25"
].sort_values("date").copy()

season_2526 = model_ready_df_v4b[
    model_ready_df_v4b["season"] == "2025_26"
].sort_values("date").copy()


split_index = int(len(season_2425) * 0.70)

season_2425_train = season_2425.iloc[:split_index].copy()
season_2425_cal = season_2425.iloc[split_index:].copy()


expanded_train_df = pd.concat(
    [season_2324, season_2425_train],
    ignore_index=True
)

expanded_calibration_df = season_2425_cal.copy()
expanded_test_df = season_2526.copy()


print("RF training matches:", len(expanded_train_df))
print("Calibration matches:", len(expanded_calibration_df))
print("Final test matches:", len(expanded_test_df))

print("\nLatest RF training date:")
print(expanded_train_df["date"].max())

print("\nEarliest calibration date:")
print(expanded_calibration_df["date"].min())

# Increased RF training from 265 matches to 507 matches

In [ ]:
# Train the expanded RF

X_expanded_train = expanded_train_df[engineered_features_v4b]
y_expanded_train = expanded_train_df["target"]

X_expanded_cal = expanded_calibration_df[engineered_features_v4b]
y_expanded_cal = expanded_calibration_df["target"]

X_expanded_test = expanded_test_df[engineered_features_v4b]
y_expanded_test = expanded_test_df["target"]


rf_expanded = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf_expanded.fit(
    X_expanded_train,
    y_expanded_train
)

print("Expanded RF trained successfully.")
print("Training shape:", X_expanded_train.shape)
print("Calibration shape:", X_expanded_cal.shape)
print("Test shape:", X_expanded_test.shape)

In [ ]:
# Calibrate the 507-match RF using only the later 105 matches

calibrated_rf_expanded = CalibratedClassifierCV(
    FrozenEstimator(rf_expanded),
    method = "sigmoid"
)

calibrated_rf_expanded.fit(
    X_expanded_cal,
    y_expanded_cal
)

print("Expanded sigmoid calibration fitted successfully.")

In [ ]:
# Evaluate on the 357 matches from 2025/26

expanded_calibrated_proba = calibrated_rf_expanded.predict_proba(
    X_expanded_test
)

# Log loss
expanded_calibrated_logloss = log_loss(
    y_expanded_test,
    expanded_calibrated_proba
)

# One-hot actual outcomes
y_expanded_test_onehot = label_binarize(
    y_expanded_test,
    classes=[0, 1, 2]
)

# Multiclass Brier score
expanded_calibrated_brier = np.mean(
    np.sum(
        (expanded_calibrated_proba - y_expanded_test_onehot) ** 2,
        axis=1
    )
)

print("Expanded Calibrated RF Log Loss:", expanded_calibrated_logloss)
print("Expanded Calibrated RF Brier Score:", expanded_calibrated_brier)

In [ ]:
# Compare the uncalibrated 507 match RF against its calibrated version
# Important because right now we're comparing the expanded calibrated model to the original 612 match V4b
# Isolate what sigmoid calibration did to the same 507 match model

expanded_uncalibrated_proba = rf_expanded.predict_proba(
    X_expanded_test
)

expanded_uncalibrated_logloss = log_loss(
    y_expanded_test,
    expanded_uncalibrated_proba
)

expanded_uncalibrated_brier = np.mean(
    np.sum(
        (expanded_uncalibrated_proba - y_expanded_test_onehot) ** 2,
        axis=1
    )
)

print("Uncalibrated Expanded RF Log Loss:", expanded_uncalibrated_logloss)
print("Uncalibrated Expanded RF Brier Score:", expanded_uncalibrated_brier)

print("\nAfter Sigmoid Calibration:")
print("Log Loss:", expanded_calibrated_logloss)
print("Brier Score:", expanded_calibrated_brier)

# PERFORM CHRONOLOGICAL BACKTEST
Past Matches Only
     |
Train Model
     |
Calibrate probabilities using recent historical matches
     |
Predict the next group of matches
     |
Those matches become history
     |
Retrain
     |
Predict again
     |
Repeat through 2025/26

In [ ]:
# Inspect 2025/26 matchdays

backtest_2526 = model_ready_df_v4b[
    model_ready_df_v4b["season"] == "2025_26"
].copy()

backtest_2526 = backtest_2526.sort_values(
    ["matchday", "date"]
)

matchday_counts = (
    backtest_2526
    .groupby("matchday")
    .size()
)

print("2025/26 usable matches:", len(backtest_2526))
print("Number of matchdays:", backtest_2526["matchday"].nunique())

print("\nMatches per matchday:")
print(matchday_counts)

In [ ]:
# Set up historical starting pool
# We want backtest to begin 2025/26 by knowing only what would have been known before that season

historical_pool = model_ready_df_v4b[
    model_ready_df_v4b["season"].isin(["2023_24", "2024_25"])
].copy()

historical_pool = historical_pool.sort_values("date")

print("Starting historical matches:", len(historical_pool))
print("First historical match:", historical_pool["date"].min())
print("Last historical match:", historical_pool["date"].max())

print("\n2025/26 backtest matches:", len(backtest_2526))
print("First backtest match:", backtest_2526["date"].min())

# At matchday 1, model can use 2023/24 + 2024/25
# After matchday 1, those results become available
# Matchday 2 can use 2023/24 + 2024/25 + 2025/26 matchday
# Then matchday 3 gets MW1-2 and so on
# Prevents futre info from leaking backward

In [ ]:
# Build walk-forward loop
backtest_results = []

training_pool = historical_pool.copy()

for matchday in sorted(backtest_2526["matchday"].unique()):

    # Matches we are predicting this week
    matchday_df = backtest_2526[
        backtest_2526["matchday"] == matchday
    ].copy()

    # Training data = everything available BEFORE this matchday
    X_train_bt = training_pool[engineered_features_v4b]
    y_train_bt = training_pool["target"]

    # Train a fresh RF
    rf_bt = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    rf_bt.fit(
        X_train_bt,
        y_train_bt
    )

    # Predict this matchday
    X_matchday = matchday_df[engineered_features_v4b]

    matchday_proba = rf_bt.predict_proba(
        X_matchday
    )

    # Store each prediction
    for i, (_, match) in enumerate(matchday_df.iterrows()):

        backtest_results.append({
            "match_id": match["match_id"],
            "date": match["date"],
            "matchday": matchday,
            "home_team": match["home_team"],
            "away_team": match["away_team"],
            "actual_result": match["target"],

            "home_prob": matchday_proba[i, 0],
            "draw_prob": matchday_proba[i, 1],
            "away_prob": matchday_proba[i, 2],

            "training_matches": len(training_pool)
        })

    # ONLY AFTER predictions are made,
    # add this matchday to the historical training pool
    training_pool = pd.concat(
        [training_pool, matchday_df],
        ignore_index=True
    )


backtest_results_df = pd.DataFrame(
    backtest_results
)

print("Backtest predictions:", len(backtest_results_df))

print(
    backtest_results_df[
        [
            "matchday",
            "home_team",
            "away_team",
            "home_prob",
            "draw_prob",
            "away_prob",
            "training_matches"
        ]
    ].head(15)
)

In [ ]:
# Score the backtest
# Calculate accuracy, log loss, and Brier score across 357 predictions
# Asking what if my system actually existed during thr 2025/26 season and retrained after each completed matchday, how would it perform?

# Actual results
y_backtest = backtest_results_df["actual_result"].astype(int)

# Predicted probabilities
backtest_proba = backtest_results_df[
    [
        "home_prob",
        "draw_prob",
        "away_prob"
    ]
].values

# Predicted class = highest probability
backtest_pred = np.argmax(
    backtest_proba,
    axis=1
)

# Accuracy
backtest_accuracy = accuracy_score(
    y_backtest,
    backtest_pred
)

# Log loss
backtest_logloss = log_loss(
    y_backtest,
    backtest_proba,
    labels=[0, 1, 2]
)

# One-hot actual results
y_backtest_onehot = label_binarize(
    y_backtest,
    classes=[0, 1, 2]
)

# Multiclass Brier score
backtest_brier = np.mean(
    np.sum(
        (backtest_proba - y_backtest_onehot) ** 2,
        axis=1
    )
)

print("Walk-Forward RF Accuracy:", backtest_accuracy)
print("Walk-Forward RF Log Loss:", backtest_logloss)
print("Walk-Forward RF Brier Score:", backtest_brier)

In [ ]:
# Walk-forward method performed worse
# Try walk-forward sigmoid calibration

calibrated_backtest_results = []

# Start with only matches known before 2025/26
training_pool_cal = historical_pool.copy()

CALIBRATION_WINDOW = 100

for matchday in sorted(backtest_2526["matchday"].unique()):

    # ---------------------------------------
    # 1. Matches we need to predict
    # ---------------------------------------
    matchday_df = backtest_2526[
        backtest_2526["matchday"] == matchday
    ].copy()

    # Keep everything chronological
    training_pool_cal = training_pool_cal.sort_values("date").reset_index(drop=True)

    # ---------------------------------------
    # 2. Split available history
    # ---------------------------------------
    rf_train_df = training_pool_cal.iloc[:-CALIBRATION_WINDOW].copy()

    calibration_window_df = training_pool_cal.iloc[-CALIBRATION_WINDOW:].copy()

    # ---------------------------------------
    # 3. Train Random Forest on older history
    # ---------------------------------------
    X_rf_train = rf_train_df[engineered_features_v4b]
    y_rf_train = rf_train_df["target"]

    rf_walk_cal = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    rf_walk_cal.fit(
        X_rf_train,
        y_rf_train
    )

    # ---------------------------------------
    # 4. Calibrate using recent history
    # ---------------------------------------
    X_cal_window = calibration_window_df[engineered_features_v4b]
    y_cal_window = calibration_window_df["target"]

    calibrated_walk_model = CalibratedClassifierCV(
        FrozenEstimator(rf_walk_cal),
        method="sigmoid"
    )

    calibrated_walk_model.fit(
        X_cal_window,
        y_cal_window
    )

    # ---------------------------------------
    # 5. Predict current matchday
    # ---------------------------------------
    X_matchday = matchday_df[engineered_features_v4b]

    matchday_proba = calibrated_walk_model.predict_proba(
        X_matchday
    )

    # ---------------------------------------
    # 6. Store predictions
    # ---------------------------------------
    for i, (_, match) in enumerate(matchday_df.iterrows()):

        calibrated_backtest_results.append({
            "match_id": match["match_id"],
            "date": match["date"],
            "matchday": matchday,
            "home_team": match["home_team"],
            "away_team": match["away_team"],
            "actual_result": match["target"],

            "home_prob": matchday_proba[i, 0],
            "draw_prob": matchday_proba[i, 1],
            "away_prob": matchday_proba[i, 2],

            "rf_training_matches": len(rf_train_df),
            "calibration_matches": len(calibration_window_df)
        })

    # ---------------------------------------
    # 7. NOW the matchday becomes history
    # ---------------------------------------
    training_pool_cal = pd.concat(
        [training_pool_cal, matchday_df],
        ignore_index=True
    )


calibrated_backtest_df = pd.DataFrame(
    calibrated_backtest_results
)

print("Calibrated backtest predictions:", len(calibrated_backtest_df))

print(
    calibrated_backtest_df[
        [
            "matchday",
            "home_team",
            "away_team",
            "rf_training_matches",
            "calibration_matches"
        ]
    ].head(15)
)

In [ ]:
# Score the calibrated walk-forward model

# Actual results
y_cal_backtest = calibrated_backtest_df[
    "actual_result"
].astype(int)

# Calibrated probabilities
cal_backtest_proba = calibrated_backtest_df[
    [
        "home_prob",
        "draw_prob",
        "away_prob"
    ]
].values

# Highest probability = predicted result
cal_backtest_pred = np.argmax(
    cal_backtest_proba,
    axis=1
)

# Accuracy
cal_backtest_accuracy = accuracy_score(
    y_cal_backtest,
    cal_backtest_pred
)

# Log loss
cal_backtest_logloss = log_loss(
    y_cal_backtest,
    cal_backtest_proba,
    labels=[0, 1, 2]
)

# Actual results to one-hot
y_cal_backtest_onehot = label_binarize(
    y_cal_backtest,
    classes=[0, 1, 2]
)

# Multiclass Brier
cal_backtest_brier = np.mean(
    np.sum(
        (cal_backtest_proba - y_cal_backtest_onehot) ** 2,
        axis=1
    )
)

print("Calibrated Walk-Forward Accuracy:", cal_backtest_accuracy)
print("Calibrated Walk-Forward Log Loss:", cal_backtest_logloss)
print("Calibrated Walk-Forward Brier Score:", cal_backtest_brier)

# MODEL VS MARKET ANALYSIS

In [ ]:
market_columns = test_df[
    [
        "match_id",
        "market_home_prob",
        "market_draw_prob",
        "market_away_prob"
    ]
].copy()

walk_forward_market_df = backtest_results_df.merge(
    market_columns,
    on="match_id",
    how="left"
)

print("Matches:", len(walk_forward_market_df))

print("\nMissing Bet365 probabilities:")
print(
    walk_forward_market_df[
        [
            "market_home_prob",
            "market_draw_prob",
            "market_away_prob"
        ]
    ].isna().sum()
)

In [ ]:
# We have a clean 357 match model vs market dataset
# Score Bet365 on the same walk-forward matches.

market_backtest_proba = walk_forward_market_df[
    [
        "market_home_prob",
        "market_draw_prob",
        "market_away_prob"
    ]
].values

y_market_backtest = walk_forward_market_df[
    "actual_result"
].astype(int)

# Bet365 predicted outcome
market_backtest_pred = np.argmax(
    market_backtest_proba,
    axis=1
)

# Accuracy
market_backtest_accuracy = accuracy_score(
    y_market_backtest,
    market_backtest_pred
)

# Log loss
market_backtest_logloss = log_loss(
    y_market_backtest,
    market_backtest_proba,
    labels=[0, 1, 2]
)

# One-hot actual outcomes
y_market_onehot = label_binarize(
    y_market_backtest,
    classes=[0, 1, 2]
)

# Brier score
market_backtest_brier = np.mean(
    np.sum(
        (market_backtest_proba - y_market_onehot) ** 2,
        axis=1
    )
)

print("Bet365 Accuracy:", market_backtest_accuracy)
print("Bet365 Log Loss:", market_backtest_logloss)
print("Bet365 Brier Score:", market_backtest_brier)

In [ ]:
# Calculate model vs market edge
# Use the uncalibrated walk-forward RF since it beat the rolling calibrated version

walk_forward_market_df["home_edge"] = (
    walk_forward_market_df["home_prob"]
    - walk_forward_market_df["market_home_prob"]
)

walk_forward_market_df["draw_edge"] = (
    walk_forward_market_df["draw_prob"]
    - walk_forward_market_df["market_draw_prob"]
)

walk_forward_market_df["away_edge"] = (
    walk_forward_market_df["away_prob"]
    - walk_forward_market_df["market_away_prob"]
)

walk_forward_market_df[
    [
        "home_team",
        "away_team",
        "home_prob",
        "market_home_prob",
        "home_edge",
        "draw_prob",
        "market_draw_prob",
        "draw_edge",
        "away_prob",
        "market_away_prob",
        "away_edge"
    ]
].head(10)

In [ ]:
edge_rows = []

for _, match in walk_forward_market_df.iterrows():

    actual_result = int(match["actual_result"])

    outcomes = [
        (
            0,
            "Home",
            match["home_prob"],
            match["market_home_prob"],
            match["home_edge"]
        ),
        (
            1,
            "Draw",
            match["draw_prob"],
            match["market_draw_prob"],
            match["draw_edge"]
        ),
        (
            2,
            "Away",
            match["away_prob"],
            match["market_away_prob"],
            match["away_edge"]
        )
    ]

    for outcome_code, outcome_name, model_prob, market_prob, edge in outcomes:

        edge_rows.append({
            "match_id": match["match_id"],
            "matchday": match["matchday"],
            "home_team": match["home_team"],
            "away_team": match["away_team"],
            "outcome": outcome_name,

            "model_prob": model_prob,
            "market_prob": market_prob,
            "edge": edge,

            # 1 if this outcome actually happened, otherwise 0
            "won": int(actual_result == outcome_code)
        })


edge_df = pd.DataFrame(edge_rows)

print("Total edge observations:", len(edge_df))

print(
    edge_df[
        [
            "home_team",
            "away_team",
            "outcome",
            "model_prob",
            "market_prob",
            "edge",
            "won"
        ]
    ].head(12)
)

In [ ]:
edge_df["edge_bin"] = pd.cut(
    edge_df["edge"],
    bins=[
        -float("inf"),
        -0.10,
        -0.05,
        0.00,
        0.05,
        0.10,
        0.15,
        float("inf")
    ],
    labels=[
        "< -10%",
        "-10% to -5%",
        "-5% to 0%",
        "0% to 5%",
        "5% to 10%",
        "10% to 15%",
        "15%+"
    ]
)

edge_summary = (
    edge_df
    .groupby("edge_bin", observed=False)
    .agg(
        observations=("won", "size"),
        actual_win_rate=("won", "mean"),
        avg_model_prob=("model_prob", "mean"),
        avg_market_prob=("market_prob", "mean"),
        avg_edge=("edge", "mean")
    )
    .reset_index()
)

edge_summary["actual_minus_market"] = (
    edge_summary["actual_win_rate"]
    - edge_summary["avg_market_prob"]
)

edge_summary["actual_minus_model"] = (
    edge_summary["actual_win_rate"]
    - edge_summary["avg_model_prob"]
)

edge_summary

In [ ]:
edge_by_outcome = (
    edge_df
    .groupby(
        ["outcome", "edge_bin"],
        observed=False
    )
    .agg(
        observations=("won", "size"),
        actual_win_rate=("won", "mean"),
        avg_model_prob=("model_prob", "mean"),
        avg_market_prob=("market_prob", "mean"),
        avg_edge=("edge", "mean")
    )
    .reset_index()
)

edge_by_outcome["actual_minus_market"] = (
    edge_by_outcome["actual_win_rate"]
    - edge_by_outcome["avg_market_prob"]
)

edge_by_outcome["actual_minus_model"] = (
    edge_by_outcome["actual_win_rate"]
    - edge_by_outcome["avg_model_prob"]
)

edge_by_outcome

In [ ]:
positive_edge_summary = edge_by_outcome[
    edge_by_outcome["edge_bin"].isin([
        "0% to 5%",
        "5% to 10%",
        "10% to 15%",
        "15%+"
    ])
][
    [
        "outcome",
        "edge_bin",
        "observations",
        "avg_market_prob",
        "avg_model_prob",
        "actual_win_rate",
        "actual_minus_market"
    ]
].copy()

print(
    positive_edge_summary.to_string(index=False)
)

# MODEL TRAINING WITH UPDATED MATCH DATASET

In [ ]:
print(model_df.columns.tolist())

In [ ]:
# Load the feature-engineered dataset

import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

model_df = pd.read_csv(
    PROCESSED_DIR / "pl_match_features.csv"
)

model_df["date_home"] = pd.to_datetime(
    model_df["date_home"],
    utc=True
)

print("Shape:", model_df.shape)
print()
print(model_df[[
    "date_home",
    "season_home",
    "team_home",
    "team_away",
    "target"
]].head())

In [ ]:
# Recreate the 31 baseline features

baseline_features = [
    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "shots_on_target_difference_last5_all_home",

    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    "days_rest_home",
    "days_rest_away",
    "matches_last_7_days_home",
    "matches_last_7_days_away",
    "matches_last_14_days_home",
    "matches_last_14_days_away",

    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]

print("Baseline features:", len(baseline_features))

missing_columns = [
    col for col in baseline_features
    if col not in model_df.columns
]

print("Missing columns:", missing_columns)

In [ ]:
# Start with training from 2020/21 all the wat to 2024/25
# Test on 2025/26

train_df = model_df[
    model_df["season_home"].isin([
        "2020_21",
        "2021_22",
        "2022_23",
        "2023_24",
        "2024_25"
    ])
].copy()

test_df = model_df[
    model_df["season_home"] == "2025_26"
].copy()

print("Training matches:", len(train_df))
print("Test matches:", len(test_df))

print()
print("Training seasons:")
print(train_df["season_home"].value_counts().sort_index())

print()
print("Test seasons:")
print(test_df["season_home"].value_counts().sort_index())

In [ ]:
# Separate the features (X) from the result (y)

X_train = train_df[baseline_features]
y_train = train_df["target"]

X_test = test_df[baseline_features]
y_test = test_df["target"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print()
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print()
print("Training target distribution:")
print(y_train.value_counts())

In [ ]:
# Build the first baseline model: Multinomial Log Reg
# Model looks at missing values, does scaling, and outputs its prediction model


from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    (
        "imputer",
        # Learns its median values from X_train only
        # Doesn't look at 2025/26 to decide how to fill anything
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000
        )
    )
])

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic regression trained successfully.")

In [ ]:
# Evaluate on completely unseen 2025/26 season

from sklearn.metrics import accuracy_score, log_loss

y_pred = logistic_model.predict(X_test)
y_prob = logistic_model.predict_proba(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

logloss = log_loss(
    y_test,
    y_prob,
    labels = logistic_model.classes_
)

print("Classes:", logistic_model.classes_)
print()
print(f"Accuracy: {accuracy:.4f}")
print(f"Log Loss: {logloss:.4f}")

In [ ]:
# Before trying Random Forest, compare this against a simple baseline:
# Always predict using the historical H/D/A frequencies from the training set

train_probabilities = (
    y_train.value_counts(normalize = True)
    .reindex(logistic_model.classes_)
    .values
)

baseline_probabilites = np.tile(
    train_probabilities,
    (len(y_test), 1)
)

baseline_logloss = log_loss(
    y_test,
    baseline_probabilites,
    labels = logistic_model.classes_
)

baseline_prediction = y_train.value_counts().idxmax()

baseline_accuracy = (
    y_test == baseline_prediction
).mean()

print("Historical probabilities:")
for label, probability in zip(
    logistic_model.classes_,
    train_probabilities
):
    print(f"{label}: {probability:.4f}")

print()
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print(f"Baseline Log Loss: {baseline_logloss:.4f}")

print()
print(f"Logistic Accuracy: {accuracy:.4f}")
print(f"Logistic Log Loss: {logloss:.4f}")

# This means the chosen features are adding predictive information.

In [ ]:
# Test a model that can capture more complicated relationships between features
# Random Forest model

from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy = "median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators = 500,
            max_depth = 8,
            min_samples_leaf = 5,
            random_state = 42,
            n_jobs = -1
        )
    )
])

random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully.")

In [ ]:
# Evaluate the Random Forest on same 380 unseen 2025/26 matches

rf_pred = random_forest_model.predict(X_test)
rf_prob = random_forest_model.predict_proba(X_test)

rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

rf_logloss = log_loss(
    y_test,
    rf_prob,
    labels = random_forest_model.classes_
)

print("Classes:", random_forest_model.classes_)

print()
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest Log Loss: {rf_logloss:.4f}")

print()
print("Comparison:")
print(f"Historical Baseline  | Accuracy: {baseline_accuracy:.4f} | Log Loss: {baseline_logloss:.4f}")
print(f"Logistic Regression | Accuracy: {accuracy:.4f} | Log Loss: {logloss:.4f}")
print(f"Random Forest       | Accuracy: {rf_accuracy:.4f} | Log Loss: {rf_logloss:.4f}")

In [ ]:
# RF performed better than the Log Reg and Baseline models
# Check to see what the RF is actually using to make its predictions

rf_classifier = random_forest_model.named_steps["model"]

feature_importance = pd.DataFrame({
    "feature": baseline_features,
    "importance": rf_classifier.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending = False
).reset_index(drop = True)

print(
    feature_importance
    .head(15)
    .to_string(index = False)
)

# team_elo_difference_home makes sense because the relative stength of the two teams should be highly predictive

In [ ]:
# Check where the RF model struggles

from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        rf_pred,
        labels=["H", "D", "A"]
    )
)

print()
print("Classification Report:")
print(
    classification_report(
        y_test,
        rf_pred,
        labels=["H", "D", "A"],
        digits=4
    )
)

In [ ]:
# Check if the model is giving draws a reasonable probability or just ignoring them

rf_probability_df = pd.DataFrame(
    rf_prob,
    columns=random_forest_model.classes_
)

print("Average predicted probabilities:")
print(rf_probability_df.mean())

print()
print("Highest draw probability:")
print(rf_probability_df["D"].max())

print()
print("Draw probability percentiles:")
print(
    rf_probability_df["D"].quantile([
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ])
)

In [ ]:
# Check draw calibration, see whether a predicted 28% draw actually occurs around 28% of the time
# Group matches based on the model's predicted draw probability and see how often they actually ended in draws

draw_check = pd.DataFrame({
    "draw_probability": rf_probability_df["D"],
    "actual_draw": (y_test.reset_index(drop=True) == "D").astype(int)
})

draw_check["probability_group"] = pd.qcut(
    draw_check["draw_probability"],
    q=5,
    duplicates="drop"
)

draw_calibration = (
    draw_check
    .groupby(
        "probability_group",
        observed=True
    )
    .agg(
        matches=("actual_draw", "size"),
        avg_predicted_draw_probability=("draw_probability", "mean"),
        actual_draw_rate=("actual_draw", "mean")
    )
)

print(draw_calibration)


In [ ]:
# Don't calibrate or tune yet, testing has only been on one season
# We don' want to accidentally fix the model specfically for the 2025/26 season and overfit the test set

# Calculate Brier score, gives another probability-quality metric

from sklearn.metrics import brier_score_loss

brier_scores = {}

for outcome in ["H", "D", "A"]:
    class_index = list(
        random_forest_model.classes_
    ).index(outcome)

    actual = (
        y_test == outcome
    ).astype(int)

    predicted = rf_prob[:, class_index]

    brier_scores[outcome] = brier_score_loss(
        actual,
        predicted
    )

multiclass_brier = sum(
    brier_scores.values()
)

print("Brier Score by outcome:")

for outcome, score in brier_scores.items():
    print(f"{outcome}: {score:.4f}")

print()
print(
    f"Multiclass Brier Score: "
    f"{multiclass_brier:.4f}"
)

In [ ]:
# Before improving the RF model, finish the comparsion between Log Reg and RF

logistic_brier_scores = {}

for outcome in ["H", "D", "A"]:
    class_index = list(
        logistic_model.classes_
    ).index(outcome)

    actual = (
        y_test == outcome
    ).astype(int)

    predicted = y_prob[:, class_index]

    logistic_brier_scores[outcome] = brier_score_loss(
        actual,
        predicted
    )

logistic_multiclass_brier = sum(
    logistic_brier_scores.values()
)

print("Logistic Brier Score by outcome:")

for outcome, score in logistic_brier_scores.items():
    print(f"{outcome}: {score:.4f}")

print()
print(
    f"Logistic Multiclass Brier Score: "
    f"{logistic_multiclass_brier:.4f}"
)

print()
print("Current comparison:")
print(
    f"Logistic Regression | "
    f"Accuracy: {accuracy:.4f} | "
    f"Log Loss: {logloss:.4f} | "
    f"Brier: {logistic_multiclass_brier:.4f}"
)
print(
    f"Random Forest       | "
    f"Accuracy: {rf_accuracy:.4f} | "
    f"Log Loss: {rf_logloss:.4f} | "
    f"Brier: {multiclass_brier:.4f}"
)

In [ ]:
# Make 2024/25 season a validation/model development season
# Don't tune RF model against 2025/26 repeatedly because then it slowly stops being a truly unseen test season

development_train_df = model_df[
    model_df["season_home"].isin([
        "2020_21",
        "2021_22",
        "2022_23",
        "2023_24"
    ])
].copy()

validation_df = model_df[
    model_df["season_home"] == "2024_25"
].copy()

final_test_df = model_df[
    model_df["season_home"] == "2025_26"
].copy()

print(
    "Development training matches:",
    len(development_train_df)
)

print(
    "Validation matches:",
    len(validation_df)
)

print(
    "Final test matches:",
    len(final_test_df)
)

In [ ]:
# Recreate the RF using only 2020/21 - 2023/24 for training and evaluate on 2024/25 for validation
# Basically asking, if we were sitting at the end of 2023/24, how well would this model have predicted the entire 2024/25 PL season?

X_dev_train = development_train_df[baseline_features]
y_dev_train = development_train_df["target"]

X_validation = validation_df[baseline_features]
y_validation = validation_df["target"]

rf_validation_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_validation_model.fit(
    X_dev_train,
    y_dev_train
)

validation_pred = rf_validation_model.predict(
    X_validation
)

validation_prob = rf_validation_model.predict_proba(
    X_validation
)

validation_accuracy = accuracy_score(
    y_validation,
    validation_pred
)

validation_logloss = log_loss(
    y_validation,
    validation_prob,
    labels=rf_validation_model.classes_
)

print(f"Validation Accuracy: {validation_accuracy:.4f}")
print(f"Validation Log Loss: {validation_logloss:.4f}")

In [ ]:
# Test to see if all-competition data is actually helping
# If model performance gets worse, there is direct evidence that the other competitions context is helping PL predictions

pl_only_features = [
    feature
    for feature in baseline_features
    if "_all_" not in feature
]

print("Full baseline features:", len(baseline_features))
print("Without all-comp form:", len(pl_only_features))

print()
print("Removed features:")

for feature in baseline_features:
    if feature not in pl_only_features:
        print(feature)

In [ ]:
# Perform a controlled test: same RF model, same training seasons and validation season, but without the 8 all-competition form features

X_dev_train_pl_only = development_train_df[
    pl_only_features
]

X_validation_pl_only = validation_df[
    pl_only_features
]

rf_pl_only_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_pl_only_model.fit(
    X_dev_train_pl_only,
    y_dev_train
)

pl_only_pred = rf_pl_only_model.predict(
    X_validation_pl_only
)

pl_only_prob = rf_pl_only_model.predict_proba(
    X_validation_pl_only
)

pl_only_accuracy = accuracy_score(
    y_validation,
    pl_only_pred
)

pl_only_logloss = log_loss(
    y_validation,
    pl_only_prob,
    labels=rf_pl_only_model.classes_
)

print("2024/25 Validation Comparison:")
print()

print(
    f"Full 31 features     | "
    f"Accuracy: {validation_accuracy:.4f} | "
    f"Log Loss: {validation_logloss:.4f}"
)

print(
    f"Without all-comp     | "
    f"Accuracy: {pl_only_accuracy:.4f} | "
    f"Log Loss: {pl_only_logloss:.4f}"
)

In [ ]:
# Test the value of ELO rating

no_elo_features = [
    feature
    for feature in pl_only_features
    if "elo" not in feature
]

print("Current features:", len(pl_only_features))
print("Without Elo:", len(no_elo_features))

print()
print("Removed features:")

for feature in pl_only_features:
    if feature not in no_elo_features:
        print(feature)

In [ ]:
# Same RF settings, same training/validation seasons

X_dev_train_no_elo = development_train_df[
    no_elo_features
]

X_validation_no_elo = validation_df[
    no_elo_features
]

rf_no_elo_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_no_elo_model.fit(
    X_dev_train_no_elo,
    y_dev_train
)

no_elo_pred = rf_no_elo_model.predict(
    X_validation_no_elo
)

no_elo_prob = rf_no_elo_model.predict_proba(
    X_validation_no_elo
)

no_elo_accuracy = accuracy_score(
    y_validation,
    no_elo_pred
)

no_elo_logloss = log_loss(
    y_validation,
    no_elo_prob,
    labels=rf_no_elo_model.classes_
)

print("2024/25 Elo Comparison:")
print()

print(
    f"With Elo    | "
    f"Accuracy: {pl_only_accuracy:.4f} | "
    f"Log Loss: {pl_only_logloss:.4f}"
)

print(
    f"Without Elo | "
    f"Accuracy: {no_elo_accuracy:.4f} | "
    f"Log Loss: {no_elo_logloss:.4f}"
)

In [ ]:
# Test the congestion/rest features

no_congestion_features = [
    feature
    for feature in pl_only_features
    if feature not in [
        "days_rest_home",
        "days_rest_away",
        "matches_last_7_days_home",
        "matches_last_7_days_away",
        "matches_last_14_days_home",
        "matches_last_14_days_away"
    ]
]

print("Current features:", len(pl_only_features))
print("Without rest/congestion:", len(no_congestion_features))

print()
print("Removed features:")

for feature in pl_only_features:
    if feature not in no_congestion_features:
        print(feature)

In [ ]:
# Run a controlled comparison again, same seasons, but removing congestion/rest features

X_dev_train_no_congestion = development_train_df[
    no_congestion_features
]

X_validation_no_congestion = validation_df[
    no_congestion_features
]

rf_no_congestion_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_no_congestion_model.fit(
    X_dev_train_no_congestion,
    y_dev_train
)

no_congestion_pred = rf_no_congestion_model.predict(
    X_validation_no_congestion
)

no_congestion_prob = rf_no_congestion_model.predict_proba(
    X_validation_no_congestion
)

no_congestion_accuracy = accuracy_score(
    y_validation,
    no_congestion_pred
)

no_congestion_logloss = log_loss(
    y_validation,
    no_congestion_prob,
    labels=rf_no_congestion_model.classes_
)

print("2024/25 Rest & Congestion Comparison:")
print()

print(
    f"With rest/congestion    | "
    f"Accuracy: {pl_only_accuracy:.4f} | "
    f"Log Loss: {pl_only_logloss:.4f}"
)

print(
    f"Without rest/congestion | "
    f"Accuracy: {no_congestion_accuracy:.4f} | "
    f"Log Loss: {no_congestion_logloss:.4f}"
)

In [ ]:
# Test redundancy in the ELO features
# Keeping only ELO difference

elo_difference_only_features = [
    feature
    for feature in pl_only_features
    if feature not in [
        "team_elo_home",
        "team_elo_away"
    ]
]

print("Current features:", len(pl_only_features))
print("Elo difference only:", len(elo_difference_only_features))

print()
print("Removed features:")

for feature in pl_only_features:
    if feature not in elo_difference_only_features:
        print(feature)

In [ ]:
# Test using same settings, just removing absolute home and away elo ratings

X_dev_train_elo_diff = development_train_df[
    elo_difference_only_features
]

X_validation_elo_diff = validation_df[
    elo_difference_only_features
]

rf_elo_diff_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_elo_diff_model.fit(
    X_dev_train_elo_diff,
    y_dev_train
)

elo_diff_pred = rf_elo_diff_model.predict(
    X_validation_elo_diff
)

elo_diff_prob = rf_elo_diff_model.predict_proba(
    X_validation_elo_diff
)

elo_diff_accuracy = accuracy_score(
    y_validation,
    elo_diff_pred
)

elo_diff_logloss = log_loss(
    y_validation,
    elo_diff_prob,
    labels=rf_elo_diff_model.classes_
)

print("2024/25 Elo Feature Comparison:")
print()

print(
    f"All 3 Elo features | "
    f"Accuracy: {pl_only_accuracy:.4f} | "
    f"Log Loss: {pl_only_logloss:.4f}"
)

print(
    f"Elo difference only | "
    f"Accuracy: {elo_diff_accuracy:.4f} | "
    f"Log Loss: {elo_diff_logloss:.4f}"
)

In [ ]:
# Test missing-value indicators 

rf_missing_indicator_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_missing_indicator_model.fit(
    development_train_df[elo_difference_only_features],
    y_dev_train
)

indicator_pred = rf_missing_indicator_model.predict(
    validation_df[elo_difference_only_features]
)

indicator_prob = rf_missing_indicator_model.predict_proba(
    validation_df[elo_difference_only_features]
)

indicator_accuracy = accuracy_score(
    y_validation,
    indicator_pred
)

indicator_logloss = log_loss(
    y_validation,
    indicator_prob,
    labels=rf_missing_indicator_model.classes_
)

print("2024/25 Missing Indicator Comparison:")
print()

print(
    f"Median only       | "
    f"Accuracy: {elo_diff_accuracy:.4f} | "
    f"Log Loss: {elo_diff_logloss:.4f}"
)

print(
    f"Median + indicator | "
    f"Accuracy: {indicator_accuracy:.4f} | "
    f"Log Loss: {indicator_logloss:.4f}"
)

In [ ]:
model_df = model_df.sort_values(
    ["date_home", "match_id"]
).reset_index(drop=True)

walk_forward_seasons = [
    "2021_22",
    "2022_23",
    "2023_24",
    "2024_25"
]

print(
    model_df[
        ["season_home", "date_home"]
    ]
    .groupby("season_home")
    .agg(["min", "max", "count"])
)

In [ ]:
# Build a walk-froward evaluation
# Test the current best setup: 21 features, median imputation + missing indicatiors, same RF settings

walk_forward_results = []

for validation_season in walk_forward_seasons:

    validation_start = model_df.loc[
        model_df["season_home"] == validation_season,
        "date_home"
    ].min()

    train_df = model_df[
        model_df["date_home"] < validation_start
    ]

    val_df = model_df[
        model_df["season_home"] == validation_season
    ]

    X_train_walk = train_df[
        elo_difference_only_features
    ]

    y_train_walk = train_df["target"]

    X_val_walk = val_df[
        elo_difference_only_features
    ]

    y_val_walk = val_df["target"]

    walk_model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    walk_model.fit(
        X_train_walk,
        y_train_walk
    )

    walk_pred = walk_model.predict(
        X_val_walk
    )

    walk_prob = walk_model.predict_proba(
        X_val_walk
    )

    season_accuracy = accuracy_score(
        y_val_walk,
        walk_pred
    )

    season_logloss = log_loss(
        y_val_walk,
        walk_prob,
        labels=walk_model.classes_
    )

    walk_forward_results.append({
        "season": validation_season,
        "train_matches": len(train_df),
        "validation_matches": len(val_df),
        "accuracy": season_accuracy,
        "log_loss": season_logloss
    })

walk_forward_results_df = pd.DataFrame(
    walk_forward_results
)

print(walk_forward_results_df.to_string(index=False))

In [ ]:
# Walk-forward the original model

baseline_walk_results = []

for validation_season in walk_forward_seasons:

    validation_start = model_df.loc[
        model_df["season_home"] == validation_season,
        "date_home"
    ].min()

    train_df = model_df[
        model_df["date_home"] < validation_start
    ]

    val_df = model_df[
        model_df["season_home"] == validation_season
    ]

    X_train_walk = train_df[
        baseline_features
    ]

    y_train_walk = train_df["target"]

    X_val_walk = val_df[
        baseline_features
    ]

    y_val_walk = val_df["target"]

    baseline_walk_model = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    baseline_walk_model.fit(
        X_train_walk,
        y_train_walk
    )

    baseline_pred = baseline_walk_model.predict(
        X_val_walk
    )

    baseline_prob = baseline_walk_model.predict_proba(
        X_val_walk
    )

    baseline_walk_results.append({
        "season": validation_season,
        "accuracy": accuracy_score(
            y_val_walk,
            baseline_pred
        ),
        "log_loss": log_loss(
            y_val_walk,
            baseline_prob,
            labels=baseline_walk_model.classes_
        )
    })

baseline_walk_df = pd.DataFrame(
    baseline_walk_results
)

comparison_df = baseline_walk_df.merge(
    walk_forward_results_df[
        ["season", "accuracy", "log_loss"]
    ],
    on="season",
    suffixes=("_original", "_current")
)

print(comparison_df.to_string(index=False))

In [ ]:
original_mean_accuracy = comparison_df[
    "accuracy_original"
].mean()

original_mean_logloss = comparison_df[
    "log_loss_original"
].mean()

current_mean_accuracy = comparison_df[
    "accuracy_current"
].mean()

current_mean_logloss = comparison_df[
    "log_loss_current"
].mean()

print("4-Season Walk-Forward Average:")
print()

print(
    f"Original 31-feature model | "
    f"Accuracy: {original_mean_accuracy:.4f} | "
    f"Log Loss: {original_mean_logloss:.4f}"
)

print(
    f"Current 21-feature model  | "
    f"Accuracy: {current_mean_accuracy:.4f} | "
    f"Log Loss: {current_mean_logloss:.4f}"
)

In [ ]:
baseline_indicator_walk_results = []

for validation_season in walk_forward_seasons:

    validation_start = model_df.loc[
        model_df["season_home"] == validation_season,
        "date_home"
    ].min()

    train_df = model_df[
        model_df["date_home"] < validation_start
    ]

    val_df = model_df[
        model_df["season_home"] == validation_season
    ]

    X_train_walk = train_df[baseline_features]
    y_train_walk = train_df["target"]

    X_val_walk = val_df[baseline_features]
    y_val_walk = val_df["target"]

    walk_model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    walk_model.fit(
        X_train_walk,
        y_train_walk
    )

    walk_pred = walk_model.predict(
        X_val_walk
    )

    walk_prob = walk_model.predict_proba(
        X_val_walk
    )

    baseline_indicator_walk_results.append({
        "season": validation_season,
        "accuracy": accuracy_score(
            y_val_walk,
            walk_pred
        ),
        "log_loss": log_loss(
            y_val_walk,
            walk_prob,
            labels=walk_model.classes_
        )
    })

baseline_indicator_walk_df = pd.DataFrame(
    baseline_indicator_walk_results
)

print(baseline_indicator_walk_df.to_string(index=False))

print()
print(
    "Average Accuracy:",
    round(
        baseline_indicator_walk_df["accuracy"].mean(),
        4
    )
)

print(
    "Average Log Loss:",
    round(
        baseline_indicator_walk_df["log_loss"].mean(),
        4
    )
)

In [ ]:
frequency_baseline_results = []

for validation_season in walk_forward_seasons:

    validation_start = model_df.loc[
        model_df["season_home"] == validation_season,
        "date_home"
    ].min()

    train_df = model_df[
        model_df["date_home"] < validation_start
    ]

    val_df = model_df[
        model_df["season_home"] == validation_season
    ]

    y_train_walk = train_df["target"]
    y_val_walk = val_df["target"]

    class_probs = (
        y_train_walk
        .value_counts(normalize=True)
        .reindex(["A", "D", "H"])
    )

    baseline_prob = np.tile(
        class_probs.values,
        (len(val_df), 1)
    )

    baseline_pred = np.repeat(
        class_probs.idxmax(),
        len(val_df)
    )

    frequency_baseline_results.append({
        "season": validation_season,
        "accuracy": accuracy_score(
            y_val_walk,
            baseline_pred
        ),
        "log_loss": log_loss(
            y_val_walk,
            baseline_prob,
            labels=["A", "D", "H"]
        )
    })

frequency_baseline_df = pd.DataFrame(
    frequency_baseline_results
)

print(frequency_baseline_df.to_string(index=False))

print()
print(
    "Average Accuracy:",
    round(
        frequency_baseline_df["accuracy"].mean(),
        4
    )
)

print(
    "Average Log Loss:",
    round(
        frequency_baseline_df["log_loss"].mean(),
        4
    )
)

In [ ]:
# Verify our log-loss calculations are correct

class_to_index = {
    class_label: i
    for i, class_label
    in enumerate(rf_missing_indicator_model.classes_)
}

true_class_probabilities = np.array([
    indicator_prob[
        row_index,
        class_to_index[actual_result]
    ]
    for row_index, actual_result
    in enumerate(y_validation)
])

manual_match_losses = -np.log(
    true_class_probabilities
)

manual_log_loss = manual_match_losses.mean()

sklearn_log_loss = log_loss(
    y_validation,
    indicator_prob,
    labels=rf_missing_indicator_model.classes_
)

print(
    "Manual Log Loss:",
    manual_log_loss
)

print(
    "Sklearn Log Loss:",
    sklearn_log_loss
)

print(
    "Difference:",
    abs(manual_log_loss - sklearn_log_loss)
)

print()
print(
    "Average probability assigned to actual result:",
    true_class_probabilities.mean()
)

In [ ]:
# Verify if the RF's improvement over the baseline numbers are statistically convincing

walk_predictions = []

for validation_season in walk_forward_seasons:

    validation_start = model_df.loc[
        model_df["season_home"] == validation_season,
        "date_home"
    ].min()

    train_df = model_df[
        model_df["date_home"] < validation_start
    ]

    val_df = model_df[
        model_df["season_home"] == validation_season
    ]

    X_train_walk = train_df[baseline_features]
    y_train_walk = train_df["target"]

    X_val_walk = val_df[baseline_features]
    y_val_walk = val_df["target"]

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    model.fit(
        X_train_walk,
        y_train_walk
    )

    probabilities = model.predict_proba(
        X_val_walk
    )

    for i in range(len(val_df)):
        walk_predictions.append({
            "season": validation_season,
            "actual": y_val_walk.iloc[i],
            "prob_A": probabilities[
                i,
                list(model.classes_).index("A")
            ],
            "prob_D": probabilities[
                i,
                list(model.classes_).index("D")
            ],
            "prob_H": probabilities[
                i,
                list(model.classes_).index("H")
            ]
        })

walk_predictions_df = pd.DataFrame(
    walk_predictions
)

print(walk_predictions_df.shape)
print()
print(walk_predictions_df.head())

In [ ]:
# Calculate each match's model loss

prob_columns = {
    "A": "prob_A",
    "D": "prob_D",
    "H": "prob_H"
}

walk_predictions_df["model_loss"] = [
    -np.log(
        row[prob_columns[row["actual"]]]
    )
    for _, row in walk_predictions_df.iterrows()
]

print(
    "Walk-forward model log loss:",
    walk_predictions_df["model_loss"].mean()
)

print()
print(
    walk_predictions_df["model_loss"].describe()
)

In [ ]:
baseline_losses = []

for validation_season in walk_forward_seasons:

    validation_start = model_df.loc[
        model_df["season_home"] == validation_season,
        "date_home"
    ].min()

    train_df = model_df[
        model_df["date_home"] < validation_start
    ]

    class_probs = (
        train_df["target"]
        .value_counts(normalize=True)
        .reindex(["A", "D", "H"])
    )

    season_predictions = walk_predictions_df[
        walk_predictions_df["season"] == validation_season
    ]

    for actual_result in season_predictions["actual"]:

        actual_probability = class_probs[
            actual_result
        ]

        baseline_losses.append(
            -np.log(actual_probability)
        )

walk_predictions_df["baseline_loss"] = baseline_losses

walk_predictions_df["loss_improvement"] = (
    walk_predictions_df["baseline_loss"]
    - walk_predictions_df["model_loss"]
)

print(
    "Model Log Loss:",
    walk_predictions_df["model_loss"].mean()
)

print(
    "Baseline Log Loss:",
    walk_predictions_df["baseline_loss"].mean()
)

print(
    "Model Improvement:",
    walk_predictions_df["loss_improvement"].mean()
)

In [ ]:
# Check to see if that 0.07 improvement is statistically convincing
# Boostrap a 95% confidence Interval
# Repeatedly resample the 1,520 match-level improvements and see how much the average improvement varies

rng = np.random.default_rng(42)

loss_improvements = (
    walk_predictions_df["loss_improvement"]
    .to_numpy()
)

n_bootstraps = 10000

bootstrap_improvements = []

for _ in range(n_bootstraps):

    bootstrap_sample = rng.choice(
        loss_improvements,
        size=len(loss_improvements),
        replace=True
    )

    bootstrap_improvements.append(
        bootstrap_sample.mean()
    )

bootstrap_improvements = np.array(
    bootstrap_improvements
)

lower_bound = np.percentile(
    bootstrap_improvements,
    2.5
)

upper_bound = np.percentile(
    bootstrap_improvements,
    97.5
)

print(
    "Observed Improvement:",
    loss_improvements.mean()
)

print(
    "95% Bootstrap CI:",
    (lower_bound, upper_bound)
)

print(
    "Probability improvement <= 0:",
    np.mean(
        bootstrap_improvements <= 0
    )
)

In [ ]:
# Match Bet365 to the 1,520 walk-forward predictions

# Load Bet365 odds
bet365_df = pd.read_csv(
    PROCESSED_DIR / "bet365_pl_odds.csv"
)

print(bet365_df.shape)
print(bet365_df.columns.tolist())

In [ ]:
walk_match_ids = []

for season in walk_forward_seasons:

    season_match_ids = (
        model_df[
            model_df["season_home"] == season
        ]
        .sort_values(
            ["date_home", "match_id"]
        )["match_id"]
        .tolist()
    )

    walk_match_ids.extend(
        season_match_ids
    )

print(
    "Prediction rows:",
    len(walk_predictions_df)
)

print(
    "Match IDs:",
    len(walk_match_ids)
)

In [ ]:
# Verify that the season ordering also lines up before attaching IDs

print("Prediction seasons:")
print(
    walk_predictions_df["season"]
    .value_counts(sort=False)
)

print()

print("Match ID seasons:")
print(
    model_df[
        model_df["match_id"].isin(walk_match_ids)
    ]["season_home"]
    .value_counts(sort=False)
)

In [ ]:
# Attach IDs to walk_predictions_df

walk_predictions_df = (
    walk_predictions_df
    .copy()
)

walk_predictions_df["match_id"] = (
    walk_match_ids
)

print(
    walk_predictions_df[
        [
            "match_id",
            "season",
            "actual",
            "prob_H",
            "prob_D",
            "prob_A"
        ]
    ].head()
)

print()

print(
    "Duplicate match IDs:",
    walk_predictions_df["match_id"]
    .duplicated()
    .sum()
)

In [ ]:
# Merge Bet365 onto the exact 1,520 walk-forward matches

walk_market_df = (
    walk_predictions_df
    .merge(
        bet365_df[
            [
                "match_id",
                "fair_prob_H",
                "fair_prob_D",
                "fair_prob_A"
            ]
        ],
        on="match_id",
        how="left"
    )
)

print(walk_market_df.shape)

print()

print(
    walk_market_df[
        [
            "match_id",
            "season",
            "actual",
            "prob_H",
            "prob_D",
            "prob_A",
            "fair_prob_H",
            "fair_prob_D",
            "fair_prob_A"
        ]
    ].head()
)

print()

print(
    "Missing Bet365 rows:",
    walk_market_df[
        [
            "fair_prob_H",
            "fair_prob_D",
            "fair_prob_A"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)

In [ ]:
# Compare our model's log loss to Bet365 market log loss

from sklearn.metrics import log_loss

model_probs = walk_market_df[
    ["prob_A", "prob_D", "prob_H"]
].to_numpy()

bet365_probs = walk_market_df[
    ["fair_prob_A", "fair_prob_D", "fair_prob_H"]
].to_numpy()

actual_results = walk_market_df[
    "actual"
]

model_log_loss = log_loss(
    actual_results,
    model_probs,
    labels=["A", "D", "H"]
)

bet365_log_loss = log_loss(
    actual_results,
    bet365_probs,
    labels=["A", "D", "H"]
)

print(
    "Our Model Log Loss:",
    model_log_loss
)

print(
    "Bet365 Fair Market Log Loss:",
    bet365_log_loss
)

print(
    "Difference (Model - Bet365):",
    model_log_loss - bet365_log_loss
)

In [ ]:
# Perform a Brier score and compare between both models

# Create one-hot encoded actual outcomes
actual_one_hot = pd.get_dummies(
    walk_market_df["actual"]
)[["A", "D", "H"]].to_numpy()

# Brier score for our model
model_brier = np.mean(
    np.sum(
        (model_probs - actual_one_hot) ** 2,
        axis=1
    )
)

# Brier score for Bet365
bet365_brier = np.mean(
    np.sum(
        (bet365_probs - actual_one_hot) ** 2,
        axis=1
    )
)

print(
    "Our Model Brier Score:",
    model_brier
)

print(
    "Bet365 Fair Market Brier Score:",
    bet365_brier
)

print(
    "Difference (Model - Bet365):",
    model_brier - bet365_brier
)

In [ ]:
# Calculate the probability difference between our model and Bet365 for each outcome.

walk_market_df["edge_H"] = (
    walk_market_df["prob_H"]
    - walk_market_df["fair_prob_H"]
)

walk_market_df["edge_D"] = (
    walk_market_df["prob_D"]
    - walk_market_df["fair_prob_D"]
)

walk_market_df["edge_A"] = (
    walk_market_df["prob_A"]
    - walk_market_df["fair_prob_A"]
)

print(
    walk_market_df[
        [
            "match_id",
            "actual",
            "edge_H",
            "edge_D",
            "edge_A"
        ]
    ].head(10)
)

In [ ]:
print("Home edge:")
print(walk_market_df["edge_H"].describe())

print()

print("Draw edge:")
print(walk_market_df["edge_D"].describe())

print()

print("Away edge:")
print(walk_market_df["edge_A"].describe())

In [ ]:
# Test disagreement size between both models
# How often does the model meaninguflly disagree with the market?

walk_market_df["max_abs_edge"] = (
    walk_market_df[
        [
            "edge_H",
            "edge_D",
            "edge_A"
        ]
    ]
    .abs()
    .max(axis=1)
)

print(
    walk_market_df["max_abs_edge"]
    .describe()
)

print()

print(
    "Matches with 5%+ disagreement:",
    (walk_market_df["max_abs_edge"] >= 0.05).sum()
)

print(
    "Matches with 10%+ disagreement:",
    (walk_market_df["max_abs_edge"] >= 0.10).sum()
)

print(
    "Matches with 15%+ disagreement:",
    (walk_market_df["max_abs_edge"] >= 0.15).sum()
)

print(
    "Matches with 20%+ disagreement:",
    (walk_market_df["max_abs_edge"] >= 0.20).sum()
)

In [ ]:
# See if this is signal or model error

thresholds = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20
]

comparison_rows = []

for threshold in thresholds:

    subset = walk_market_df[
        walk_market_df["max_abs_edge"] >= threshold
    ].copy()

    subset_model_probs = subset[
        ["prob_A", "prob_D", "prob_H"]
    ].to_numpy()

    subset_market_probs = subset[
        ["fair_prob_A", "fair_prob_D", "fair_prob_H"]
    ].to_numpy()

    subset_actual = subset["actual"]

    model_loss = log_loss(
        subset_actual,
        subset_model_probs,
        labels=["A", "D", "H"]
    )

    market_loss = log_loss(
        subset_actual,
        subset_market_probs,
        labels=["A", "D", "H"]
    )

    comparison_rows.append({
        "threshold": threshold,
        "matches": len(subset),
        "model_log_loss": model_loss,
        "bet365_log_loss": market_loss,
        "model_minus_bet365": model_loss - market_loss
    })

disagreement_comparison = pd.DataFrame(
    comparison_rows
)

print(disagreement_comparison)

In [ ]:
walk_market_df["disagreement_bucket"] = pd.cut(
    walk_market_df["max_abs_edge"],
    bins=[
        0,
        0.05,
        0.10,
        0.15,
        0.20,
        np.inf
    ],
    labels=[
        "0-5%",
        "5-10%",
        "10-15%",
        "15-20%",
        "20%+"
    ],
    include_lowest=True
)

bucket_rows = []

for bucket, subset in walk_market_df.groupby(
    "disagreement_bucket",
    observed=True
):

    model_loss = log_loss(
        subset["actual"],
        subset[
            ["prob_A", "prob_D", "prob_H"]
        ],
        labels=["A", "D", "H"]
    )

    market_loss = log_loss(
        subset["actual"],
        subset[
            [
                "fair_prob_A",
                "fair_prob_D",
                "fair_prob_H"
            ]
        ],
        labels=["A", "D", "H"]
    )

    bucket_rows.append({
        "disagreement": bucket,
        "matches": len(subset),
        "model_log_loss": model_loss,
        "bet365_log_loss": market_loss,
        "model_minus_bet365": (
            model_loss - market_loss
        )
    })

bucket_comparison = pd.DataFrame(
    bucket_rows
)

print(bucket_comparison)

In [ ]:
# Split the models' disagreement by outcome first and look at cases where our model gives an outcome a higher probability than Bet365

outcome_edge_rows = []

for outcome in ["H", "D", "A"]:

    edge_col = f"edge_{outcome}"

    # Only cases where our model is more optimistic
    # about this outcome than Bet365
    subset = walk_market_df[
        walk_market_df[edge_col] > 0
    ].copy()

    outcome_edge_rows.append({
        "outcome": outcome,
        "matches": len(subset),
        "average_edge": subset[edge_col].mean(),
        "actual_win_rate": (
            subset["actual"] == outcome
        ).mean(),
        "average_model_prob": subset[
            f"prob_{outcome}"
        ].mean(),
        "average_bet365_prob": subset[
            f"fair_prob_{outcome}"
        ].mean()
    })

outcome_edge_summary = pd.DataFrame(
    outcome_edge_rows
)

print(outcome_edge_summary)

In [ ]:
# Check if smaller positive disagreements behave differently
# Maybe a + 15% model disagreemnent is meaningless, but +2-5% contain some useful signal.

positive_edge_rows = []

for outcome in ["H", "D", "A"]:

    edge_col = f"edge_{outcome}"

    for lower, upper, label in [
        (0.00, 0.05, "0-5%"),
        (0.05, 0.10, "5-10%"),
        (0.10, 0.15, "10-15%"),
        (0.15, np.inf, "15%+")
    ]:

        subset = walk_market_df[
            (walk_market_df[edge_col] > lower)
            & (walk_market_df[edge_col] <= upper)
        ]

        if len(subset) == 0:
            continue

        positive_edge_rows.append({
            "outcome": outcome,
            "edge_range": label,
            "matches": len(subset),

            "model_prob": subset[
                f"prob_{outcome}"
            ].mean(),

            "bet365_prob": subset[
                f"fair_prob_{outcome}"
            ].mean(),

            "actual_rate": (
                subset["actual"] == outcome
            ).mean()
        })

positive_edge_summary = pd.DataFrame(
    positive_edge_rows
)

print(positive_edge_summary)

In [ ]:
# Run Log Reg through the same four-season walk-forward setup as the RF

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_walk_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [ ]:
logistic_walk_results = []

for validation_season in walk_forward_seasons:

    validation_start = (
        model_df.loc[
            model_df["season_home"]
            == validation_season,
            "date_home"
        ]
        .min()
    )

    train_df = model_df[
        model_df["date_home"]
        < validation_start
    ].copy()

    validation_df = model_df[
        model_df["season_home"]
        == validation_season
    ].copy()

    X_train = train_df[
        baseline_features
    ]

    y_train = train_df[
        "target"
    ]

    X_validation = validation_df[
        baseline_features
    ]

    y_validation = validation_df[
        "target"
    ]

    logistic_walk_pipeline.fit(
        X_train,
        y_train
    )

    probabilities = (
        logistic_walk_pipeline
        .predict_proba(X_validation)
    )

    predictions = (
        logistic_walk_pipeline
        .predict(X_validation)
    )

    accuracy = (
        predictions == y_validation
    ).mean()

    loss = log_loss(
        y_validation,
        probabilities,
        labels=logistic_walk_pipeline[
            "model"
        ].classes_
    )

    logistic_walk_results.append({
        "season": validation_season,
        "train_matches": len(train_df),
        "validation_matches": len(validation_df),
        "accuracy": accuracy,
        "log_loss": loss
    })

logistic_walk_results_df = pd.DataFrame(
    logistic_walk_results
)

print(logistic_walk_results_df)

print()

print(
    "Average Accuracy:",
    logistic_walk_results_df[
        "accuracy"
    ].mean()
)

print(
    "Average Log Loss:",
    logistic_walk_results_df[
        "log_loss"
    ].mean()
)

In [ ]:
# Collect match-level Log Reg probabilities using same walk-forward splits

logistic_walk_predictions = []

for validation_season in walk_forward_seasons:

    validation_start = (
        model_df.loc[
            model_df["season_home"] == validation_season,
            "date_home"
        ]
        .min()
    )

    train_df = model_df[
        model_df["date_home"] < validation_start
    ].copy()

    validation_df = (
        model_df[
            model_df["season_home"] == validation_season
        ]
        .copy()
        .sort_values(["date_home", "match_id"])
    )

    X_train = train_df[baseline_features]
    y_train = train_df["target"]

    X_validation = validation_df[
        baseline_features
    ]

    logistic_walk_pipeline.fit(
        X_train,
        y_train
    )

    probabilities = (
        logistic_walk_pipeline
        .predict_proba(X_validation)
    )

    classes = (
        logistic_walk_pipeline[
            "model"
        ].classes_
    )

    class_to_index = {
        label: i
        for i, label in enumerate(classes)
    }

    for i, (_, row) in enumerate(
        validation_df.iterrows()
    ):

        logistic_walk_predictions.append({
            "match_id": row["match_id"],
            "season": validation_season,
            "actual": row["target"],
            "log_prob_A": probabilities[
                i, class_to_index["A"]
            ],
            "log_prob_D": probabilities[
                i, class_to_index["D"]
            ],
            "log_prob_H": probabilities[
                i, class_to_index["H"]
            ]
        })

logistic_predictions_df = pd.DataFrame(
    logistic_walk_predictions
)

print(logistic_predictions_df.shape)
print()
print(logistic_predictions_df.head())
print()
print(
    "Duplicate IDs:",
    logistic_predictions_df[
        "match_id"
    ].duplicated().sum()
)

In [ ]:
# Merge RF and Logistic predictions

rf_logistic_comparison = (
    walk_predictions_df[
        [
            "match_id",
            "season",
            "actual",
            "prob_A",
            "prob_D",
            "prob_H"
        ]
    ]
    .merge(
        logistic_predictions_df[
            [
                "match_id",
                "log_prob_A",
                "log_prob_D",
                "log_prob_H"
            ]
        ],
        on="match_id",
        how="inner"
    )
)

print(
    rf_logistic_comparison.shape
)

print()

print(
    "Actual outcome mismatches:",
    (
        rf_logistic_comparison["actual"]
        != logistic_predictions_df.set_index(
            "match_id"
        ).loc[
            rf_logistic_comparison["match_id"],
            "actual"
        ].to_numpy()
    ).sum()
)

print()

print(
    rf_logistic_comparison.head()
)

In [ ]:
# Perform paired bootstrap
# Logistic Loss - RF Loss
# Positive = RF better

prob_columns_rf = {
    "A": "prob_A",
    "D": "prob_D",
    "H": "prob_H"
}

prob_columns_logistic = {
    "A": "log_prob_A",
    "D": "log_prob_D",
    "H": "log_prob_H"
}

rf_logistic_comparison["rf_loss"] = [
    -np.log(
        row[
            prob_columns_rf[
                row["actual"]
            ]
        ]
    )
    for _, row
    in rf_logistic_comparison.iterrows()
]

rf_logistic_comparison["logistic_loss"] = [
    -np.log(
        row[
            prob_columns_logistic[
                row["actual"]
            ]
        ]
    )
    for _, row
    in rf_logistic_comparison.iterrows()
]

rf_logistic_comparison["rf_improvement"] = (
    rf_logistic_comparison["logistic_loss"]
    - rf_logistic_comparison["rf_loss"]
)

print(
    "RF Log Loss:",
    rf_logistic_comparison[
        "rf_loss"
    ].mean()
)

print(
    "Logistic Log Loss:",
    rf_logistic_comparison[
        "logistic_loss"
    ].mean()
)

print(
    "Observed RF Improvement:",
    rf_logistic_comparison[
        "rf_improvement"
    ].mean()
)

In [ ]:
# Test whether that difference is statistically convincing or just sampling noise

rng = np.random.default_rng(42)

rf_improvements = (
    rf_logistic_comparison[
        "rf_improvement"
    ].to_numpy()
)

n_bootstraps = 10000

bootstrap_rf_improvements = []

for _ in range(n_bootstraps):

    bootstrap_sample = rng.choice(
        rf_improvements,
        size=len(rf_improvements),
        replace=True
    )

    bootstrap_rf_improvements.append(
        bootstrap_sample.mean()
    )

bootstrap_rf_improvements = np.array(
    bootstrap_rf_improvements
)

lower_bound = np.percentile(
    bootstrap_rf_improvements,
    2.5
)

upper_bound = np.percentile(
    bootstrap_rf_improvements,
    97.5
)

print(
    "Observed RF Improvement:",
    rf_improvements.mean()
)

print(
    "95% Bootstrap CI:",
    (lower_bound, upper_bound)
)

print(
    "Probability RF improvement <= 0:",
    np.mean(
        bootstrap_rf_improvements <= 0
    )
)

# IMPROVING/TUNING OUR CHAMPION RANDOM FOREST MODEL

In [ ]:
# Step 1: Test tree depth
# max_depth controls how complex each decision tree is allowed to become.
# If trees are too shallow, RF may miss useful relationships between features
# If trees are too deep, they can memorize noise and produce worse probabilities on new matches.
# Trying to find the best balance between underfitting and overfitting

from sklearn.ensemble import RandomForestClassifier

depth_values = [
    4,
    6,
    8,
    10,
    12,
    None
]

depth_results = []

for depth in depth_values:

    season_losses = []
    season_accuracies = []

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"] == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"] < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"] == validation_season
        ].copy()

        X_train = train_df[baseline_features]
        y_train = train_df["target"]

        X_validation = validation_df[
            baseline_features
        ]

        y_validation = validation_df["target"]

        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=depth,
                    min_samples_leaf=5,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])

        rf_pipeline.fit(
            X_train,
            y_train
        )

        probabilities = rf_pipeline.predict_proba(
            X_validation
        )

        predictions = rf_pipeline.predict(
            X_validation
        )

        season_losses.append(
            log_loss(
                y_validation,
                probabilities,
                labels=rf_pipeline[
                    "model"
                ].classes_
            )
        )

        season_accuracies.append(
            (
                predictions
                == y_validation
            ).mean()
        )

    depth_results.append({
        "max_depth": str(depth),
        "avg_accuracy": np.mean(
            season_accuracies
        ),
        "avg_log_loss": np.mean(
            season_losses
        )
    })

depth_results_df = pd.DataFrame(
    depth_results
)

print(
    depth_results_df.sort_values(
        "avg_log_loss"
    )
)

# As the tree gets deeper than 6, probability performance gets worse.
# Suggests that the RF benefits from being somewhat constrained.
# Football results can contain a lot of randomness, so allowing individual trees learn specific patterns can make them fit noise rather than relationships that generalize to future seasons

In [ ]:
# Step 2: Change minimum number of training matches that must end up in a leaf of each tree
# A very small value allows the RF to create predictions based on tiny groups of historically similar matches, which can make probabilities unstable.
# A larger value forces the model to base predictions on broader groups of matches, which can improve generalization and probability quality

leaf_values = [
    2,
    5,
    10,
    15,
    20,
    30
]

leaf_results = []

for leaf_size in leaf_values:

    season_losses = []
    season_accuracies = []

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"] == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"] < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"] == validation_season
        ].copy()

        X_train = train_df[
            baseline_features
        ]

        y_train = train_df[
            "target"
        ]

        X_validation = validation_df[
            baseline_features
        ]

        y_validation = validation_df[
            "target"
        ]

        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=leaf_size,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])

        rf_pipeline.fit(
            X_train,
            y_train
        )

        probabilities = rf_pipeline.predict_proba(
            X_validation
        )

        predictions = rf_pipeline.predict(
            X_validation
        )

        season_losses.append(
            log_loss(
                y_validation,
                probabilities,
                labels=rf_pipeline[
                    "model"
                ].classes_
            )
        )

        season_accuracies.append(
            (
                predictions
                == y_validation
            ).mean()
        )

    leaf_results.append({
        "min_samples_leaf": leaf_size,
        "avg_accuracy": np.mean(
            season_accuracies
        ),
        "avg_log_loss": np.mean(
            season_losses
        )
    })


leaf_results_df = pd.DataFrame(
    leaf_results
)

print(
    leaf_results_df.sort_values(
        "avg_log_loss"
    )
)

In [ ]:
# Step 3: Test Max Features
# Each tree in a RF doesn't necessarily consider every feature when deciding where to split
# max_features controls how many features each tree gets to consider at each split.
# Using fewer features make the individual trees more different from each other
# Using more features lets each tree consistently choose from the strongest predictors
# The right balance can make the overall forest generalize better.

max_feature_values = [
    "sqrt",
    "log2",
    0.3,
    0.5,
    0.7,
    1.0
]

max_feature_results = []

for max_features_value in max_feature_values:

    season_losses = []
    season_accuracies = []

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"] == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"] < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"] == validation_season
        ].copy()

        X_train = train_df[
            baseline_features
        ]

        y_train = train_df[
            "target"
        ]

        X_validation = validation_df[
            baseline_features
        ]

        y_validation = validation_df[
            "target"
        ]

        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features=max_features_value,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])

        rf_pipeline.fit(
            X_train,
            y_train
        )

        probabilities = rf_pipeline.predict_proba(
            X_validation
        )

        predictions = rf_pipeline.predict(
            X_validation
        )

        season_losses.append(
            log_loss(
                y_validation,
                probabilities,
                labels=rf_pipeline[
                    "model"
                ].classes_
            )
        )

        season_accuracies.append(
            (
                predictions
                == y_validation
            ).mean()
        )

    max_feature_results.append({
        "max_features": str(
            max_features_value
        ),
        "avg_accuracy": np.mean(
            season_accuracies
        ),
        "avg_log_loss": np.mean(
            season_losses
        )
    })


max_feature_results_df = pd.DataFrame(
    max_feature_results
)

print(
    max_feature_results_df.sort_values(
        "avg_log_loss"
    )
)

In [ ]:
# Step 4: Test number of estimators
# n_estimators is the number of decision trees in that RF
# More trees make the model's probabiltiies more stable because we're averaging across more individual trees
# But eventually the improvement levels off.
# Unlike depth, adding trees usualyl doesn't cause overfitting, it mainly costs more computation

estimator_values = [
    100,
    250,
    500,
    750,
    1000
]

estimator_results = []

for n_estimators_value in estimator_values:

    season_losses = []
    season_accuracies = []

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"] == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"] < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"] == validation_season
        ].copy()

        X_train = train_df[
            baseline_features
        ]

        y_train = train_df[
            "target"
        ]

        X_validation = validation_df[
            baseline_features
        ]

        y_validation = validation_df[
            "target"
        ]

        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=n_estimators_value,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])

        rf_pipeline.fit(
            X_train,
            y_train
        )

        probabilities = rf_pipeline.predict_proba(
            X_validation
        )

        predictions = rf_pipeline.predict(
            X_validation
        )

        season_losses.append(
            log_loss(
                y_validation,
                probabilities,
                labels=rf_pipeline[
                    "model"
                ].classes_
            )
        )

        season_accuracies.append(
            (
                predictions
                == y_validation
            ).mean()
        )

    estimator_results.append({
        "n_estimators": n_estimators_value,
        "avg_accuracy": np.mean(
            season_accuracies
        ),
        "avg_log_loss": np.mean(
            season_losses
        )
    })


estimator_results_df = pd.DataFrame(
    estimator_results
)

print(
    estimator_results_df.sort_values(
        "avg_log_loss"
    )
)

# Adding more than 500 would mainly increase computation without helping predictions

In [ ]:
# Step 5: Test class weight
# PL outcomes aren't evenly distributed between home wins, draws, and away wins.
# A RF can naturally favor the more common outcomes, mainly home wins, and struggle with the less common class, draws.
# class_weight lets us tell the model to give more improtance to less-common outcoms during training.

class_weight_values = [
    None,
    "balanced",
    "balanced_subsample"
]

class_weight_results = []

for class_weight_value in class_weight_values:

    season_losses = []
    season_accuracies = []

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"] == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"] < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"] == validation_season
        ].copy()

        X_train = train_df[
            baseline_features
        ]

        y_train = train_df[
            "target"
        ]

        X_validation = validation_df[
            baseline_features
        ]

        y_validation = validation_df[
            "target"
        ]

        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    class_weight=class_weight_value,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])

        rf_pipeline.fit(
            X_train,
            y_train
        )

        probabilities = rf_pipeline.predict_proba(
            X_validation
        )

        predictions = rf_pipeline.predict(
            X_validation
        )

        season_losses.append(
            log_loss(
                y_validation,
                probabilities,
                labels=rf_pipeline[
                    "model"
                ].classes_
            )
        )

        season_accuracies.append(
            (
                predictions
                == y_validation
            ).mean()
        )

    class_weight_results.append({
        "class_weight": str(
            class_weight_value
        ),
        "avg_accuracy": np.mean(
            season_accuracies
        ),
        "avg_log_loss": np.mean(
            season_losses
        )
    })


class_weight_results_df = pd.DataFrame(
    class_weight_results
)

print(
    class_weight_results_df.sort_values(
        "avg_log_loss"
    )
)

In [ ]:
# Test criterion
# Determines how much each decision tree decides which feature and split is best
# Ultimate goal is good probabilities, rather than classification accuracy, it's worth seeing whether a different splitting rule produces better out-of-sample probabiltiies.

criterion_values = [
    "gini",
    "entropy",
    "log_loss"
]

criterion_results = []

for criterion_value in criterion_values:

    season_losses = []
    season_accuracies = []

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"] == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"] < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"] == validation_season
        ].copy()

        X_train = train_df[
            baseline_features
        ]

        y_train = train_df[
            "target"
        ]

        X_validation = validation_df[
            baseline_features
        ]

        y_validation = validation_df[
            "target"
        ]

        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    class_weight=None,
                    criterion=criterion_value,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])

        rf_pipeline.fit(
            X_train,
            y_train
        )

        probabilities = rf_pipeline.predict_proba(
            X_validation
        )

        predictions = rf_pipeline.predict(
            X_validation
        )

        season_losses.append(
            log_loss(
                y_validation,
                probabilities,
                labels=rf_pipeline[
                    "model"
                ].classes_
            )
        )

        season_accuracies.append(
            (
                predictions
                == y_validation
            ).mean()
        )

    criterion_results.append({
        "criterion": criterion_value,
        "avg_accuracy": np.mean(
            season_accuracies
        ),
        "avg_log_loss": np.mean(
            season_losses
        )
    })


criterion_results_df = pd.DataFrame(
    criterion_results
)

print(
    criterion_results_df.sort_values(
        "avg_log_loss"
    )
)

In [ ]:
# Step 6: Calibration
# A RF can be good at ranking which outcome is most likely while still producing probabilites that are slightly too confident or too conservative.
# Calibration adjusts the probabiltiies so that outcomes predicted around 60% actually happen 60% of them time
# Log loss directly rewards accurate probabilities, but calibration could improve the model without changing the underlying RF predictions

# Recreate walk-forward predictions using our tuned RF

tuned_rf_predictions = []

for validation_season in walk_forward_seasons:

    validation_start = (
        model_df.loc[
            model_df["season_home"] == validation_season,
            "date_home"
        ]
        .min()
    )

    train_df = model_df[
        model_df["date_home"] < validation_start
    ].copy()

    validation_df = model_df[
        model_df["season_home"] == validation_season
    ].copy()

    X_train = train_df[
        baseline_features
    ]

    y_train = train_df[
        "target"
    ]

    X_validation = validation_df[
        baseline_features
    ]

    y_validation = validation_df[
        "target"
    ]

    rf_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=6,
                min_samples_leaf=10,
                max_features="sqrt",
                criterion="gini",
                class_weight=None,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    rf_pipeline.fit(
        X_train,
        y_train
    )

    probabilities = rf_pipeline.predict_proba(
        X_validation
    )

    classes = rf_pipeline[
        "model"
    ].classes_

    for i, (_, row) in enumerate(
        validation_df.iterrows()
    ):

        probability_dict = dict(
            zip(
                classes,
                probabilities[i]
            )
        )

        tuned_rf_predictions.append({
            "match_id": row["match_id"],
            "season": validation_season,
            "actual": row["target"],
            "prob_A": probability_dict["A"],
            "prob_D": probability_dict["D"],
            "prob_H": probability_dict["H"]
        })


tuned_rf_predictions_df = pd.DataFrame(
    tuned_rf_predictions
)

print(tuned_rf_predictions_df.shape)

print()

print(
    tuned_rf_predictions_df.head()
)

In [ ]:
# Measure calibration

calibration_results = []

outcome_columns = {
    "H": "prob_H",
    "D": "prob_D",
    "A": "prob_A"
}

for outcome, probability_column in outcome_columns.items():

    temp_df = tuned_rf_predictions_df[
        ["actual", probability_column]
    ].copy()

    temp_df["actual_binary"] = (
        temp_df["actual"] == outcome
    ).astype(int)

    temp_df["probability_bin"] = pd.cut(
        temp_df[probability_column],
        bins=np.arange(0, 1.1, 0.1),
        include_lowest=True
    )

    grouped = (
        temp_df
        .groupby(
            "probability_bin",
            observed=True
        )
        .agg(
            matches=("actual_binary", "size"),
            avg_predicted_probability=(
                probability_column,
                "mean"
            ),
            actual_rate=(
                "actual_binary",
                "mean"
            )
        )
        .reset_index()
    )

    grouped["outcome"] = outcome

    calibration_results.append(
        grouped
    )


calibration_df = pd.concat(
    calibration_results,
    ignore_index=True
)

calibration_df[
    "calibration_difference"
] = (
    calibration_df[
        "actual_rate"
    ]
    - calibration_df[
        "avg_predicted_probability"
    ]
)

print(
    calibration_df[
        [
            "outcome",
            "probability_bin",
            "matches",
            "avg_predicted_probability",
            "actual_rate",
            "calibration_difference"
        ]
    ].to_string(index=False)
)

In [ ]:
# Test sigmoid calibration
# Learns a relatively simple adjustment between RF's raw probabilities and raw outcomes

# When predicting 2024/25, the calibration process can use earlier seasons, but nothing from 2024/25 itself
# The calibration table showed that the RF probabilities aren't terrible but there are systematic differences, especially with away wins
# Sigmoid calibration learns a correction to RF probabilities

# For the first test, evaluate 2022/23. 2023/24, and 2024/25. 
# Don't use 2021/22 because we need 2020/21 both to train the RF and provide a separate calibration period.
# Starting in 2022/23 gives enough historical data to separate

from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

calibration_test_seasons = [
    "2022_23",
    "2023_24",
    "2024_25"
]

previous_season = {
    "2022_23": "2021_22",
    "2023_24": "2022_23",
    "2024_25": "2023_24"
}

calibration_results = []

for validation_season in calibration_test_seasons:

    calibration_season = previous_season[
        validation_season
    ]

    calibration_start = (
        model_df.loc[
            model_df["season_home"]
            == calibration_season,
            "date_home"
        ]
        .min()
    )

    train_df = model_df[
        model_df["date_home"] < calibration_start
    ].copy()

    calibration_df = model_df[
        model_df["season_home"]
        == calibration_season
    ].copy()

    validation_df = model_df[
        model_df["season_home"]
        == validation_season
    ].copy()

    X_train = train_df[
        baseline_features
    ]

    y_train = train_df[
        "target"
    ]

    X_calibration = calibration_df[
        baseline_features
    ]

    y_calibration = calibration_df[
        "target"
    ]

    X_validation = validation_df[
        baseline_features
    ]

    y_validation = validation_df[
        "target"
    ]

    # -----------------------------
    # Train the tuned RF
    # -----------------------------

    rf_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=6,
                min_samples_leaf=10,
                max_features="sqrt",
                criterion="gini",
                class_weight=None,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    rf_pipeline.fit(
        X_train,
        y_train
    )

    # -----------------------------
    # Uncalibrated probabilities
    # -----------------------------

    raw_probabilities = (
        rf_pipeline.predict_proba(
            X_validation
        )
    )

    raw_log_loss = log_loss(
        y_validation,
        raw_probabilities,
        labels=rf_pipeline.classes_
    )

    # -----------------------------
    # Sigmoid calibration
    # -----------------------------

    calibrated_rf = CalibratedClassifierCV(
        estimator = FrozenEstimator(
            rf_pipeline
        ),
        method = "sigmoid"
    )

    calibrated_rf.fit(
        X_calibration,
        y_calibration
    )

    calibrated_probabilities = (
        calibrated_rf.predict_proba(
            X_validation
        )
    )

    calibrated_log_loss = log_loss(
        y_validation,
        calibrated_probabilities,
        labels=calibrated_rf.classes_
    )

    calibration_results.append({
        "validation_season":
            validation_season,

        "training_matches":
            len(train_df),

        "calibration_matches":
            len(calibration_df),

        "validation_matches":
            len(validation_df),

        "raw_log_loss":
            raw_log_loss,

        "calibrated_log_loss":
            calibrated_log_loss,

        "improvement":
            raw_log_loss
            - calibrated_log_loss
    })


calibration_test_df = pd.DataFrame(
    calibration_results
)

print(
    calibration_test_df.to_string(
        index=False
    )
)

print()

print(
    "Average Raw Log Loss:",
    calibration_test_df[
        "raw_log_loss"
    ].mean()
)

print(
    "Average Calibrated Log Loss:",
    calibration_test_df[
        "calibrated_log_loss"
    ].mean()
)

print(
    "Average Improvement:",
    calibration_test_df[
        "improvement"
    ].mean()
)

In [ ]:
# Train the RF on as much historical data as possible while still generating leakage-safe calibration predictions from the past
# Lets us have more RF training data + time-safe probability calibration

# Step 1: Generate historical out-of-fold predictions
# Previously we had to sacrifice an entire season to calibrate the RF
# Instead we create predictions for older matches using models that had not seen those matches yet.
# The historical predictions can later teach the calibrator how the RF probabilites tend to behave
# The final RF can still train on all available past data

calibration_oof_seasons = [
    "2021_22",
    "2022_23",
    "2023_24"
]

calibration_oof_predictions = []

for prediction_season in calibration_oof_seasons:

    prediction_start = (
        model_df.loc[
            model_df["season_home"] == prediction_season,
            "date_home"
        ]
        .min()
    )

    train_df = model_df[
        model_df["date_home"] < prediction_start
    ].copy()

    prediction_df = model_df[
        model_df["season_home"] == prediction_season
    ].copy()

    X_train = train_df[
        baseline_features
    ]

    y_train = train_df[
        "target"
    ]

    X_prediction = prediction_df[
        baseline_features
    ]

    rf_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=6,
                min_samples_leaf=10,
                max_features="sqrt",
                criterion="gini",
                class_weight=None,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    rf_pipeline.fit(
        X_train,
        y_train
    )

    probabilities = rf_pipeline.predict_proba(
        X_prediction
    )

    classes = rf_pipeline[
        "model"
    ].classes_

    for i, (_, row) in enumerate(
        prediction_df.iterrows()
    ):

        probability_dict = dict(
            zip(
                classes,
                probabilities[i]
            )
        )

        calibration_oof_predictions.append({
            "match_id": row["match_id"],
            "season": prediction_season,
            "actual": row["target"],
            "prob_A": probability_dict["A"],
            "prob_D": probability_dict["D"],
            "prob_H": probability_dict["H"]
        })


calibration_oof_df = pd.DataFrame(
    calibration_oof_predictions
)

print(
    calibration_oof_df.shape
)

print()

print(
    calibration_oof_df[
        "season"
    ].value_counts().sort_index()
)

print()

print(
    calibration_oof_df.head()
)

In [ ]:
# Step 2: Properly test calibration on 2024/25

# Use the 1,140 historical out-of-sample predictions to learn the calibrationa adjustment.
# The RF that predicts 2024/25 can still train on every match through 2023/24.
# Unlike previous experiments, no need to sacrifice an entire recent season of RF training data just to calibrate

from sklearn.linear_model import LogisticRegression

# --------------------------------------------------
# 1. Train the probability calibrator using
#    historical out-of-sample RF predictions
# --------------------------------------------------

X_calibrator = calibration_oof_df[
    [
        "prob_A",
        "prob_D",
        "prob_H"
    ]
]

y_calibrator = calibration_oof_df[
    "actual"
]

probability_calibrator = LogisticRegression(
    max_iter=2000,
    random_state=42
)

probability_calibrator.fit(
    X_calibrator,
    y_calibrator
)


# --------------------------------------------------
# 2. Train our tuned RF on ALL data before 2024/25
# --------------------------------------------------

validation_season = "2024_25"

validation_start = (
    model_df.loc[
        model_df["season_home"]
        == validation_season,
        "date_home"
    ]
    .min()
)

train_df = model_df[
    model_df["date_home"]
    < validation_start
].copy()

validation_df = model_df[
    model_df["season_home"]
    == validation_season
].copy()

X_train = train_df[
    baseline_features
]

y_train = train_df[
    "target"
]

X_validation = validation_df[
    baseline_features
]

y_validation = validation_df[
    "target"
]


tuned_rf = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=10,
            max_features="sqrt",
            criterion="gini",
            class_weight=None,
            random_state=42,
            n_jobs=-1
        )
    )
])

tuned_rf.fit(
    X_train,
    y_train
)


# --------------------------------------------------
# 3. Get raw RF probabilities for 2024/25
# --------------------------------------------------

raw_probabilities = (
    tuned_rf.predict_proba(
        X_validation
    )
)

rf_classes = tuned_rf[
    "model"
].classes_

raw_probability_df = pd.DataFrame(
    raw_probabilities,
    columns=rf_classes
)

raw_probability_df = (
    raw_probability_df[
        ["A", "D", "H"]
    ]
)

raw_probability_df.columns = [
    "prob_A",
    "prob_D",
    "prob_H"
]


# --------------------------------------------------
# 4. Apply our historical calibrator
# --------------------------------------------------

calibrated_probabilities = (
    probability_calibrator.predict_proba(
        raw_probability_df
    )
)

calibrator_classes = (
    probability_calibrator.classes_
)


# --------------------------------------------------
# 5. Compare raw vs calibrated log loss
# --------------------------------------------------

raw_log_loss = log_loss(
    y_validation,
    raw_probabilities,
    labels=rf_classes
)

calibrated_log_loss = log_loss(
    y_validation,
    calibrated_probabilities,
    labels=calibrator_classes
)


print(
    "RF training matches:",
    len(train_df)
)

print(
    "Calibration predictions:",
    len(calibration_oof_df)
)

print(
    "2024/25 validation matches:",
    len(validation_df)
)

print()

print(
    "Raw RF Log Loss:",
    raw_log_loss
)

print(
    "Calibrated RF Log Loss:",
    calibrated_log_loss
)

print(
    "Improvement:",
    raw_log_loss
    - calibrated_log_loss
)

In [ ]:
# Improve the actual football information

feature_importance_train = model_df[
    model_df["season_home"].isin([
        "2020_21",
        "2021_22",
        "2022_23",
        "2023_24",
        "2024_25"
    ])
].copy()

X_feature_train = feature_importance_train[
    baseline_features
]

y_feature_train = feature_importance_train[
    "target"
]


importance_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=10,
            max_features="sqrt",
            criterion="gini",
            class_weight=None,
            random_state=42,
            n_jobs=-1
        )
    )
])

importance_pipeline.fit(
    X_feature_train,
    y_feature_train
)


# Get names after missing-value indicators
feature_names = (
    importance_pipeline[
        "imputer"
    ].get_feature_names_out(
        baseline_features
    )
)

feature_importances = (
    importance_pipeline[
        "model"
    ].feature_importances_
)


feature_importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": feature_importances
    })
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    feature_importance_df.to_string(
        index=False
    )
)

In [ ]:
# Create matchup-difference features

model_df[
    "points_last5_pl_difference"
] = (
    model_df["points_last5_pl_home"]
    - model_df["points_last5_pl_away"]
)

model_df[
    "goal_difference_last5_pl_difference"
] = (
    model_df["goal_difference_last5_pl_home"]
    - model_df["goal_difference_last5_pl_away"]
)

model_df[
    "shots_on_target_last5_pl_difference"
] = (
    model_df[
        "shots_on_target_difference_last5_pl_home"
    ]
    - model_df[
        "shots_on_target_difference_last5_pl_away"
    ]
)

model_df[
    "win_rate_last5_pl_difference"
] = (
    model_df["win_rate_last5_pl_home"]
    - model_df["win_rate_last5_pl_away"]
)

model_df[
    "points_last10_pl_difference"
] = (
    model_df["points_last10_pl_home"]
    - model_df["points_last10_pl_away"]
)

model_df[
    "goal_difference_last10_pl_difference"
] = (
    model_df["goal_difference_last10_pl_home"]
    - model_df["goal_difference_last10_pl_away"]
)

model_df[
    "win_rate_last10_pl_difference"
] = (
    model_df["win_rate_last10_pl_home"]
    - model_df["win_rate_last10_pl_away"]
)

model_df[
    "days_rest_difference"
] = (
    model_df["days_rest_home"]
    - model_df["days_rest_away"]
)

model_df[
    "matches_last_7_days_difference"
] = (
    model_df["matches_last_7_days_home"]
    - model_df["matches_last_7_days_away"]
)

model_df[
    "matches_last_14_days_difference"
] = (
    model_df["matches_last_14_days_home"]
    - model_df["matches_last_14_days_away"]
)


matchup_difference_features = [
    "points_last5_pl_difference",
    "goal_difference_last5_pl_difference",
    "shots_on_target_last5_pl_difference",
    "win_rate_last5_pl_difference",
    "points_last10_pl_difference",
    "goal_difference_last10_pl_difference",
    "win_rate_last10_pl_difference",
    "days_rest_difference",
    "matches_last_7_days_difference",
    "matches_last_14_days_difference"
]


print(
    model_df[
        matchup_difference_features
    ].head()
)

print()

print(
    "New matchup features:",
    len(matchup_difference_features)
)

In [ ]:
# Clean up DataFrame fragmentation warning
model_df = model_df.copy()


# Original features + new matchup difference features
matchup_feature_set = (
    baseline_features
    + matchup_difference_features
)


feature_set_results = []

feature_sets = {
    "Original 31": baseline_features,
    "Original 31 + Matchup Differences": matchup_feature_set
}


for feature_set_name, feature_list in feature_sets.items():

    season_losses = []
    season_accuracies = []

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"]
                == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"]
            < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"]
            == validation_season
        ].copy()


        X_train = train_df[
            feature_list
        ]

        y_train = train_df[
            "target"
        ]

        X_validation = validation_df[
            feature_list
        ]

        y_validation = validation_df[
            "target"
        ]


        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    criterion="gini",
                    class_weight=None,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])


        rf_pipeline.fit(
            X_train,
            y_train
        )


        probabilities = (
            rf_pipeline.predict_proba(
                X_validation
            )
        )

        predictions = (
            rf_pipeline.predict(
                X_validation
            )
        )


        season_loss = log_loss(
            y_validation,
            probabilities,
            labels=rf_pipeline[
                "model"
            ].classes_
        )

        season_accuracy = (
            predictions
            == y_validation
        ).mean()


        season_losses.append(
            season_loss
        )

        season_accuracies.append(
            season_accuracy
        )


        feature_set_results.append({
            "feature_set": feature_set_name,
            "season": validation_season,
            "accuracy": season_accuracy,
            "log_loss": season_loss
        })


feature_set_results_df = pd.DataFrame(
    feature_set_results
)


print("RESULTS BY SEASON")
print()

print(
    feature_set_results_df.to_string(
        index=False
    )
)


print()
print("AVERAGE RESULTS")
print()


feature_set_average_df = (
    feature_set_results_df
    .groupby("feature_set")
    .agg(
        avg_accuracy=(
            "accuracy",
            "mean"
        ),
        avg_log_loss=(
            "log_loss",
            "mean"
        )
    )
    .sort_values(
        "avg_log_loss"
    )
)


print(
    feature_set_average_df
)

In [ ]:
# Separate attacking and defensive form

goal_columns = [
    column
    for column in model_df.columns
    if (
        "goals_for" in column
        or "goals_against" in column
    )
]

print(goal_columns)

In [ ]:
attack_defense_features = [
    "goals_for_last5_pl_home",
    "goals_against_last5_pl_home",
    "goals_for_last5_pl_away",
    "goals_against_last5_pl_away"
]


attack_defense_feature_set = (
    baseline_features
    + attack_defense_features
)


attack_defense_results = []

feature_sets = {
    "Original 31": baseline_features,
    "Original 31 + Attack/Defense": attack_defense_feature_set
}


for feature_set_name, feature_list in feature_sets.items():

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"]
                == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"]
            < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"]
            == validation_season
        ].copy()


        X_train = train_df[
            feature_list
        ]

        y_train = train_df[
            "target"
        ]

        X_validation = validation_df[
            feature_list
        ]

        y_validation = validation_df[
            "target"
        ]


        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    criterion="gini",
                    class_weight=None,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])


        rf_pipeline.fit(
            X_train,
            y_train
        )


        probabilities = (
            rf_pipeline.predict_proba(
                X_validation
            )
        )

        predictions = (
            rf_pipeline.predict(
                X_validation
            )
        )


        season_loss = log_loss(
            y_validation,
            probabilities,
            labels=rf_pipeline[
                "model"
            ].classes_
        )

        season_accuracy = (
            predictions
            == y_validation
        ).mean()


        attack_defense_results.append({
            "feature_set": feature_set_name,
            "season": validation_season,
            "accuracy": season_accuracy,
            "log_loss": season_loss
        })


attack_defense_results_df = pd.DataFrame(
    attack_defense_results
)


print("RESULTS BY SEASON")
print()

print(
    attack_defense_results_df.to_string(
        index=False
    )
)


print()
print("AVERAGE RESULTS")
print()


attack_defense_average_df = (
    attack_defense_results_df
    .groupby("feature_set")
    .agg(
        avg_accuracy=(
            "accuracy",
            "mean"
        ),
        avg_log_loss=(
            "log_loss",
            "mean"
        )
    )
    .sort_values(
        "avg_log_loss"
    )
)


print(
    attack_defense_average_df
)

In [ ]:
feature_groups_to_remove = {
    "Original 31": [],

    "Remove All-Competition Form": [
        "points_last5_all_home",
        "goal_difference_last5_all_home",
        "win_rate_last5_all_home",
        "shots_on_target_difference_last5_all_home",

        "points_last5_all_away",
        "goal_difference_last5_all_away",
        "win_rate_last5_all_away",
        "shots_on_target_difference_last5_all_away"
    ],

    "Remove PL Last-5 Form": [
        "points_last5_pl_home",
        "goal_difference_last5_pl_home",
        "win_rate_last5_pl_home",
        "shots_on_target_difference_last5_pl_home",

        "points_last5_pl_away",
        "goal_difference_last5_pl_away",
        "win_rate_last5_pl_away",
        "shots_on_target_difference_last5_pl_away"
    ],

    "Remove PL Last-10 Form": [
        "points_last10_pl_home",
        "goal_difference_last10_pl_home",
        "win_rate_last10_pl_home",

        "points_last10_pl_away",
        "goal_difference_last10_pl_away",
        "win_rate_last10_pl_away"
    ],

    "Remove Rest/Congestion": [
        "days_rest_home",
        "days_rest_away",
        "matches_last_7_days_home",
        "matches_last_7_days_away",
        "matches_last_14_days_home",
        "matches_last_14_days_away"
    ],

    "Remove Elo Home/Away, Keep Difference": [
        "team_elo_home",
        "team_elo_away"
    ]
}


ablation_results = []


for test_name, remove_features in feature_groups_to_remove.items():

    test_features = [
        feature
        for feature in baseline_features
        if feature not in remove_features
    ]

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"]
                == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"]
            < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"]
            == validation_season
        ].copy()


        X_train = train_df[
            test_features
        ]

        y_train = train_df[
            "target"
        ]

        X_validation = validation_df[
            test_features
        ]

        y_validation = validation_df[
            "target"
        ]


        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    criterion="gini",
                    class_weight=None,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])


        rf_pipeline.fit(
            X_train,
            y_train
        )


        probabilities = (
            rf_pipeline.predict_proba(
                X_validation
            )
        )

        predictions = (
            rf_pipeline.predict(
                X_validation
            )
        )


        season_loss = log_loss(
            y_validation,
            probabilities,
            labels=rf_pipeline[
                "model"
            ].classes_
        )

        season_accuracy = (
            predictions
            == y_validation
        ).mean()


        ablation_results.append({
            "feature_set": test_name,
            "feature_count": len(test_features),
            "season": validation_season,
            "accuracy": season_accuracy,
            "log_loss": season_loss
        })


ablation_results_df = pd.DataFrame(
    ablation_results
)


ablation_average_df = (
    ablation_results_df
    .groupby(
        [
            "feature_set",
            "feature_count"
        ]
    )
    .agg(
        avg_accuracy=(
            "accuracy",
            "mean"
        ),
        avg_log_loss=(
            "log_loss",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "avg_log_loss"
    )
)


print("AVERAGE RESULTS")
print()

print(
    ablation_average_df.to_string(
        index=False
    )
)

print()
print("RESULTS BY SEASON")
print()

print(
    ablation_results_df[
        [
            "feature_set",
            "season",
            "log_loss"
        ]
    ].to_string(
        index=False
    )
)

In [ ]:
# Measure home advantage in our data

home_advantage_summary = (
    model_df["target"]
    .value_counts()
    .reindex(["H", "D", "A"])
)

home_advantage_pct = (
    home_advantage_summary
    / len(model_df)
)

home_advantage_df = pd.DataFrame({
    "result": [
        "Home Win",
        "Draw",
        "Away Win"
    ],
    "matches": home_advantage_summary.values,
    "percentage": (
        home_advantage_pct.values * 100
    )
})


print(
    home_advantage_df.to_string(
        index=False
    )
)

print()

print(
    "Total matches:",
    len(model_df)
)

In [ ]:
# Provide and ELO boost to teams with home-field advantage

elo_home_advantage_tests = []


# Actual Elo-style result:
# Home win = 1
# Draw = 0.5
# Away win = 0
elo_actual_result = (
    model_df["target"]
    .map({
        "H": 1.0,
        "D": 0.5,
        "A": 0.0
    })
)


for home_advantage in range(
    0,
    151,
    10
):

    expected_home_score = (
        1
        / (
            1
            + 10 ** (
                (
                    model_df["team_elo_away"]
                    - (
                        model_df["team_elo_home"]
                        + home_advantage
                    )
                )
                / 400
            )
        )
    )


    mean_squared_error = (
        (
            elo_actual_result
            - expected_home_score
        ) ** 2
    ).mean()


    mean_expected_home_score = (
        expected_home_score.mean()
    )


    elo_home_advantage_tests.append({
        "home_advantage_elo": home_advantage,
        "mse": mean_squared_error,
        "mean_expected_home_score": mean_expected_home_score
    })


elo_home_advantage_df = (
    pd.DataFrame(
        elo_home_advantage_tests
    )
    .sort_values(
        "mse"
    )
    .reset_index(drop=True)
)


print(
    elo_home_advantage_df.to_string(
        index=False
    )
)

print()

print(
    "Best home advantage:",
    elo_home_advantage_df.loc[
        0,
        "home_advantage_elo"
    ]
)


In [ ]:
historical_df = pd.read_csv(
    "../data/processed/historical_matches.csv"
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True
)

print(
    "Historical matches:",
    len(historical_df)
)

print(
    "Competitions:",
    historical_df["competition"].nunique()
)

print()

print(
    historical_df[
        [
            "match_id",
            "date",
            "competition",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals"
        ]
    ].head()
)

In [ ]:
# Rebuild ELO with +40 home advantage

HOME_ELO_ADVANTAGE = 40
ELO_K = 20
STARTING_ELO = 1500


# Use the full historical match dataset so Elo still
# learns from all competitions, just like our original Elo.
elo_source_df = historical_df.copy()

elo_source_df["date"] = pd.to_datetime(
    elo_source_df["date"],
    utc=True
)

elo_source_df = (
    elo_source_df
    .sort_values(
        ["date", "match_id"]
    )
    .reset_index(drop=True)
)


home_adjusted_ratings = {}

home_adjusted_elo_history = []


for _, row in elo_source_df.iterrows():

    home_team = row["home_team"]
    away_team = row["away_team"]

    home_rating = (
        home_adjusted_ratings.get(
            home_team,
            STARTING_ELO
        )
    )

    away_rating = (
        home_adjusted_ratings.get(
            away_team,
            STARTING_ELO
        )
    )


    # Home advantage is used ONLY when calculating
    # the expected result of this particular match.
    expected_home = (
        1
        / (
            1
            + 10 ** (
                (
                    away_rating
                    - (
                        home_rating
                        + HOME_ELO_ADVANTAGE
                    )
                )
                / 400
            )
        )
    )

    expected_away = (
        1
        - expected_home
    )


    if row["home_goals"] > row["away_goals"]:
        actual_home = 1.0
        actual_away = 0.0

    elif row["home_goals"] < row["away_goals"]:
        actual_home = 0.0
        actual_away = 1.0

    else:
        actual_home = 0.5
        actual_away = 0.5


    # Store PRE-MATCH ratings.
    # These are leakage-safe.
    home_adjusted_elo_history.append({
        "match_id": row["match_id"],
        "home_elo_ha": home_rating,
        "away_elo_ha": away_rating,
        "elo_difference_ha": (
            home_rating
            + HOME_ELO_ADVANTAGE
            - away_rating
        )
    })


    # Update ratings AFTER the result.
    new_home_rating = (
        home_rating
        + ELO_K
        * (
            actual_home
            - expected_home
        )
    )

    new_away_rating = (
        away_rating
        + ELO_K
        * (
            actual_away
            - expected_away
        )
    )


    home_adjusted_ratings[
        home_team
    ] = new_home_rating

    home_adjusted_ratings[
        away_team
    ] = new_away_rating


home_adjusted_elo_df = pd.DataFrame(
    home_adjusted_elo_history
)


print(
    home_adjusted_elo_df.shape
)

print()

print(
    home_adjusted_elo_df.head()
)

print()

print(
    "Teams rated:",
    len(home_adjusted_ratings)
)

In [ ]:
# Remove these columns first if this cell has already
# been run before, so we don't create duplicate columns.
columns_to_remove = [
    "home_elo_ha",
    "away_elo_ha",
    "elo_difference_ha"
]

model_df = model_df.drop(
    columns=[
        column
        for column in columns_to_remove
        if column in model_df.columns
    ]
)


# Merge the new home-adjusted Elo ratings
model_df = model_df.merge(
    home_adjusted_elo_df,
    on="match_id",
    how="left"
)


print(
    model_df[
        [
            "match_id",
            "team_elo_home",
            "team_elo_away",
            "team_elo_difference_home",
            "home_elo_ha",
            "away_elo_ha",
            "elo_difference_ha"
        ]
    ].head(10)
)

print()

print(
    "Missing home-adjusted Elo rows:",
    model_df[
        "home_elo_ha"
    ].isna().sum()
)

print(
    "Total PL matches:",
    len(model_df)
)

In [ ]:
# Compare original ELO vs home-adjusted ELO

original_elo_features = baseline_features.copy()


home_adjusted_features = [
    feature
    for feature in baseline_features
    if feature not in [
        "team_elo_home",
        "team_elo_away",
        "team_elo_difference_home"
    ]
]

home_adjusted_features += [
    "home_elo_ha",
    "away_elo_ha",
    "elo_difference_ha"
]


elo_comparison_results = []


feature_sets = {
    "Original Elo": original_elo_features,
    "Home-Adjusted Elo": home_adjusted_features
}


for feature_set_name, feature_list in feature_sets.items():

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"]
                == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"]
            < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"]
            == validation_season
        ].copy()


        X_train = train_df[
            feature_list
        ]

        y_train = train_df[
            "target"
        ]

        X_validation = validation_df[
            feature_list
        ]

        y_validation = validation_df[
            "target"
        ]


        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    criterion="gini",
                    class_weight=None,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])


        rf_pipeline.fit(
            X_train,
            y_train
        )


        probabilities = (
            rf_pipeline.predict_proba(
                X_validation
            )
        )

        predictions = (
            rf_pipeline.predict(
                X_validation
            )
        )


        season_loss = log_loss(
            y_validation,
            probabilities,
            labels=rf_pipeline[
                "model"
            ].classes_
        )

        season_accuracy = (
            predictions
            == y_validation
        ).mean()


        elo_comparison_results.append({
            "feature_set": feature_set_name,
            "season": validation_season,
            "accuracy": season_accuracy,
            "log_loss": season_loss
        })


elo_comparison_df = pd.DataFrame(
    elo_comparison_results
)


print("RESULTS BY SEASON")
print()

print(
    elo_comparison_df.to_string(
        index=False
    )
)


print()
print("AVERAGE RESULTS")
print()


elo_comparison_average_df = (
    elo_comparison_df
    .groupby("feature_set")
    .agg(
        avg_accuracy=(
            "accuracy",
            "mean"
        ),
        avg_log_loss=(
            "log_loss",
            "mean"
        )
    )
    .sort_values(
        "avg_log_loss"
    )
)


print(
    elo_comparison_average_df
)

In [ ]:
print(
    "pl_team_matches exists:",
    "pl_team_matches" in globals()
)

if "pl_team_matches" in globals():

    print(
        "Shape:",
        pl_team_matches.shape
    )

    print()

    print(
        [
            column
            for column in pl_team_matches.columns
            if (
                "elo" in column.lower()
                or column in [
                    "team",
                    "opponent",
                    "match_id",
                    "date",
                    "venue"
                ]
            )
        ]
    )

In [ ]:
opponent_strength_features = pd.read_csv(
    "../data/processed/opponent_strength_features.csv"
)


# Prevent duplicate columns if we rerun this cell
model_df = model_df.drop(
    columns=[
        column
        for column in [
            "opponent_elo_last5_pl_home",
            "opponent_elo_last5_pl_away"
        ]
        if column in model_df.columns
    ]
)


model_df = model_df.merge(
    opponent_strength_features,
    on="match_id",
    how="left"
)


print(
    model_df[
        [
            "match_id",
            "opponent_elo_last5_pl_home",
            "opponent_elo_last5_pl_away"
        ]
    ].head()
)

print()

print(
    "Total matches:",
    len(model_df)
)

print(
    "Missing home:",
    model_df[
        "opponent_elo_last5_pl_home"
    ].isna().sum()
)

print(
    "Missing away:",
    model_df[
        "opponent_elo_last5_pl_away"
    ].isna().sum()
)

In [ ]:
opponent_strength_feature_list = (
    baseline_features
    + [
        "opponent_elo_last5_pl_home",
        "opponent_elo_last5_pl_away"
    ]
)


opponent_strength_results = []


feature_sets = {
    "Original 31": baseline_features,
    "Original 31 + Opponent Strength": opponent_strength_feature_list
}


for feature_set_name, feature_list in feature_sets.items():

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"]
                == validation_season,
                "date_home"
            ]
            .min()
        )

        train_df = model_df[
            model_df["date_home"]
            < validation_start
        ].copy()

        validation_df = model_df[
            model_df["season_home"]
            == validation_season
        ].copy()


        X_train = train_df[
            feature_list
        ]

        y_train = train_df[
            "target"
        ]

        X_validation = validation_df[
            feature_list
        ]

        y_validation = validation_df[
            "target"
        ]


        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    criterion="gini",
                    class_weight=None,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])


        rf_pipeline.fit(
            X_train,
            y_train
        )


        probabilities = (
            rf_pipeline.predict_proba(
                X_validation
            )
        )

        predictions = (
            rf_pipeline.predict(
                X_validation
            )
        )


        season_loss = log_loss(
            y_validation,
            probabilities,
            labels=rf_pipeline[
                "model"
            ].classes_
        )

        season_accuracy = (
            predictions
            == y_validation
        ).mean()


        opponent_strength_results.append({
            "feature_set": feature_set_name,
            "season": validation_season,
            "accuracy": season_accuracy,
            "log_loss": season_loss
        })


opponent_strength_results_df = pd.DataFrame(
    opponent_strength_results
)


print("RESULTS BY SEASON")
print()

print(
    opponent_strength_results_df.to_string(
        index=False
    )
)


print()
print("AVERAGE RESULTS")
print()


opponent_strength_average_df = (
    opponent_strength_results_df
    .groupby("feature_set")
    .agg(
        avg_accuracy=(
            "accuracy",
            "mean"
        ),
        avg_log_loss=(
            "log_loss",
            "mean"
        )
    )
    .sort_values(
        "avg_log_loss"
    )
)


print(
    opponent_strength_average_df
)

In [ ]:
# Check how ELO behaves across seasons
# ELO currently carries a team's rating forward indefinitely.
# But teams can change between seasons through transfer, managers, aging, injuries.
# We want to see whether pullign ratings partially back toward 1500 at each new season could make them more realistic

elo_season_check = (
    model_df
    .groupby("season_home")
    .agg(
        avg_home_elo=("team_elo_home", "mean"),
        avg_away_elo=("team_elo_away", "mean"),
        min_elo=("team_elo_home", "min"),
        max_elo=("team_elo_home", "max"),
        std_elo=("team_elo_home", "std")
    )
)

print("ELO DISTRIBUTION BY SEASON")
print()
print(elo_season_check)


print()
print("FIRST MATCH ELO BY TEAM AND SEASON")
print()

first_team_elo = (
    model_df[
        [
            "season_home",
            "date_home",
            "team_home",
            "team_elo_home"
        ]
    ]
    .sort_values(
        [
            "season_home",
            "team_home",
            "date_home"
        ]
    )
    .groupby(
        [
            "season_home",
            "team_home"
        ],
        as_index=False
    )
    .first()
)

first_team_elo_summary = (
    first_team_elo
    .groupby("season_home")["team_elo_home"]
    .agg(
        ["mean", "std", "min", "max"]
    )
)

print(first_team_elo_summary)

In [ ]:
# Test season regression strengths
# Preserves most of what ELO learned about team quality while gradually forgetting older information and controlling the rating drift we see.
# Too much regression could erase real differences between Man City and a newly promoted team

print("SEASONS BY COMPETITION")
print()

season_competition_check = (
    historical_df
    .groupby("competition")["season"]
    .unique()
)

for competition, seasons in season_competition_check.items():
    print(competition)
    print(sorted(seasons))
    print()

In [ ]:
def build_season_regressed_elo(
    historical_data,
    regression_strength=0.0,
    initial_rating=1500,
    k_factor=20
):

    matches = (
        historical_data
        .copy()
        .sort_values(
            ["date", "match_id"]
        )
        .reset_index(drop=True)
    )

    ratings = {}

    elo_rows = []

    current_season = None


    for _, row in matches.iterrows():

        season = row["season"]

        home_team = row["home_team"]
        away_team = row["away_team"]


        # ---------------------------------------
        # NEW SEASON
        # ---------------------------------------

        if season != current_season:

            if current_season is not None:

                for team in ratings:

                    ratings[team] = (
                        initial_rating
                        + (
                            ratings[team]
                            - initial_rating
                        )
                        * (
                            1
                            - regression_strength
                        )
                    )

            current_season = season


        # ---------------------------------------
        # CURRENT PRE-MATCH RATINGS
        # ---------------------------------------

        home_rating = ratings.get(
            home_team,
            initial_rating
        )

        away_rating = ratings.get(
            away_team,
            initial_rating
        )


        elo_rows.append({
            "match_id": row["match_id"],
            "home_elo": home_rating,
            "away_elo": away_rating,
            "elo_difference": (
                home_rating
                - away_rating
            )
        })


        # ---------------------------------------
        # EXPECTED RESULT
        # ---------------------------------------

        expected_home = (
            1
            / (
                1
                + 10 ** (
                    (
                        away_rating
                        - home_rating
                    )
                    / 400
                )
            )
        )


        # ---------------------------------------
        # ACTUAL RESULT
        # ---------------------------------------

        if row["home_goals"] > row["away_goals"]:
            actual_home = 1.0

        elif row["home_goals"] < row["away_goals"]:
            actual_home = 0.0

        else:
            actual_home = 0.5


        # ---------------------------------------
        # UPDATE RATINGS
        # ---------------------------------------

        rating_change = (
            k_factor
            * (
                actual_home
                - expected_home
            )
        )


        ratings[home_team] = (
            home_rating
            + rating_change
        )

        ratings[away_team] = (
            away_rating
            - rating_change
        )


    return pd.DataFrame(elo_rows)


regression_strengths = [
    0.0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5
]


regressed_elo_versions = {}


for strength in regression_strengths:

    elo_version = build_season_regressed_elo(
        historical_data=historical_df,
        regression_strength=strength,
        initial_rating=1500,
        k_factor=20
    )

    regressed_elo_versions[
        strength
    ] = elo_version


    print(
        f"Regression {strength:.0%}:",
        elo_version.shape
    )

In [ ]:
# Sanity check ELO regression
# Make sure 0% reproduces the original ELO and that increasing regression actually pulls ratings towards 1500 at season boundaries.

# ---------------------------------------------------------
# 1. CHECK THAT 0% REGRESSION MATCHES ORIGINAL PL ELO
# ---------------------------------------------------------

original_elo_check = (
    model_df[
        [
            "match_id",
            "season_home",
            "team_elo_home",
            "team_elo_away"
        ]
    ]
    .merge(
        regressed_elo_versions[0.0][
            [
                "match_id",
                "home_elo",
                "away_elo"
            ]
        ],
        on="match_id",
        how="left"
    )
)


original_elo_check[
    "home_difference"
] = (
    original_elo_check["team_elo_home"]
    - original_elo_check["home_elo"]
)

original_elo_check[
    "away_difference"
] = (
    original_elo_check["team_elo_away"]
    - original_elo_check["away_elo"]
)


print("0% REGRESSION SANITY CHECK")
print()

print(
    "Maximum home Elo difference:",
    original_elo_check[
        "home_difference"
    ].abs().max()
)

print(
    "Maximum away Elo difference:",
    original_elo_check[
        "away_difference"
    ].abs().max()
)


# ---------------------------------------------------------
# 2. COMPARE START-OF-SEASON PL ELO DISTRIBUTIONS
# ---------------------------------------------------------

comparison_rows = []


for strength in [
    0.0,
    0.2,
    0.5
]:

    elo_version = (
        historical_df[
            [
                "match_id",
                "date",
                "season",
                "competition",
                "home_team"
            ]
        ]
        .merge(
            regressed_elo_versions[strength][
                [
                    "match_id",
                    "home_elo"
                ]
            ],
            on="match_id",
            how="left"
        )
    )


    pl_elo = elo_version[
        elo_version["competition"]
        == "premier_league"
    ].copy()


    first_team_rating = (
        pl_elo
        .sort_values(
            [
                "season",
                "home_team",
                "date"
            ]
        )
        .groupby(
            [
                "season",
                "home_team"
            ],
            as_index=False
        )
        .first()
    )


    summary = (
        first_team_rating
        .groupby("season")["home_elo"]
        .agg(
            [
                "mean",
                "std",
                "min",
                "max"
            ]
        )
        .reset_index()
    )


    summary[
        "regression"
    ] = f"{strength:.0%}"


    comparison_rows.append(
        summary
    )


season_regression_check = pd.concat(
    comparison_rows,
    ignore_index=True
)


print()
print("START-OF-SEASON PL ELO")
print()

print(
    season_regression_check[
        [
            "regression",
            "season",
            "mean",
            "std",
            "min",
            "max"
        ]
    ].to_string(
        index=False
    )
)

In [ ]:
# Test all ELO regression strengths
# Lower regression preserves long-term team strength
# Higher regression adapts faster to season-to-season changes

season_regression_results = []


for strength in regression_strengths:

    # -------------------------------------------
    # GET THIS VERSION OF ELO
    # -------------------------------------------

    elo_version = (
        regressed_elo_versions[strength]
        .rename(
            columns={
                "home_elo": "reg_home_elo",
                "away_elo": "reg_away_elo",
                "elo_difference": "reg_elo_difference"
            }
        )
    )


    # -------------------------------------------
    # MERGE INTO MODEL DATA
    # -------------------------------------------

    experiment_df = (
        model_df
        .drop(
            columns=[
                column
                for column in [
                    "reg_home_elo",
                    "reg_away_elo",
                    "reg_elo_difference"
                ]
                if column in model_df.columns
            ]
        )
        .merge(
            elo_version[
                [
                    "match_id",
                    "reg_home_elo",
                    "reg_away_elo",
                    "reg_elo_difference"
                ]
            ],
            on="match_id",
            how="left"
        )
    )


    # -------------------------------------------
    # REPLACE ORIGINAL ELO FEATURES
    # -------------------------------------------

    regression_features = [
        feature
        for feature in baseline_features
        if feature not in [
            "team_elo_home",
            "team_elo_away",
            "team_elo_difference_home"
        ]
    ]


    regression_features += [
        "reg_home_elo",
        "reg_away_elo",
        "reg_elo_difference"
    ]


    # -------------------------------------------
    # WALK-FORWARD VALIDATION
    # -------------------------------------------

    for validation_season in walk_forward_seasons:

        validation_start = (
            experiment_df.loc[
                experiment_df["season_home"]
                == validation_season,
                "date_home"
            ]
            .min()
        )


        train_df = experiment_df[
            experiment_df["date_home"]
            < validation_start
        ].copy()


        validation_df = experiment_df[
            experiment_df["season_home"]
            == validation_season
        ].copy()


        X_train = train_df[
            regression_features
        ]

        y_train = train_df[
            "target"
        ]


        X_validation = validation_df[
            regression_features
        ]

        y_validation = validation_df[
            "target"
        ]


        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    criterion="gini",
                    class_weight=None,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])


        rf_pipeline.fit(
            X_train,
            y_train
        )


        probabilities = rf_pipeline.predict_proba(
            X_validation
        )

        predictions = rf_pipeline.predict(
            X_validation
        )


        season_loss = log_loss(
            y_validation,
            probabilities,
            labels=rf_pipeline[
                "model"
            ].classes_
        )


        season_accuracy = (
            predictions
            == y_validation
        ).mean()


        season_regression_results.append({
            "regression": f"{strength:.0%}",
            "strength": strength,
            "season": validation_season,
            "accuracy": season_accuracy,
            "log_loss": season_loss
        })


# -------------------------------------------
# RESULTS
# -------------------------------------------

season_regression_results_df = pd.DataFrame(
    season_regression_results
)


print("RESULTS BY SEASON")
print()

print(
    season_regression_results_df[
        [
            "regression",
            "season",
            "accuracy",
            "log_loss"
        ]
    ].to_string(
        index=False
    )
)


print()
print("AVERAGE RESULTS")
print()


season_regression_average_df = (
    season_regression_results_df
    .groupby(
        [
            "regression",
            "strength"
        ],
        as_index=False
    )
    .agg(
        avg_accuracy=(
            "accuracy",
            "mean"
        ),
        avg_log_loss=(
            "log_loss",
            "mean"
        )
    )
    .sort_values(
        "avg_log_loss"
    )
)


print(
    season_regression_average_df.to_string(
        index=False
    )
)

In [ ]:
# Merge and walk-forward test
# points_lat5_pl tells us what happened, this feature tells us whether the results were better or worse than we'd expect given the teams' relative strength

# ---------------------------------------------------------
# LOAD AND MERGE ELO PERFORMANCE FEATURES
# ---------------------------------------------------------

elo_performance_features = pd.read_csv(
    "../data/processed/elo_performance_features.csv"
)


model_df = model_df.drop(
    columns=[
        column
        for column in [
            "elo_performance_last5_pl_home",
            "elo_performance_last5_pl_away"
        ]
        if column in model_df.columns
    ]
)


model_df = model_df.merge(
    elo_performance_features,
    on="match_id",
    how="left"
)


print(
    "Missing home:",
    model_df[
        "elo_performance_last5_pl_home"
    ].isna().sum()
)

print(
    "Missing away:",
    model_df[
        "elo_performance_last5_pl_away"
    ].isna().sum()
)


# ---------------------------------------------------------
# FEATURE SETS
# ---------------------------------------------------------

elo_performance_feature_list = (
    baseline_features
    + [
        "elo_performance_last5_pl_home",
        "elo_performance_last5_pl_away"
    ]
)


feature_sets = {
    "Original 31": baseline_features,

    "Original 31 + Elo Performance":
        elo_performance_feature_list
}


# ---------------------------------------------------------
# WALK-FORWARD TEST
# ---------------------------------------------------------

elo_performance_results = []


for feature_set_name, feature_list in feature_sets.items():

    for validation_season in walk_forward_seasons:

        validation_start = (
            model_df.loc[
                model_df["season_home"]
                == validation_season,
                "date_home"
            ]
            .min()
        )


        train_df = model_df[
            model_df["date_home"]
            < validation_start
        ].copy()


        validation_df = model_df[
            model_df["season_home"]
            == validation_season
        ].copy()


        X_train = train_df[
            feature_list
        ]

        y_train = train_df[
            "target"
        ]


        X_validation = validation_df[
            feature_list
        ]

        y_validation = validation_df[
            "target"
        ]


        rf_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    criterion="gini",
                    class_weight=None,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])


        rf_pipeline.fit(
            X_train,
            y_train
        )


        probabilities = (
            rf_pipeline.predict_proba(
                X_validation
            )
        )


        predictions = (
            rf_pipeline.predict(
                X_validation
            )
        )


        season_loss = log_loss(
            y_validation,
            probabilities,
            labels=rf_pipeline[
                "model"
            ].classes_
        )


        season_accuracy = (
            predictions
            == y_validation
        ).mean()


        elo_performance_results.append({
            "feature_set": feature_set_name,
            "season": validation_season,
            "accuracy": season_accuracy,
            "log_loss": season_loss
        })


# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

elo_performance_results_df = pd.DataFrame(
    elo_performance_results
)


print()
print("RESULTS BY SEASON")
print()

print(
    elo_performance_results_df.to_string(
        index=False
    )
)


print()
print("AVERAGE RESULTS")
print()


elo_performance_average_df = (
    elo_performance_results_df
    .groupby("feature_set")
    .agg(
        avg_accuracy=(
            "accuracy",
            "mean"
        ),
        avg_log_loss=(
            "log_loss",
            "mean"
        )
    )
    .sort_values(
        "avg_log_loss"
    )
)


print(
    elo_performance_average_df
)

# A missing piece is goals-based team strength. 
# Build a separate attacking and defensive strength ratings that update sequentially

In [ ]:
# Inspect goal distributions
# Before designing attack/defense ratings, we need to understand the distribution we're trying to model

pl_goal_check = (
    historical_df[
        historical_df["competition"]
        == "premier_league"
    ]
    .copy()
)


print("PREMIER LEAGUE GOAL SUMMARY")
print()

print(
    "Matches:",
    len(pl_goal_check)
)

print(
    "Average home goals:",
    pl_goal_check[
        "home_goals"
    ].mean()
)

print(
    "Average away goals:",
    pl_goal_check[
        "away_goals"
    ].mean()
)

print(
    "Average total goals:",
    (
        pl_goal_check["home_goals"]
        + pl_goal_check["away_goals"]
    ).mean()
)


print()
print("BY SEASON")
print()

goal_by_season = (
    pl_goal_check
    .groupby("season")
    .agg(
        matches=(
            "match_id",
            "count"
        ),
        avg_home_goals=(
            "home_goals",
            "mean"
        ),
        avg_away_goals=(
            "away_goals",
            "mean"
        )
    )
)

goal_by_season[
    "avg_total_goals"
] = (
    goal_by_season["avg_home_goals"]
    + goal_by_season["avg_away_goals"]
)

print(goal_by_season)


print()
print("HOME GOAL FREQUENCIES")
print()

print(
    pl_goal_check[
        "home_goals"
    ]
    .value_counts()
    .sort_index()
)


print()
print("AWAY GOAL FREQUENCIES")
print()

print(
    pl_goal_check[
        "away_goals"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# Build towards a Poisson-based model
# A Poisson regression works more naturally with one row representing one team's goal count.
# Each PL match becomes 2 observations: one for the home team's goals and one for the away team's goals.
# This way the model can learn team-specfici attack and defensive effects

poisson_rows = []


for _, row in pl_goal_check.iterrows():

    # Home team's scoring observation
    poisson_rows.append({
        "match_id": row["match_id"],
        "date": row["date"],
        "season": row["season"],
        "team": row["home_team"],
        "opponent": row["away_team"],
        "is_home": 1,
        "goals": row["home_goals"]
    })


    # Away team's scoring observation
    poisson_rows.append({
        "match_id": row["match_id"],
        "date": row["date"],
        "season": row["season"],
        "team": row["away_team"],
        "opponent": row["home_team"],
        "is_home": 0,
        "goals": row["away_goals"]
    })


poisson_df = pd.DataFrame(
    poisson_rows
)


poisson_df["date"] = pd.to_datetime(
    poisson_df["date"],
    utc=True
)


poisson_df = (
    poisson_df
    .sort_values(
        [
            "date",
            "match_id",
            "is_home"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Shape:",
    poisson_df.shape
)

print()

print(
    poisson_df.head(10)
)

print()

print(
    "Missing goals:",
    poisson_df["goals"].isna().sum()
)

print(
    "Unique matches:",
    poisson_df["match_id"].nunique()
)

print(
    "Unique teams:",
    poisson_df["team"].nunique()
)

print()

print(
    poisson_df.groupby(
        "is_home"
    )["goals"].agg(
        [
            "count",
            "mean",
            "std"
        ]
    )
)

In [ ]:
# Fit the Poisson goal model
# Confirm the model can learn separate attacking strength, defensive strength, and home advantage using only past data
# Train on 2020/21 and validate on 2021/22 

import statsmodels.api as sm
import statsmodels.formula.api as smf


# ---------------------------------------------------------
# FIRST POISSON TEST
# Train: 2020/21
# Validate: 2021/22
# ---------------------------------------------------------

poisson_train = poisson_df[
    poisson_df["season"] == "2020_21"
].copy()


poisson_validation = poisson_df[
    poisson_df["season"] == "2021_22"
].copy()


print(
    "Training observations:",
    len(poisson_train)
)

print(
    "Validation observations:",
    len(poisson_validation)
)


# ---------------------------------------------------------
# FIT POISSON MODEL
# ---------------------------------------------------------

poisson_model = smf.glm(
    formula=(
        "goals ~ is_home "
        "+ C(team) "
        "+ C(opponent)"
    ),
    data=poisson_train,
    family=sm.families.Poisson()
).fit()


print()
print(
    "Model fitted successfully."
)

print(
    "Training mean goals:",
    poisson_train["goals"].mean()
)

print(
    "Training predicted mean:",
    poisson_model.predict(
        poisson_train
    ).mean()
)


print()
print(
    poisson_model.params[
        ["Intercept", "is_home"]
    ]
)

In [ ]:
# ---------------------------------------------------------
# POISSON DIAGNOSTIC
# Train: everything before 2024/25
# Validate: 2024/25
# ---------------------------------------------------------

validation_season = "2024_25"


validation_start = (
    poisson_df.loc[
        poisson_df["season"] == validation_season,
        "date"
    ]
    .min()
)


poisson_train_2024 = (
    poisson_df[
        poisson_df["date"] < validation_start
    ]
    .copy()
)


poisson_validation_2024 = (
    poisson_df[
        poisson_df["season"] == validation_season
    ]
    .copy()
)


# ---------------------------------------------------------
# CHECK TEAM COVERAGE
# ---------------------------------------------------------

training_teams_2024 = set(
    poisson_train_2024["team"].unique()
)


validation_teams_2024 = set(
    poisson_validation_2024["team"].unique()
)


unseen_teams_2024 = sorted(
    validation_teams_2024
    - training_teams_2024
)


print(
    "Training observations:",
    len(poisson_train_2024)
)

print(
    "Training matches:",
    poisson_train_2024["match_id"].nunique()
)

print(
    "Validation matches:",
    poisson_validation_2024["match_id"].nunique()
)

print()

print(
    "Unseen teams:",
    unseen_teams_2024
)


# ---------------------------------------------------------
# FIT MODEL
# ---------------------------------------------------------

poisson_model_2024 = smf.glm(
    formula=(
        "goals ~ is_home "
        "+ C(team) "
        "+ C(opponent)"
    ),
    data=poisson_train_2024,
    family=sm.families.Poisson()
).fit()


print()
print("MODEL FITTED")

print(
    "Training mean goals:",
    poisson_train_2024["goals"].mean()
)

print(
    "Training predicted mean:",
    poisson_model_2024.predict(
        poisson_train_2024
    ).mean()
)

print()

print(
    poisson_model_2024.params[
        ["Intercept", "is_home"]
    ]
)


# ---------------------------------------------------------
# COUNT USABLE VALIDATION MATCHES
# ---------------------------------------------------------

known_validation_2024 = (
    poisson_validation_2024[
        poisson_validation_2024["team"].isin(
            training_teams_2024
        )
        &
        poisson_validation_2024["opponent"].isin(
            training_teams_2024
        )
    ]
    .copy()
)


usable_match_ids = (
    known_validation_2024[
        "match_id"
    ]
    .value_counts()
)


usable_match_ids = (
    usable_match_ids[
        usable_match_ids == 2
    ]
    .index
)


print()
print(
    "Usable validation matches:",
    len(usable_match_ids)
)

print(
    "Temporarily excluded matches:",
    380 - len(usable_match_ids)
)

In [ ]:
# Turn expected goals into match probabilities
# Poisson model predicts an expected number of goals (lambda) for each team, not directly H/D/A.
# We can use those two expected-goals values to calculate the probabiliy of every possible scoreline, sum them into home-win, draw, and away-win probabiltiies.

from scipy.stats import poisson


# ---------------------------------------------------------
# KEEP ONLY THE 342 USABLE MATCHES
# ---------------------------------------------------------

poisson_validation_usable = (
    known_validation_2024[
        known_validation_2024["match_id"].isin(
            usable_match_ids
        )
    ]
    .copy()
)


# ---------------------------------------------------------
# PREDICT EXPECTED GOALS
# ---------------------------------------------------------

poisson_validation_usable[
    "expected_goals"
] = poisson_model_2024.predict(
    poisson_validation_usable
)


# ---------------------------------------------------------
# SPLIT HOME AND AWAY ROWS
# ---------------------------------------------------------

poisson_home_predictions = (
    poisson_validation_usable[
        poisson_validation_usable["is_home"] == 1
    ][
        [
            "match_id",
            "team",
            "opponent",
            "goals",
            "expected_goals"
        ]
    ]
    .rename(
        columns={
            "team": "home_team",
            "opponent": "away_team",
            "goals": "home_goals",
            "expected_goals": "expected_home_goals"
        }
    )
)


poisson_away_predictions = (
    poisson_validation_usable[
        poisson_validation_usable["is_home"] == 0
    ][
        [
            "match_id",
            "team",
            "opponent",
            "goals",
            "expected_goals"
        ]
    ]
    .rename(
        columns={
            "team": "away_team",
            "opponent": "home_team",
            "goals": "away_goals",
            "expected_goals": "expected_away_goals"
        }
    )
)


poisson_match_predictions = (
    poisson_home_predictions
    .merge(
        poisson_away_predictions,
        on=[
            "match_id",
            "home_team",
            "away_team"
        ],
        how="inner"
    )
)


# ---------------------------------------------------------
# CONVERT EXPECTED GOALS INTO H/D/A PROBABILITIES
# ---------------------------------------------------------

probability_rows = []


for _, row in poisson_match_predictions.iterrows():

    home_lambda = row[
        "expected_home_goals"
    ]

    away_lambda = row[
        "expected_away_goals"
    ]


    home_goal_probs = poisson.pmf(
        np.arange(0, 11),
        home_lambda
    )

    away_goal_probs = poisson.pmf(
        np.arange(0, 11),
        away_lambda
    )


    score_matrix = np.outer(
        home_goal_probs,
        away_goal_probs
    )


    prob_home = np.tril(
        score_matrix,
        -1
    ).sum()

    prob_draw = np.trace(
        score_matrix
    )

    prob_away = np.triu(
        score_matrix,
        1
    ).sum()


    total_probability = (
        prob_home
        + prob_draw
        + prob_away
    )


    # Normalize the tiny amount of probability
    # beyond 10 goals per team.
    prob_home /= total_probability
    prob_draw /= total_probability
    prob_away /= total_probability


    if row["home_goals"] > row["away_goals"]:
        actual = "H"

    elif row["home_goals"] < row["away_goals"]:
        actual = "A"

    else:
        actual = "D"


    probability_rows.append({
        "match_id": row["match_id"],
        "home_team": row["home_team"],
        "away_team": row["away_team"],
        "expected_home_goals": home_lambda,
        "expected_away_goals": away_lambda,
        "actual": actual,
        "prob_H": prob_home,
        "prob_D": prob_draw,
        "prob_A": prob_away
    })


poisson_predictions_2024 = pd.DataFrame(
    probability_rows
)


# ---------------------------------------------------------
# EVALUATE
# ---------------------------------------------------------

probability_matrix = (
    poisson_predictions_2024[
        [
            "prob_A",
            "prob_D",
            "prob_H"
        ]
    ]
    .values
)


poisson_logloss = log_loss(
    poisson_predictions_2024["actual"],
    probability_matrix,
    labels=[
        "A",
        "D",
        "H"
    ]
)


poisson_predicted_result = (
    poisson_predictions_2024[
        [
            "prob_A",
            "prob_D",
            "prob_H"
        ]
    ]
    .idxmax(
        axis=1
    )
    .str.replace(
        "prob_",
        "",
        regex=False
    )
)


poisson_accuracy = (
    poisson_predicted_result
    == poisson_predictions_2024["actual"]
).mean()


print(
    "Matches evaluated:",
    len(poisson_predictions_2024)
)

print(
    "Accuracy:",
    poisson_accuracy
)

print(
    "Log loss:",
    poisson_logloss
)


print()
print("AVERAGE EXPECTED GOALS")
print()

print(
    "Home:",
    poisson_predictions_2024[
        "expected_home_goals"
    ].mean()
)

print(
    "Away:",
    poisson_predictions_2024[
        "expected_away_goals"
    ].mean()
)


print()
print("AVERAGE PREDICTED PROBABILITIES")
print()

print(
    poisson_predictions_2024[
        [
            "prob_H",
            "prob_D",
            "prob_A"
        ]
    ].mean()
)

In [ ]:
# Compare the RF model against Poisson model on the exact same matches

# ---------------------------------------------------------
# TRAIN RF FOR 2024/25
# ---------------------------------------------------------

validation_start = (
    model_df.loc[
        model_df["season_home"] == "2024_25",
        "date_home"
    ]
    .min()
)


rf_train_2024 = (
    model_df[
        model_df["date_home"] < validation_start
    ]
    .copy()
)


rf_validation_2024 = (
    model_df[
        model_df["season_home"] == "2024_25"
    ]
    .copy()
)


rf_pipeline_2024 = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=10,
            max_features="sqrt",
            criterion="gini",
            class_weight=None,
            random_state=42,
            n_jobs=-1
        )
    )
])


rf_pipeline_2024.fit(
    rf_train_2024[baseline_features],
    rf_train_2024["target"]
)


rf_probabilities_2024 = (
    rf_pipeline_2024.predict_proba(
        rf_validation_2024[baseline_features]
    )
)


rf_classes = (
    rf_pipeline_2024[
        "model"
    ].classes_
)


rf_predictions_2024 = pd.DataFrame({
    "match_id": rf_validation_2024["match_id"].values
})


for class_index, class_name in enumerate(rf_classes):

    rf_predictions_2024[
        f"rf_prob_{class_name}"
    ] = (
        rf_probabilities_2024[
            :,
            class_index
        ]
    )


# ---------------------------------------------------------
# MERGE RF AND POISSON
# ---------------------------------------------------------

comparison_2024 = (
    poisson_predictions_2024
    .merge(
        rf_predictions_2024,
        on="match_id",
        how="inner"
    )
)


# ---------------------------------------------------------
# RF LOG LOSS ON EXACT SAME 342 MATCHES
# ---------------------------------------------------------

rf_probability_matrix = (
    comparison_2024[
        [
            "rf_prob_A",
            "rf_prob_D",
            "rf_prob_H"
        ]
    ]
    .values
)


rf_same_matches_loss = log_loss(
    comparison_2024["actual"],
    rf_probability_matrix,
    labels=[
        "A",
        "D",
        "H"
    ]
)


rf_predicted_result = (
    comparison_2024[
        [
            "rf_prob_A",
            "rf_prob_D",
            "rf_prob_H"
        ]
    ]
    .idxmax(
        axis=1
    )
    .str.replace(
        "rf_prob_",
        "",
        regex=False
    )
)


rf_same_matches_accuracy = (
    rf_predicted_result
    == comparison_2024["actual"]
).mean()


# ---------------------------------------------------------
# SIMPLE 50/50 ENSEMBLE
# ---------------------------------------------------------

for result in [
    "H",
    "D",
    "A"
]:

    comparison_2024[
        f"ensemble_prob_{result}"
    ] = (
        0.5
        * comparison_2024[
            f"rf_prob_{result}"
        ]
        +
        0.5
        * comparison_2024[
            f"prob_{result}"
        ]
    )


ensemble_probability_matrix = (
    comparison_2024[
        [
            "ensemble_prob_A",
            "ensemble_prob_D",
            "ensemble_prob_H"
        ]
    ]
    .values
)


ensemble_loss = log_loss(
    comparison_2024["actual"],
    ensemble_probability_matrix,
    labels=[
        "A",
        "D",
        "H"
    ]
)


ensemble_predicted_result = (
    comparison_2024[
        [
            "ensemble_prob_A",
            "ensemble_prob_D",
            "ensemble_prob_H"
        ]
    ]
    .idxmax(
        axis=1
    )
    .str.replace(
        "ensemble_prob_",
        "",
        regex=False
    )
)


ensemble_accuracy = (
    ensemble_predicted_result
    == comparison_2024["actual"]
).mean()


# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

print(
    "Matches:",
    len(comparison_2024)
)

print()

print("RANDOM FOREST")
print(
    "Accuracy:",
    rf_same_matches_accuracy
)
print(
    "Log loss:",
    rf_same_matches_loss
)

print()

print("POISSON")
print(
    "Accuracy:",
    poisson_accuracy
)
print(
    "Log loss:",
    poisson_logloss
)

print()

print("50/50 ENSEMBLE")
print(
    "Accuracy:",
    ensemble_accuracy
)
print(
    "Log loss:",
    ensemble_loss
)

In [ ]:
# ---------------------------------------------------------
# LOAD EUROPEAN PERFORMANCE FEATURES
# ---------------------------------------------------------

europe_features_df = pd.read_csv(
    "../data/processed/european_performance_features.csv"
)


print(
    "European feature shape:",
    europe_features_df.shape
)

print(
    "Duplicate match IDs:",
    europe_features_df[
        "match_id"
    ].duplicated().sum()
)


# ---------------------------------------------------------
# KEEP ONLY THE FEATURES WE WANT TO TEST
# ---------------------------------------------------------

europe_features_to_merge = (
    europe_features_df[
        [
            "match_id",
            "recent_europe_home",
            "recent_europe_away",
            "european_performance_recent_home",
            "european_performance_recent_away"
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# MERGE INTO MODEL DATA
# ---------------------------------------------------------

# Remove these columns first if this cell is rerun
existing_europe_cols = [
    col
    for col in europe_features_to_merge.columns
    if (
        col != "match_id"
        and col in model_df.columns
    )
]


if existing_europe_cols:

    model_df = model_df.drop(
        columns=existing_europe_cols
    )


model_df = model_df.merge(
    europe_features_to_merge,
    on="match_id",
    how="left"
)


# ---------------------------------------------------------
# CHECK MERGE
# ---------------------------------------------------------

print()
print(
    "Model shape after merge:",
    model_df.shape
)

print(
    "Unique matches:",
    model_df[
        "match_id"
    ].nunique()
)

print()

print("MISSING VALUES")
print()

print(
    model_df[
        [
            "recent_europe_home",
            "recent_europe_away",
            "european_performance_recent_home",
            "european_performance_recent_away"
        ]
    ]
    .isna()
    .sum()
)

In [ ]:
# ---------------------------------------------------------
# EUROPEAN FEATURES
# ---------------------------------------------------------

european_features = [
    "recent_europe_home",
    "recent_europe_away",
    "european_performance_recent_home",
    "european_performance_recent_away"
]


features_with_europe = (
    baseline_features
    + european_features
)


# ---------------------------------------------------------
# WALK-FORWARD COMPARISON
# ---------------------------------------------------------

comparison_results = []


for feature_name, feature_set in [
    (
        "Original 31",
        baseline_features
    ),
    (
        "Original 31 + Europe",
        features_with_europe
    )
]:

    season_results = []


    for validation_season in [
        "2021_22",
        "2022_23",
        "2023_24",
        "2024_25"
    ]:

        validation_start = (
            model_df.loc[
                model_df["season_home"]
                == validation_season,
                "date_home"
            ]
            .min()
        )


        train_df = model_df[
            model_df["date_home"]
            < validation_start
        ].copy()


        validation_df = model_df[
            model_df["season_home"]
            == validation_season
        ].copy()


        X_train = train_df[
            feature_set
        ]

        y_train = train_df[
            "target"
        ]


        X_validation = validation_df[
            feature_set
        ]

        y_validation = validation_df[
            "target"
        ]


        # ---------------------------------------------
        # IMPUTATION
        # ---------------------------------------------

        imputer = SimpleImputer(
            strategy="median",
            add_indicator=True
        )


        X_train_imputed = (
            imputer.fit_transform(
                X_train
            )
        )


        X_validation_imputed = (
            imputer.transform(
                X_validation
            )
        )


        # ---------------------------------------------
        # TUNED RANDOM FOREST
        # ---------------------------------------------

        rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=10,
            max_features="sqrt",
            criterion="gini",
            class_weight=None,
            random_state=42,
            n_jobs=-1
        )


        rf.fit(
            X_train_imputed,
            y_train
        )


        probabilities = rf.predict_proba(
            X_validation_imputed
        )


        predictions = rf.predict(
            X_validation_imputed
        )


        season_accuracy = accuracy_score(
            y_validation,
            predictions
        )


        season_log_loss = log_loss(
            y_validation,
            probabilities,
            labels=rf.classes_
        )


        season_results.append({
            "season": validation_season,
            "accuracy": season_accuracy,
            "log_loss": season_log_loss
        })


    results_df = pd.DataFrame(
        season_results
    )


    comparison_results.append({
        "model": feature_name,
        "average_accuracy":
            results_df["accuracy"].mean(),
        "average_log_loss":
            results_df["log_loss"].mean()
    })


    print()
    print(feature_name)
    print(
        results_df.to_string(
            index=False
        )
    )

    print(
        "Average accuracy:",
        results_df["accuracy"].mean()
    )

    print(
        "Average log loss:",
        results_df["log_loss"].mean()
    )


print()
print("FINAL COMPARISON")
print()

comparison_df = pd.DataFrame(
    comparison_results
)

print(
    comparison_df.to_string(
        index=False
    )
)

In [ ]:
# ---------------------------------------------------------
# EUROPEAN FEATURE ABLATION
# ---------------------------------------------------------

europe_feature_sets = {
    
    "Original 31":
        baseline_features,

    "Recent Europe Only":
        baseline_features
        + [
            "recent_europe_home",
            "recent_europe_away"
        ],

    "Europe Performance Only":
        baseline_features
        + [
            "european_performance_recent_home",
            "european_performance_recent_away"
        ],

    "Recent + Performance":
        baseline_features
        + [
            "recent_europe_home",
            "recent_europe_away",
            "european_performance_recent_home",
            "european_performance_recent_away"
        ]
}


europe_ablation_results = []


for feature_name, feature_set in europe_feature_sets.items():

    season_results = []


    for validation_season in [
        "2021_22",
        "2022_23",
        "2023_24",
        "2024_25"
    ]:

        validation_start = (
            model_df.loc[
                model_df["season_home"]
                == validation_season,
                "date_home"
            ]
            .min()
        )


        train_df = model_df[
            model_df["date_home"]
            < validation_start
        ].copy()


        validation_df = model_df[
            model_df["season_home"]
            == validation_season
        ].copy()


        X_train = train_df[
            feature_set
        ]

        y_train = train_df[
            "target"
        ]


        X_validation = validation_df[
            feature_set
        ]

        y_validation = validation_df[
            "target"
        ]


        # ---------------------------------------------
        # IMPUTATION
        # ---------------------------------------------

        imputer = SimpleImputer(
            strategy="median",
            add_indicator=True
        )


        X_train_imputed = (
            imputer.fit_transform(
                X_train
            )
        )


        X_validation_imputed = (
            imputer.transform(
                X_validation
            )
        )


        # ---------------------------------------------
        # TUNED RANDOM FOREST
        # ---------------------------------------------

        rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=10,
            max_features="sqrt",
            criterion="gini",
            class_weight=None,
            random_state=42,
            n_jobs=-1
        )


        rf.fit(
            X_train_imputed,
            y_train
        )


        probabilities = rf.predict_proba(
            X_validation_imputed
        )


        predictions = rf.predict(
            X_validation_imputed
        )


        season_results.append({
            "season": validation_season,

            "accuracy": accuracy_score(
                y_validation,
                predictions
            ),

            "log_loss": log_loss(
                y_validation,
                probabilities,
                labels=rf.classes_
            )
        })


    results_df = pd.DataFrame(
        season_results
    )


    europe_ablation_results.append({
        "model": feature_name,

        "average_accuracy":
            results_df["accuracy"].mean(),

        "average_log_loss":
            results_df["log_loss"].mean()
    })


    print()
    print(feature_name)

    print(
        results_df.to_string(
            index=False
        )
    )

    print(
        "Average accuracy:",
        results_df["accuracy"].mean()
    )

    print(
        "Average log loss:",
        results_df["log_loss"].mean()
    )


print()
print("FINAL EUROPEAN FEATURE ABLATION")
print()

europe_ablation_df = (
    pd.DataFrame(
        europe_ablation_results
    )
    .sort_values(
        "average_log_loss"
    )
)

print(
    europe_ablation_df.to_string(
        index=False
    )
)

In [ ]:
# Test new added features

# ---------------------------------------------------------
# LOAD DOMINANCE FEATURES
# ---------------------------------------------------------

dominance_features_df = pd.read_csv(
    "../data/processed/dominance_features.csv"
)


print(
    "Dominance feature shape:",
    dominance_features_df.shape
)

print(
    "Duplicate match IDs:",
    dominance_features_df[
        "match_id"
    ].duplicated().sum()
)


# ---------------------------------------------------------
# REMOVE OLD COLUMNS IF CELL IS RERUN
# ---------------------------------------------------------

dominance_columns = [
    col
    for col in dominance_features_df.columns
    if (
        col != "match_id"
        and col in model_df.columns
    )
]


if dominance_columns:

    model_df = model_df.drop(
        columns=dominance_columns
    )


# ---------------------------------------------------------
# MERGE
# ---------------------------------------------------------

model_df = model_df.merge(
    dominance_features_df,
    on="match_id",
    how="left"
)


print()
print(
    "Model rows:",
    len(model_df)
)

print(
    "Unique matches:",
    model_df[
        "match_id"
    ].nunique()
)

print()

print("MISSING VALUES")
print()

print(
    model_df[
        [
            "shot_difference_last5_all_home",
            "shot_difference_last5_all_away",
            "possession_difference_last5_all_home",
            "possession_difference_last5_all_away",
            "corner_difference_last5_all_home",
            "corner_difference_last5_all_away"
        ]
    ]
    .isna()
    .sum()
)

In [ ]:
# Compare the feature sets

# ---------------------------------------------------------
# DOMINANCE FEATURE SETS
# ---------------------------------------------------------

dominance_feature_sets = {

    "Original 31":
        baseline_features,

    "+ Shots":
        baseline_features
        + [
            "shot_difference_last5_all_home",
            "shot_difference_last5_all_away"
        ],

    "+ Possession":
        baseline_features
        + [
            "possession_difference_last5_all_home",
            "possession_difference_last5_all_away"
        ],

    "+ Corners":
        baseline_features
        + [
            "corner_difference_last5_all_home",
            "corner_difference_last5_all_away"
        ],

    "+ All Dominance":
        baseline_features
        + [
            "shot_difference_last5_all_home",
            "shot_difference_last5_all_away",
            "possession_difference_last5_all_home",
            "possession_difference_last5_all_away",
            "corner_difference_last5_all_home",
            "corner_difference_last5_all_away"
        ]
}


dominance_results = []


# ---------------------------------------------------------
# WALK-FORWARD TEST
# ---------------------------------------------------------

for feature_name, feature_set in dominance_feature_sets.items():

    season_results = []


    for validation_season in [
        "2021_22",
        "2022_23",
        "2023_24",
        "2024_25"
    ]:

        validation_start = (
            model_df.loc[
                model_df["season_home"]
                == validation_season,
                "date_home"
            ]
            .min()
        )


        train_df = model_df[
            model_df["date_home"]
            < validation_start
        ].copy()


        validation_df = model_df[
            model_df["season_home"]
            == validation_season
        ].copy()


        X_train = train_df[
            feature_set
        ]

        y_train = train_df[
            "target"
        ]


        X_validation = validation_df[
            feature_set
        ]

        y_validation = validation_df[
            "target"
        ]


        # ---------------------------------------------
        # IMPUTER
        # ---------------------------------------------

        imputer = SimpleImputer(
            strategy="median",
            add_indicator=True
        )


        X_train_imputed = (
            imputer.fit_transform(
                X_train
            )
        )


        X_validation_imputed = (
            imputer.transform(
                X_validation
            )
        )


        # ---------------------------------------------
        # TUNED RANDOM FOREST
        # ---------------------------------------------

        rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=10,
            max_features="sqrt",
            criterion="gini",
            class_weight=None,
            random_state=42,
            n_jobs=-1
        )


        rf.fit(
            X_train_imputed,
            y_train
        )


        probabilities = rf.predict_proba(
            X_validation_imputed
        )


        predictions = rf.predict(
            X_validation_imputed
        )


        season_results.append({

            "season":
                validation_season,

            "accuracy":
                accuracy_score(
                    y_validation,
                    predictions
                ),

            "log_loss":
                log_loss(
                    y_validation,
                    probabilities,
                    labels=rf.classes_
                )
        })


    results_df = pd.DataFrame(
        season_results
    )


    dominance_results.append({

        "model":
            feature_name,

        "average_accuracy":
            results_df[
                "accuracy"
            ].mean(),

        "average_log_loss":
            results_df[
                "log_loss"
            ].mean()
    })


    print()
    print(feature_name)

    print(
        results_df.to_string(
            index=False
        )
    )

    print(
        "Average accuracy:",
        results_df[
            "accuracy"
        ].mean()
    )

    print(
        "Average log loss:",
        results_df[
            "log_loss"
        ].mean()
    )


# ---------------------------------------------------------
# FINAL COMPARISON
# ---------------------------------------------------------

print()
print("FINAL DOMINANCE COMPARISON")
print()


dominance_comparison_df = (
    pd.DataFrame(
        dominance_results
    )
    .sort_values(
        "average_log_loss"
    )
)


print(
    dominance_comparison_df.to_string(
        index=False
    )
)

In [ ]:
# Test if this improvement is real or actually just noise

# ---------------------------------------------------------
# PAIRED WALK-FORWARD PREDICTIONS
# BASELINE VS ALL DOMINANCE
# ---------------------------------------------------------

paired_predictions = []


all_dominance_features = (
    baseline_features
    + [
        "shot_difference_last5_all_home",
        "shot_difference_last5_all_away",
        "possession_difference_last5_all_home",
        "possession_difference_last5_all_away",
        "corner_difference_last5_all_home",
        "corner_difference_last5_all_away"
    ]
)


for validation_season in [
    "2021_22",
    "2022_23",
    "2023_24",
    "2024_25"
]:

    validation_start = (
        model_df.loc[
            model_df["season_home"]
            == validation_season,
            "date_home"
        ]
        .min()
    )


    train_df = model_df[
        model_df["date_home"]
        < validation_start
    ].copy()


    validation_df = (
        model_df[
            model_df["season_home"]
            == validation_season
        ]
        .copy()
        .sort_values(
            [
                "date_home",
                "match_id"
            ]
        )
    )


    y_train = train_df["target"]
    y_validation = validation_df["target"]


    model_probabilities = {}


    # -----------------------------------------------------
    # TRAIN BOTH MODELS
    # -----------------------------------------------------

    for model_name, feature_set in [
        (
            "baseline",
            baseline_features
        ),
        (
            "dominance",
            all_dominance_features
        )
    ]:

        X_train = train_df[
            feature_set
        ]

        X_validation = validation_df[
            feature_set
        ]


        imputer = SimpleImputer(
            strategy="median",
            add_indicator=True
        )


        X_train_imputed = (
            imputer.fit_transform(
                X_train
            )
        )


        X_validation_imputed = (
            imputer.transform(
                X_validation
            )
        )


        rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=10,
            max_features="sqrt",
            criterion="gini",
            class_weight=None,
            random_state=42,
            n_jobs=-1
        )


        rf.fit(
            X_train_imputed,
            y_train
        )


        probabilities = rf.predict_proba(
            X_validation_imputed
        )


        probability_df = pd.DataFrame(
            probabilities,
            columns=rf.classes_,
            index=validation_df.index
        )


        model_probabilities[
            model_name
        ] = probability_df


    # -----------------------------------------------------
    # PER-MATCH LOG LOSS
    # -----------------------------------------------------

    for idx in validation_df.index:

        actual = validation_df.loc[
            idx,
            "target"
        ]


        baseline_probability = (
            model_probabilities[
                "baseline"
            ].loc[
                idx,
                actual
            ]
        )


        dominance_probability = (
            model_probabilities[
                "dominance"
            ].loc[
                idx,
                actual
            ]
        )


        paired_predictions.append({

            "match_id":
                validation_df.loc[
                    idx,
                    "match_id"
                ],

            "season":
                validation_season,

            "actual":
                actual,

            "baseline_loss":
                -np.log(
                    baseline_probability
                ),

            "dominance_loss":
                -np.log(
                    dominance_probability
                )
        })


paired_dominance_df = pd.DataFrame(
    paired_predictions
)


paired_dominance_df[
    "loss_improvement"
] = (
    paired_dominance_df[
        "baseline_loss"
    ]
    - paired_dominance_df[
        "dominance_loss"
    ]
)


print(
    "Matches:",
    len(paired_dominance_df)
)

print()

print(
    "Baseline mean loss:",
    paired_dominance_df[
        "baseline_loss"
    ].mean()
)

print(
    "Dominance mean loss:",
    paired_dominance_df[
        "dominance_loss"
    ].mean()
)

print(
    "Mean improvement:",
    paired_dominance_df[
        "loss_improvement"
    ].mean()
)


# ---------------------------------------------------------
# PAIRED BOOTSTRAP
# ---------------------------------------------------------

rng = np.random.default_rng(
    42
)


improvements = (
    paired_dominance_df[
        "loss_improvement"
    ]
    .to_numpy()
)


bootstrap_means = []


for _ in range(10000):

    sample = rng.choice(
        improvements,
        size=len(improvements),
        replace=True
    )

    bootstrap_means.append(
        sample.mean()
    )


bootstrap_means = np.array(
    bootstrap_means
)


lower = np.percentile(
    bootstrap_means,
    2.5
)

upper = np.percentile(
    bootstrap_means,
    97.5
)


print()
print("PAIRED BOOTSTRAP")

print(
    "95% CI:",
    (
        lower,
        upper
    )
)

print(
    "P(improvement <= 0):",
    (
        bootstrap_means
        <= 0
    ).mean()
)


print()
print("IMPROVEMENT BY SEASON")

print(
    paired_dominance_df
    .groupby("season")[
        "loss_improvement"
    ]
    .mean()
)

# DONE TESTING AND TWEAKING FEATURES
 Move on to locking in the RF model and evaluating it on 2025/26

In [ ]:
# ---------------------------------------------------------
# FINAL LOCKED RANDOM FOREST
# TRAIN: 2020/21 - 2024/25
# TEST:  2025/26
# ---------------------------------------------------------

final_train_df = (
    model_df[
        model_df["season_home"]
        != "2025_26"
    ]
    .copy()
)


final_test_df = (
    model_df[
        model_df["season_home"]
        == "2025_26"
    ]
    .copy()
    .sort_values(
        [
            "date_home",
            "match_id"
        ]
    )
)


print(
    "Training matches:",
    len(final_train_df)
)

print(
    "Test matches:",
    len(final_test_df)
)


# ---------------------------------------------------------
# FEATURES
# ---------------------------------------------------------

X_train = final_train_df[
    baseline_features
]

y_train = final_train_df[
    "target"
]


X_test = final_test_df[
    baseline_features
]

y_test = final_test_df[
    "target"
]


# ---------------------------------------------------------
# IMPUTATION
# ---------------------------------------------------------

final_imputer = SimpleImputer(
    strategy="median",
    add_indicator=True
)


X_train_imputed = (
    final_imputer.fit_transform(
        X_train
    )
)


X_test_imputed = (
    final_imputer.transform(
        X_test
    )
)


# ---------------------------------------------------------
# LOCKED RF
# ---------------------------------------------------------

final_rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=10,
    max_features="sqrt",
    criterion="gini",
    class_weight=None,
    random_state=42,
    n_jobs=-1
)


final_rf.fit(
    X_train_imputed,
    y_train
)


# ---------------------------------------------------------
# PREDICTIONS
# ---------------------------------------------------------

final_probabilities = (
    final_rf.predict_proba(
        X_test_imputed
    )
)


final_predictions = (
    final_rf.predict(
        X_test_imputed
    )
)


final_accuracy = accuracy_score(
    y_test,
    final_predictions
)


final_log_loss = log_loss(
    y_test,
    final_probabilities,
    labels=final_rf.classes_
)


print()
print("2025/26 LOCKED RF")

print(
    "Accuracy:",
    final_accuracy
)

print(
    "Log loss:",
    final_log_loss
)


# ---------------------------------------------------------
# CREATE PREDICTION DATAFRAME
# ---------------------------------------------------------

final_probability_df = pd.DataFrame(
    final_probabilities,
    columns=final_rf.classes_,
    index=final_test_df.index
)


final_predictions_df = pd.DataFrame({

    "match_id":
        final_test_df["match_id"],

    "actual":
        y_test,

    "prediction":
        final_predictions,

    "prob_H":
        final_probability_df["H"],

    "prob_D":
        final_probability_df["D"],

    "prob_A":
        final_probability_df["A"]
})


# ---------------------------------------------------------
# MERGE BET365 FAIR PROBABILITIES
# ---------------------------------------------------------

final_market_comparison = (
    final_predictions_df
    .merge(
        bet365_df[
            [
                "match_id",
                "fair_prob_H",
                "fair_prob_D",
                "fair_prob_A"
            ]
        ],
        on="match_id",
        how="left"
    )
)


print()

print(
    "Missing Bet365 matches:",
    final_market_comparison[
        [
            "fair_prob_H",
            "fair_prob_D",
            "fair_prob_A"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)


# ---------------------------------------------------------
# BET365 LOG LOSS
# ---------------------------------------------------------

market_probabilities = (
    final_market_comparison[
        [
            "fair_prob_A",
            "fair_prob_D",
            "fair_prob_H"
        ]
    ]
    .to_numpy()
)


market_log_loss = log_loss(
    final_market_comparison[
        "actual"
    ],
    market_probabilities,
    labels=[
        "A",
        "D",
        "H"
    ]
)


print()
print("2025/26 FINAL COMPARISON")

print(
    "RF accuracy:",
    final_accuracy
)

print(
    "RF log loss:",
    final_log_loss
)

print(
    "Bet365 fair log loss:",
    market_log_loss
)

print(
    "RF - Bet365:",
    final_log_loss
    - market_log_loss
)

In [ ]:
# ---------------------------------------------------------
# LOAD UPDATED MATCH FEATURES
# ---------------------------------------------------------

import pandas as pd
import numpy as np


model_df = pd.read_csv(
    "../data/processed/pl_match_features.csv"
)


model_df["date"] = pd.to_datetime(
    model_df["date"],
    utc=True,
    errors="coerce"
)


model_df = (
    model_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Model dataframe shape:",
    model_df.shape
)

print(
    "Has season column:",
    "season" in model_df.columns
)

print()

print(
    model_df["season"]
    .value_counts()
    .sort_index()
)

In [ ]:
# Perform backtest on 2026/27 season

# ---------------------------------------------------------
# 2026/27 OUT-OF-SAMPLE BACKTEST
# LOCKED RF V1
# ---------------------------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    log_loss
)


# ---------------------------------------------------------
# LOCKED 31 FEATURES
# ---------------------------------------------------------

baseline_features = [

    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "shots_on_target_difference_last5_all_home",

    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    "days_rest_home",
    "days_rest_away",
    "matches_last_7_days_home",
    "matches_last_7_days_away",
    "matches_last_14_days_home",
    "matches_last_14_days_away",

    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]


# ---------------------------------------------------------
# TRAIN / TEST SPLIT
# ---------------------------------------------------------

train_df = (
    model_df[
        model_df["season"] != "2026_27"
    ]
    .copy()
)


test_2026 = (
    model_df[
        model_df["season"] == "2026_27"
    ]
    .copy()
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


X_train = train_df[
    baseline_features
]

y_train = train_df[
    "target"
]


X_test = test_2026[
    baseline_features
]

y_test = test_2026[
    "target"
]


print(
    "Training matches:",
    len(train_df)
)

print(
    "2026/27 test matches:",
    len(test_2026)
)


# ---------------------------------------------------------
# TRAINING-ONLY MEDIAN IMPUTATION
# ---------------------------------------------------------

imputer = SimpleImputer(
    strategy="median",
    add_indicator=True
)


X_train_imputed = imputer.fit_transform(
    X_train
)


X_test_imputed = imputer.transform(
    X_test
)


print(
    "Model input columns after imputation:",
    X_train_imputed.shape[1]
)


# ---------------------------------------------------------
# LOCKED RF V1
# ---------------------------------------------------------

rf_2026 = RandomForestClassifier(

    n_estimators=500,

    max_depth=6,

    min_samples_leaf=10,

    max_features="sqrt",

    criterion="gini",

    class_weight=None,

    random_state=42,

    n_jobs=-1
)


rf_2026.fit(
    X_train_imputed,
    y_train
)


# ---------------------------------------------------------
# PREDICTIONS
# ---------------------------------------------------------

predicted_class = rf_2026.predict(
    X_test_imputed
)


predicted_probabilities = rf_2026.predict_proba(
    X_test_imputed
)


print(
    "Model class order:",
    rf_2026.classes_
)


# ---------------------------------------------------------
# CREATE RESULTS TABLE
# ---------------------------------------------------------

prob_df = pd.DataFrame(
    predicted_probabilities,
    columns=[
        f"prob_{label}"
        for label in rf_2026.classes_
    ]
)


results_2026 = pd.concat(

    [
        test_2026[
            [
                "match_id",
                "date",
                "home_team",
                "away_team",
                "home_goals",
                "away_goals",
                "target"
            ]
        ].reset_index(drop=True),

        prob_df
    ],

    axis=1
)


results_2026["prediction"] = (
    predicted_class
)


results_2026["correct"] = (
    results_2026["prediction"]
    ==
    results_2026["target"]
)


# ---------------------------------------------------------
# METRICS
# ---------------------------------------------------------

accuracy_2026 = accuracy_score(
    y_test,
    predicted_class
)


logloss_2026 = log_loss(
    y_test,
    predicted_probabilities,
    labels=rf_2026.classes_
)


print()

print(
    "2026/27 Accuracy:",
    accuracy_2026
)

print(
    "2026/27 Log Loss:",
    logloss_2026
)

print(
    "Correct predictions:",
    results_2026["correct"].sum(),
    "/",
    len(results_2026)
)


# ---------------------------------------------------------
# DISPLAY ALL 20 PREDICTIONS
# ---------------------------------------------------------

display_results = (
    results_2026[
        [
            "date",
            "home_team",
            "away_team",
            "prob_H",
            "prob_D",
            "prob_A",
            "prediction",
            "home_goals",
            "away_goals",
            "target",
            "correct"
        ]
    ]
    .copy()
)


display_results[
    [
        "prob_H",
        "prob_D",
        "prob_A"
    ]
] = (
    display_results[
        [
            "prob_H",
            "prob_D",
            "prob_A"
        ]
    ]
    .round(3)
)


print()

print(
    display_results.to_string(
        index=False
    )
)

In [ ]:
# ---------------------------------------------------------
# SAVE 2026/27 LOCKED RF V1 BACKTEST
# ---------------------------------------------------------

output_path = (
    "../data/processed/"
    "rf_v1_2026_27_backtest.csv"
)


results_2026.to_csv(
    output_path,
    index=False
)


print(
    "Saved:",
    output_path
)

print(
    "Matches:",
    len(results_2026)
)

print(
    "Accuracy:",
    results_2026["correct"].mean()
)

print(
    "Log Loss:",
    logloss_2026
)

In [ ]:
# ---------------------------------------------------------
# EXTRACT 2026/27 BET365 ODDS
# CORRECTED FOR ACTUAL API STRUCTURE
# ---------------------------------------------------------

import json
from pathlib import Path
import pandas as pd


RAW_2026_DIR = Path(
    "../data/raw/matches/premier_league/2026_27"
)


def extract_bet365_match_odds(file_path):

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        match_data = json.load(f)


    # -----------------------------------------------------
    # MATCH ID
    # -----------------------------------------------------

    details = match_data.get(
        "details",
        {}
    )

    if isinstance(details, dict):

        details = details.get(
            "data",
            details
        )


    match_id = details.get(
        "id"
    )


    if match_id is None:
        return None


    # -----------------------------------------------------
    # ODDS DATA
    # -----------------------------------------------------

    odds_data = match_data.get(
        "odds"
    )


    if not isinstance(
        odds_data,
        dict
    ):
        return None


    odds_data = odds_data.get(
        "data",
        odds_data
    )


    bookmakers = odds_data.get(
        "bookmakers",
        []
    )


    if not isinstance(
        bookmakers,
        list
    ):
        return None


    # -----------------------------------------------------
    # FIND BET365
    # -----------------------------------------------------

    bet365 = None


    for bookmaker in bookmakers:

        if (
            isinstance(bookmaker, dict)
            and
            bookmaker.get("bookmaker") == "Bet365"
        ):

            bet365 = bookmaker
            break


    if bet365 is None:
        return None


    # -----------------------------------------------------
    # MATCH ODDS MARKET
    # -----------------------------------------------------

    markets = bet365.get(
        "markets",
        {}
    )


    if not isinstance(
        markets,
        dict
    ):
        return None


    match_odds = markets.get(
        "match_odds"
    )


    if not isinstance(
        match_odds,
        dict
    ):
        return None


    # -----------------------------------------------------
    # LAST-SEEN 1X2 ODDS
    # -----------------------------------------------------

    try:

        home_odds = float(
            match_odds["home"]["last_seen"]
        )

        draw_odds = float(
            match_odds["draw"]["last_seen"]
        )

        away_odds = float(
            match_odds["away"]["last_seen"]
        )

    except (
        KeyError,
        TypeError,
        ValueError
    ):

        return None


    # -----------------------------------------------------
    # RAW IMPLIED PROBABILITIES
    # -----------------------------------------------------

    raw_prob_H = (
        1 / home_odds
    )

    raw_prob_D = (
        1 / draw_odds
    )

    raw_prob_A = (
        1 / away_odds
    )


    # -----------------------------------------------------
    # BOOKMAKER OVERROUND
    # -----------------------------------------------------

    overround = (
        raw_prob_H
        +
        raw_prob_D
        +
        raw_prob_A
    )


    # -----------------------------------------------------
    # FAIR / NO-VIG PROBABILITIES
    # -----------------------------------------------------

    fair_prob_H = (
        raw_prob_H
        /
        overround
    )

    fair_prob_D = (
        raw_prob_D
        /
        overround
    )

    fair_prob_A = (
        raw_prob_A
        /
        overround
    )


    return {

        "match_id":
            match_id,

        "bet365_home_odds":
            home_odds,

        "bet365_draw_odds":
            draw_odds,

        "bet365_away_odds":
            away_odds,

        "overround":
            overround,

        "fair_prob_H":
            fair_prob_H,

        "fair_prob_D":
            fair_prob_D,

        "fair_prob_A":
            fair_prob_A
    }


# ---------------------------------------------------------
# PROCESS ALL 2026/27 RAW FILES
# ---------------------------------------------------------

bet365_rows_2026 = []


for file_path in RAW_2026_DIR.glob(
    "*.json"
):

    row = extract_bet365_match_odds(
        file_path
    )

    if row is not None:

        bet365_rows_2026.append(
            row
        )


bet365_2026 = pd.DataFrame(
    bet365_rows_2026
)


# ---------------------------------------------------------
# CHECK RESULTS
# ---------------------------------------------------------

print(
    "Bet365 matches extracted:",
    len(bet365_2026)
)


if len(bet365_2026) > 0:

    print(
        "Duplicate match IDs:",
        bet365_2026[
            "match_id"
        ].duplicated().sum()
    )

    print()

    print(
        "Average overround:",
        bet365_2026[
            "overround"
        ].mean()
    )

    print()

    print(
        "Average fair probability sum:",
        (
            bet365_2026[
                [
                    "fair_prob_H",
                    "fair_prob_D",
                    "fair_prob_A"
                ]
            ]
            .sum(axis=1)
            .mean()
        )
    )

    print()

    print(
        bet365_2026.head(
            10
        ).to_string(
            index=False
        )
    )

In [ ]:
# Compare Bet365 performance on 2026/27 season so far to our model

# ---------------------------------------------------------
# COMPARE LOCKED RF V1 VS BET365
# 2026/27 - SAME 20 MATCHES
# ---------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    log_loss
)

import pandas as pd
import numpy as np


# ---------------------------------------------------------
# MERGE MODEL RESULTS WITH BET365 ODDS
# ---------------------------------------------------------

comparison_2026 = (
    results_2026
    .merge(
        bet365_2026,
        on="match_id",
        how="inner"
    )
    .copy()
)


print(
    "Matched games:",
    len(comparison_2026)
)


# ---------------------------------------------------------
# BET365 PREDICTION
# HIGHEST FAIR PROBABILITY
# ---------------------------------------------------------

bet365_probability_columns = [
    "fair_prob_H",
    "fair_prob_D",
    "fair_prob_A"
]


bet365_label_map = {
    "fair_prob_H": "H",
    "fair_prob_D": "D",
    "fair_prob_A": "A"
}


comparison_2026[
    "bet365_prediction"
] = (
    comparison_2026[
        bet365_probability_columns
    ]
    .idxmax(axis=1)
    .map(bet365_label_map)
)


comparison_2026[
    "bet365_correct"
] = (
    comparison_2026[
        "bet365_prediction"
    ]
    ==
    comparison_2026[
        "target"
    ]
)


# ---------------------------------------------------------
# BET365 PROBABILITY MATRIX
# ORDER MUST MATCH ['A', 'D', 'H']
# ---------------------------------------------------------

bet365_probs = (
    comparison_2026[
        [
            "fair_prob_A",
            "fair_prob_D",
            "fair_prob_H"
        ]
    ]
    .to_numpy()
)


# ---------------------------------------------------------
# RF PROBABILITY MATRIX
# SAME CLASS ORDER ['A', 'D', 'H']
# ---------------------------------------------------------

rf_probs = (
    comparison_2026[
        [
            "prob_A",
            "prob_D",
            "prob_H"
        ]
    ]
    .to_numpy()
)


# ---------------------------------------------------------
# METRICS
# ---------------------------------------------------------

rf_accuracy = accuracy_score(
    comparison_2026["target"],
    comparison_2026["prediction"]
)


bet365_accuracy = accuracy_score(
    comparison_2026["target"],
    comparison_2026["bet365_prediction"]
)


rf_logloss = log_loss(
    comparison_2026["target"],
    rf_probs,
    labels=[
        "A",
        "D",
        "H"
    ]
)


bet365_logloss = log_loss(
    comparison_2026["target"],
    bet365_probs,
    labels=[
        "A",
        "D",
        "H"
    ]
)


print()

print(
    "LOCKED RF V1"
)

print(
    "Accuracy:",
    round(
        rf_accuracy,
        4
    )
)

print(
    "Correct:",
    comparison_2026[
        "correct"
    ].sum(),
    "/",
    len(comparison_2026)
)

print(
    "Log Loss:",
    round(
        rf_logloss,
        4
    )
)


print()

print(
    "BET365 FAIR MARKET"
)

print(
    "Accuracy:",
    round(
        bet365_accuracy,
        4
    )
)

print(
    "Correct:",
    comparison_2026[
        "bet365_correct"
    ].sum(),
    "/",
    len(comparison_2026)
)

print(
    "Log Loss:",
    round(
        bet365_logloss,
        4
    )
)


# ---------------------------------------------------------
# MODEL VS MARKET DIFFERENCES
# ---------------------------------------------------------

comparison_2026[
    "rf_market_diff_H"
] = (
    comparison_2026["prob_H"]
    -
    comparison_2026["fair_prob_H"]
)


comparison_2026[
    "rf_market_diff_D"
] = (
    comparison_2026["prob_D"]
    -
    comparison_2026["fair_prob_D"]
)


comparison_2026[
    "rf_market_diff_A"
] = (
    comparison_2026["prob_A"]
    -
    comparison_2026["fair_prob_A"]
)


comparison_2026[
    "same_prediction"
] = (
    comparison_2026["prediction"]
    ==
    comparison_2026["bet365_prediction"]
)


# ---------------------------------------------------------
# DISPLAY
# ---------------------------------------------------------

display_comparison = (
    comparison_2026[
        [
            "date",
            "home_team",
            "away_team",

            "prob_H",
            "prob_D",
            "prob_A",
            "prediction",

            "fair_prob_H",
            "fair_prob_D",
            "fair_prob_A",
            "bet365_prediction",

            "target",
            "correct",
            "bet365_correct",
            "same_prediction"
        ]
    ]
    .copy()
)


probability_columns = [
    "prob_H",
    "prob_D",
    "prob_A",
    "fair_prob_H",
    "fair_prob_D",
    "fair_prob_A"
]


display_comparison[
    probability_columns
] = (
    display_comparison[
        probability_columns
    ]
    .round(3)
)


print()

print(
    display_comparison.to_string(
        index=False
    )
)

In [ ]:
# ---------------------------------------------------------
# EXACT-SCORE MODEL - STEP 1
# INSPECT HISTORICAL GOAL DISTRIBUTION
# ---------------------------------------------------------

score_train = (
    model_df[
        model_df["season"] != "2026_27"
    ]
    .copy()
)


print(
    "Training matches:",
    len(score_train)
)

print()

print(
    "Average home goals:",
    round(
        score_train["home_goals"].mean(),
        3
    )
)

print(
    "Average away goals:",
    round(
        score_train["away_goals"].mean(),
        3
    )
)

print(
    "Average total goals:",
    round(
        (
            score_train["home_goals"]
            +
            score_train["away_goals"]
        ).mean(),
        3
    )
)


print()

print(
    "HOME GOAL DISTRIBUTION"
)

print(
    score_train[
        "home_goals"
    ]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .round(4)
)


print()

print(
    "AWAY GOAL DISTRIBUTION"
)

print(
    score_train[
        "away_goals"
    ]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .round(4)
)


print()

print(
    "Most common exact scores:"
)

print(
    score_train[
        [
            "home_goals",
            "away_goals"
        ]
    ]
    .value_counts()
    .head(15)
)

In [ ]:
# ---------------------------------------------------------
# EXACT-SCORE MODEL - STEP 2
# TRAIN HOME + AWAY GOAL MODELS
# ---------------------------------------------------------

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# TRAIN / TEST SPLIT
# ---------------------------------------------------------

score_train = (
    model_df[
        model_df["season"] != "2026_27"
    ]
    .copy()
)


score_test = (
    model_df[
        model_df["season"] == "2026_27"
    ]
    .copy()
)


print(
    "Training matches:",
    len(score_train)
)

print(
    "2026/27 test matches:",
    len(score_test)
)


# ---------------------------------------------------------
# FEATURES
# SAME LOCKED 31 FEATURES AS RF V1
# ---------------------------------------------------------

X_score_train = (
    score_train[
        baseline_features
    ]
    .copy()
)


X_score_test = (
    score_test[
        baseline_features
    ]
    .copy()
)


# ---------------------------------------------------------
# TARGETS
# ---------------------------------------------------------

y_home_train = (
    score_train[
        "home_goals"
    ]
)


y_away_train = (
    score_train[
        "away_goals"
    ]
)


y_home_test = (
    score_test[
        "home_goals"
    ]
)


y_away_test = (
    score_test[
        "away_goals"
    ]
)


# ---------------------------------------------------------
# IMPUTATION
# FIT ONLY ON HISTORICAL TRAINING DATA
# ---------------------------------------------------------

score_imputer = SimpleImputer(
    strategy="median",
    add_indicator=True
)


X_score_train_imp = (
    score_imputer.fit_transform(
        X_score_train
    )
)


X_score_test_imp = (
    score_imputer.transform(
        X_score_test
    )
)


print(
    "Model input columns after imputation:",
    X_score_train_imp.shape[1]
)


# ---------------------------------------------------------
# HOME GOAL MODEL
# ---------------------------------------------------------

home_goal_model = (
    HistGradientBoostingRegressor(
        loss="poisson",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=42
    )
)


home_goal_model.fit(
    X_score_train_imp,
    y_home_train
)


# ---------------------------------------------------------
# AWAY GOAL MODEL
# ---------------------------------------------------------

away_goal_model = (
    HistGradientBoostingRegressor(
        loss="poisson",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=42
    )
)


away_goal_model.fit(
    X_score_train_imp,
    y_away_train
)


# ---------------------------------------------------------
# EXPECTED-GOAL PREDICTIONS
# ---------------------------------------------------------

pred_home_goals = (
    home_goal_model.predict(
        X_score_test_imp
    )
)


pred_away_goals = (
    away_goal_model.predict(
        X_score_test_imp
    )
)


# Safety floor
pred_home_goals = np.clip(
    pred_home_goals,
    0.05,
    None
)


pred_away_goals = np.clip(
    pred_away_goals,
    0.05,
    None
)


# ---------------------------------------------------------
# BASIC GOAL-PREDICTION METRICS
# ---------------------------------------------------------

home_mae = mean_absolute_error(
    y_home_test,
    pred_home_goals
)


away_mae = mean_absolute_error(
    y_away_test,
    pred_away_goals
)


combined_mae = (
    home_mae
    +
    away_mae
) / 2


print()

print(
    "HOME GOALS MAE:",
    round(
        home_mae,
        4
    )
)


print(
    "AWAY GOALS MAE:",
    round(
        away_mae,
        4
    )
)


print(
    "AVERAGE GOALS MAE:",
    round(
        combined_mae,
        4
    )
)


# ---------------------------------------------------------
# DISPLAY EXPECTED GOALS
# ---------------------------------------------------------

score_results_2026 = (
    score_test[
        [
            "match_id",
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals"
        ]
    ]
    .copy()
)


score_results_2026[
    "pred_home_goals"
] = pred_home_goals


score_results_2026[
    "pred_away_goals"
] = pred_away_goals


score_results_2026[
    "actual_score"
] = (
    score_results_2026[
        "home_goals"
    ]
    .astype(int)
    .astype(str)
    +
    "-"
    +
    score_results_2026[
        "away_goals"
    ]
    .astype(int)
    .astype(str)
)


display_score_results = (
    score_results_2026[
        [
            "date",
            "home_team",
            "away_team",
            "pred_home_goals",
            "pred_away_goals",
            "actual_score"
        ]
    ]
    .copy()
)


display_score_results[
    [
        "pred_home_goals",
        "pred_away_goals"
    ]
] = (
    display_score_results[
        [
            "pred_home_goals",
            "pred_away_goals"
        ]
    ]
    .round(2)
)


print()

print(
    display_score_results.to_string(
        index=False
    )
)

In [ ]:
# ---------------------------------------------------------
# EXACT-SCORE MODEL - STEP 3
# CONVERT EXPECTED GOALS INTO SCORE PROBABILITIES
# ---------------------------------------------------------

from scipy.stats import poisson
import numpy as np
import pandas as pd


MAX_GOALS = 8


def calculate_score_probabilities(
    home_lambda,
    away_lambda,
    max_goals=MAX_GOALS
):

    score_matrix = np.zeros(
        (
            max_goals + 1,
            max_goals + 1
        )
    )


    # -----------------------------------------------------
    # EXACT SCORE PROBABILITIES
    # -----------------------------------------------------

    for home_goals in range(
        max_goals + 1
    ):

        home_probability = poisson.pmf(
            home_goals,
            home_lambda
        )


        for away_goals in range(
            max_goals + 1
        ):

            away_probability = poisson.pmf(
                away_goals,
                away_lambda
            )


            score_matrix[
                home_goals,
                away_goals
            ] = (
                home_probability
                *
                away_probability
            )


    # -----------------------------------------------------
    # MOST LIKELY EXACT SCORE
    # -----------------------------------------------------

    best_index = np.unravel_index(
        np.argmax(score_matrix),
        score_matrix.shape
    )


    predicted_home_goals = best_index[0]
    predicted_away_goals = best_index[1]


    predicted_score_probability = (
        score_matrix[
            predicted_home_goals,
            predicted_away_goals
        ]
    )


    # -----------------------------------------------------
    # H / D / A PROBABILITIES
    # -----------------------------------------------------

    home_win_probability = np.tril(
        score_matrix,
        k=-1
    ).sum()


    draw_probability = np.trace(
        score_matrix
    )


    away_win_probability = np.triu(
        score_matrix,
        k=1
    ).sum()


    total_probability = (
        home_win_probability
        +
        draw_probability
        +
        away_win_probability
    )


    # Normalize tiny amount of probability outside 0-8
    home_win_probability /= total_probability
    draw_probability /= total_probability
    away_win_probability /= total_probability


    return {
        "predicted_home_goals":
            predicted_home_goals,

        "predicted_away_goals":
            predicted_away_goals,

        "predicted_score":
            (
                f"{predicted_home_goals}-"
                f"{predicted_away_goals}"
            ),

        "score_probability":
            predicted_score_probability,

        "score_prob_H":
            home_win_probability,

        "score_prob_D":
            draw_probability,

        "score_prob_A":
            away_win_probability
    }


# ---------------------------------------------------------
# APPLY TO ALL 20 MATCHES
# ---------------------------------------------------------

score_probability_rows = []


for _, row in score_results_2026.iterrows():

    probabilities = (
        calculate_score_probabilities(
            row["pred_home_goals"],
            row["pred_away_goals"]
        )
    )


    score_probability_rows.append(
        probabilities
    )


score_probability_df = pd.DataFrame(
    score_probability_rows
)


score_results_2026 = pd.concat(
    [
        score_results_2026.reset_index(
            drop=True
        ),

        score_probability_df.reset_index(
            drop=True
        )
    ],
    axis=1
)


# ---------------------------------------------------------
# CHECK EXACT SCORE ACCURACY
# ---------------------------------------------------------

score_results_2026[
    "exact_score_correct"
] = (
    score_results_2026[
        "predicted_score"
    ]
    ==
    score_results_2026[
        "actual_score"
    ]
)


exact_score_hits = (
    score_results_2026[
        "exact_score_correct"
    ]
    .sum()
)


exact_score_accuracy = (
    score_results_2026[
        "exact_score_correct"
    ]
    .mean()
)


# ---------------------------------------------------------
# SCORE MODEL H/D/A PREDICTION
# ---------------------------------------------------------

score_results_2026[
    "score_outcome_prediction"
] = (
    score_results_2026[
        [
            "score_prob_H",
            "score_prob_D",
            "score_prob_A"
        ]
    ]
    .idxmax(axis=1)
    .str[-1]
)


score_results_2026[
    "actual_outcome"
] = np.select(
    [
        score_results_2026[
            "home_goals"
        ]
        >
        score_results_2026[
            "away_goals"
        ],

        score_results_2026[
            "home_goals"
        ]
        <
        score_results_2026[
            "away_goals"
        ]
    ],
    [
        "H",
        "A"
    ],
    default="D"
)


score_results_2026[
    "outcome_correct"
] = (
    score_results_2026[
        "score_outcome_prediction"
    ]
    ==
    score_results_2026[
        "actual_outcome"
    ]
)


# ---------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------

print(
    "Exact score hits:",
    exact_score_hits,
    "/",
    len(score_results_2026)
)


print(
    "Exact score accuracy:",
    round(
        exact_score_accuracy,
        4
    )
)


print(
    "Score-model outcome accuracy:",
    round(
        score_results_2026[
            "outcome_correct"
        ].mean(),
        4
    )
)


print()


# ---------------------------------------------------------
# DISPLAY RESULTS
# ---------------------------------------------------------

score_display = (
    score_results_2026[
        [
            "date",
            "home_team",
            "away_team",

            "pred_home_goals",
            "pred_away_goals",

            "predicted_score",
            "score_probability",

            "actual_score",
            "exact_score_correct",

            "score_prob_H",
            "score_prob_D",
            "score_prob_A",

            "score_outcome_prediction",
            "actual_outcome"
        ]
    ]
    .copy()
)


score_display[
    [
        "pred_home_goals",
        "pred_away_goals",
        "score_probability",
        "score_prob_H",
        "score_prob_D",
        "score_prob_A"
    ]
] = (
    score_display[
        [
            "pred_home_goals",
            "pred_away_goals",
            "score_probability",
            "score_prob_H",
            "score_prob_D",
            "score_prob_A"
        ]
    ]
    .round(3)
)


print(
    score_display.to_string(
        index=False
    )
)

In [ ]:
# ---------------------------------------------------------
# EXACT-SCORE MODEL - STEP 4
# HISTORICAL WALK-FORWARD EVALUATION
# ---------------------------------------------------------

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    accuracy_score,
    log_loss
)

from scipy.stats import poisson

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# WALK-FORWARD TEST SEASONS
# ---------------------------------------------------------

score_test_seasons = [
    "2021_22",
    "2022_23",
    "2023_24",
    "2024_25",
    "2025_26"
]


all_score_predictions = []


# ---------------------------------------------------------
# LOOP THROUGH EACH TEST SEASON
# ---------------------------------------------------------

for test_season in score_test_seasons:

    test_season_index = (
        list(
            model_df["season"]
            .drop_duplicates()
        )
        .index(test_season)
    )


    previous_seasons = (
        list(
            model_df["season"]
            .drop_duplicates()
        )[
            :test_season_index
        ]
    )


    train_fold = (
        model_df[
            model_df["season"].isin(
                previous_seasons
            )
        ]
        .copy()
    )


    test_fold = (
        model_df[
            model_df["season"]
            ==
            test_season
        ]
        .copy()
    )


    print(
        test_season,
        "| train:",
        len(train_fold),
        "| test:",
        len(test_fold)
    )


    # -----------------------------------------------------
    # FEATURES
    # -----------------------------------------------------

    X_train_fold = (
        train_fold[
            baseline_features
        ]
        .copy()
    )


    X_test_fold = (
        test_fold[
            baseline_features
        ]
        .copy()
    )


    # -----------------------------------------------------
    # IMPUTATION
    # -----------------------------------------------------

    fold_imputer = SimpleImputer(
        strategy="median",
        add_indicator=True
    )


    X_train_fold_imp = (
        fold_imputer.fit_transform(
            X_train_fold
        )
    )


    X_test_fold_imp = (
        fold_imputer.transform(
            X_test_fold
        )
    )


    # -----------------------------------------------------
    # HOME MODEL
    # -----------------------------------------------------

    fold_home_model = (
        HistGradientBoostingRegressor(
            loss="poisson",
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=15,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=42
        )
    )


    fold_home_model.fit(
        X_train_fold_imp,
        train_fold["home_goals"]
    )


    # -----------------------------------------------------
    # AWAY MODEL
    # -----------------------------------------------------

    fold_away_model = (
        HistGradientBoostingRegressor(
            loss="poisson",
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=15,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=42
        )
    )


    fold_away_model.fit(
        X_train_fold_imp,
        train_fold["away_goals"]
    )


    # -----------------------------------------------------
    # EXPECTED GOALS
    # -----------------------------------------------------

    fold_home_lambda = np.clip(
        fold_home_model.predict(
            X_test_fold_imp
        ),
        0.05,
        None
    )


    fold_away_lambda = np.clip(
        fold_away_model.predict(
            X_test_fold_imp
        ),
        0.05,
        None
    )


    # -----------------------------------------------------
    # BUILD MATCH PREDICTIONS
    # -----------------------------------------------------

    fold_results = (
        test_fold[
            [
                "match_id",
                "season",
                "date",
                "home_team",
                "away_team",
                "home_goals",
                "away_goals",
                "target"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    fold_results[
        "pred_home_goals"
    ] = fold_home_lambda


    fold_results[
        "pred_away_goals"
    ] = fold_away_lambda


    predicted_scores = []

    exact_score_probabilities = []

    outcome_H = []
    outcome_D = []
    outcome_A = []


    # -----------------------------------------------------
    # POISSON SCORE MATRIX
    # -----------------------------------------------------

    for home_lambda, away_lambda in zip(
        fold_home_lambda,
        fold_away_lambda
    ):

        result = (
            calculate_score_probabilities(
                home_lambda,
                away_lambda,
                max_goals=8
            )
        )


        predicted_scores.append(
            result["predicted_score"]
        )


        exact_score_probabilities.append(
            result["score_probability"]
        )


        outcome_H.append(
            result["score_prob_H"]
        )


        outcome_D.append(
            result["score_prob_D"]
        )


        outcome_A.append(
            result["score_prob_A"]
        )


    fold_results[
        "predicted_score"
    ] = predicted_scores


    fold_results[
        "predicted_score_probability"
    ] = exact_score_probabilities


    fold_results[
        "prob_H"
    ] = outcome_H


    fold_results[
        "prob_D"
    ] = outcome_D


    fold_results[
        "prob_A"
    ] = outcome_A


    # -----------------------------------------------------
    # ACTUAL SCORE
    # -----------------------------------------------------

    fold_results[
        "actual_score"
    ] = (
        fold_results[
            "home_goals"
        ]
        .astype(int)
        .astype(str)
        +
        "-"
        +
        fold_results[
            "away_goals"
        ]
        .astype(int)
        .astype(str)
    )


    fold_results[
        "exact_score_correct"
    ] = (
        fold_results[
            "predicted_score"
        ]
        ==
        fold_results[
            "actual_score"
        ]
    )


    # -----------------------------------------------------
    # H/D/A PREDICTION
    # -----------------------------------------------------

    probability_columns = [
        "prob_H",
        "prob_D",
        "prob_A"
    ]


    label_map = {
        "prob_H": "H",
        "prob_D": "D",
        "prob_A": "A"
    }


    fold_results[
        "prediction"
    ] = (
        fold_results[
            probability_columns
        ]
        .idxmax(axis=1)
        .map(label_map)
    )


    all_score_predictions.append(
        fold_results
    )


# ---------------------------------------------------------
# COMBINE ALL WALK-FORWARD SEASONS
# ---------------------------------------------------------

score_walkforward = pd.concat(
    all_score_predictions,
    ignore_index=True
)


# ---------------------------------------------------------
# OVERALL METRICS
# ---------------------------------------------------------

home_mae_wf = mean_absolute_error(
    score_walkforward["home_goals"],
    score_walkforward["pred_home_goals"]
)


away_mae_wf = mean_absolute_error(
    score_walkforward["away_goals"],
    score_walkforward["pred_away_goals"]
)


average_mae_wf = (
    home_mae_wf
    +
    away_mae_wf
) / 2


exact_accuracy_wf = (
    score_walkforward[
        "exact_score_correct"
    ]
    .mean()
)


outcome_accuracy_wf = accuracy_score(
    score_walkforward["target"],
    score_walkforward["prediction"]
)


score_probs_wf = (
    score_walkforward[
        [
            "prob_A",
            "prob_D",
            "prob_H"
        ]
    ]
    .to_numpy()
)


outcome_logloss_wf = log_loss(
    score_walkforward["target"],
    score_probs_wf,
    labels=[
        "A",
        "D",
        "H"
    ]
)


# ---------------------------------------------------------
# PRINT OVERALL RESULTS
# ---------------------------------------------------------

print()

print(
    "TOTAL WALK-FORWARD MATCHES:",
    len(score_walkforward)
)


print()

print(
    "HOME GOALS MAE:",
    round(
        home_mae_wf,
        4
    )
)


print(
    "AWAY GOALS MAE:",
    round(
        away_mae_wf,
        4
    )
)


print(
    "AVERAGE GOALS MAE:",
    round(
        average_mae_wf,
        4
    )
)


print()

print(
    "EXACT SCORE ACCURACY:",
    round(
        exact_accuracy_wf,
        4
    )
)


print(
    "EXACT SCORE HITS:",
    score_walkforward[
        "exact_score_correct"
    ].sum(),
    "/",
    len(score_walkforward)
)


print()

print(
    "H/D/A ACCURACY:",
    round(
        outcome_accuracy_wf,
        4
    )
)


print(
    "H/D/A LOG LOSS:",
    round(
        outcome_logloss_wf,
        4
    )
)


# ---------------------------------------------------------
# RESULTS BY SEASON
# ---------------------------------------------------------

season_summary = (
    score_walkforward
    .groupby("season")
    .apply(
        lambda x: pd.Series(
            {
                "matches":
                    len(x),

                "home_mae":
                    mean_absolute_error(
                        x["home_goals"],
                        x["pred_home_goals"]
                    ),

                "away_mae":
                    mean_absolute_error(
                        x["away_goals"],
                        x["pred_away_goals"]
                    ),

                "exact_score_accuracy":
                    x[
                        "exact_score_correct"
                    ].mean(),

                "outcome_accuracy":
                    (
                        x["prediction"]
                        ==
                        x["target"]
                    ).mean()
            }
        ),
        include_groups=False
    )
)


print()

print(
    "BY SEASON:"
)

print(
    season_summary.round(4)
)

# TESTING V2 MODEL: ADDING ON API-FOOTBALL STATS

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL EXPERIMENT
# LOCKED V1 VS V1 + INJURY FEATURES
# ---------------------------------------------------------

import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    log_loss
)


# ---------------------------------------------------------
# LOAD EXPERIMENTAL DATA
# ---------------------------------------------------------

injury_model_df = pd.read_csv(
    "../data/processed/"
    "pl_match_features_with_injuries_experimental.csv"
)


injury_model_df["date"] = pd.to_datetime(
    injury_model_df["date"],
    utc=True
)


injury_model_df = (
    injury_model_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Dataset shape:",
    injury_model_df.shape
)


print()

print(
    injury_model_df[
        "season"
    ]
    .value_counts()
    .sort_index()
)


# ---------------------------------------------------------
# LOCKED 31 V1 FEATURES
# ---------------------------------------------------------

baseline_features = [

    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "shots_on_target_difference_last5_all_home",

    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    "days_rest_home",
    "days_rest_away",

    "matches_last_7_days_home",
    "matches_last_7_days_away",

    "matches_last_14_days_home",
    "matches_last_14_days_away",

    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]


# ---------------------------------------------------------
# NEW API-FOOTBALL FEATURES
# ---------------------------------------------------------

injury_features = [

    "missing_fixture_home",
    "questionable_home",

    "missing_fixture_away",
    "questionable_away",

    "missing_fixture_difference",
    "questionable_difference"
]


v2_features = (
    baseline_features
    +
    injury_features
)


print()

print(
    "V1 feature count:",
    len(baseline_features)
)


print(
    "V2 feature count:",
    len(v2_features)
)


# ---------------------------------------------------------
# LOCKED RANDOM FOREST
#
# EXACT SAME SETTINGS FOR V1 AND V2
# ---------------------------------------------------------

def create_locked_rf():

    return RandomForestClassifier(

        n_estimators=500,

        max_depth=6,

        min_samples_leaf=10,

        max_features="sqrt",

        criterion="gini",

        class_weight=None,

        random_state=42,

        n_jobs=-1
    )


# ---------------------------------------------------------
# WALK-FORWARD TEST SEASONS
# ---------------------------------------------------------

test_seasons = [
    "2023_24",
    "2024_25"
]


results = []


for test_season in test_seasons:

    print()
    print("=" * 70)
    print(
        "TEST SEASON:",
        test_season
    )
    print("=" * 70)


    # Find beginning of test season

    test_start_date = (
        injury_model_df.loc[
            injury_model_df["season"]
            ==
            test_season,
            "date"
        ]
        .min()
    )


    # Everything BEFORE test season
    # becomes training data

    train_df = (
        injury_model_df[
            injury_model_df["date"]
            <
            test_start_date
        ]
        .copy()
    )


    test_df = (
        injury_model_df[
            injury_model_df["season"]
            ==
            test_season
        ]
        .copy()
    )


    print(
        "Training matches:",
        len(train_df)
    )


    print(
        "Test matches:",
        len(test_df)
    )


    y_train = (
        train_df["target"]
    )


    y_test = (
        test_df["target"]
    )


    # =====================================================
    # V1
    # =====================================================

    v1_imputer = SimpleImputer(
        strategy="median",
        add_indicator=True
    )


    X_train_v1 = (
        v1_imputer.fit_transform(
            train_df[
                baseline_features
            ]
        )
    )


    X_test_v1 = (
        v1_imputer.transform(
            test_df[
                baseline_features
            ]
        )
    )


    v1_model = (
        create_locked_rf()
    )


    v1_model.fit(
        X_train_v1,
        y_train
    )


    v1_prob = (
        v1_model.predict_proba(
            X_test_v1
        )
    )


    v1_pred = (
        v1_model.predict(
            X_test_v1
        )
    )


    v1_accuracy = (
        accuracy_score(
            y_test,
            v1_pred
        )
    )


    v1_logloss = (
        log_loss(
            y_test,
            v1_prob,
            labels=v1_model.classes_
        )
    )


    # =====================================================
    # V2
    # =====================================================

    v2_imputer = SimpleImputer(
        strategy="median",
        add_indicator=True
    )


    X_train_v2 = (
        v2_imputer.fit_transform(
            train_df[
                v2_features
            ]
        )
    )


    X_test_v2 = (
        v2_imputer.transform(
            test_df[
                v2_features
            ]
        )
    )


    v2_model = (
        create_locked_rf()
    )


    v2_model.fit(
        X_train_v2,
        y_train
    )


    v2_prob = (
        v2_model.predict_proba(
            X_test_v2
        )
    )


    v2_pred = (
        v2_model.predict(
            X_test_v2
        )
    )


    v2_accuracy = (
        accuracy_score(
            y_test,
            v2_pred
        )
    )


    v2_logloss = (
        log_loss(
            y_test,
            v2_prob,
            labels=v2_model.classes_
        )
    )


    # =====================================================
    # PRINT SEASON RESULTS
    # =====================================================

    print()

    print(
        "V1 Accuracy:",
        round(
            v1_accuracy,
            4
        )
    )


    print(
        "V2 Accuracy:",
        round(
            v2_accuracy,
            4
        )
    )


    print(
        "Accuracy change:",
        round(
            v2_accuracy
            -
            v1_accuracy,
            4
        )
    )


    print()

    print(
        "V1 Log Loss:",
        round(
            v1_logloss,
            4
        )
    )


    print(
        "V2 Log Loss:",
        round(
            v2_logloss,
            4
        )
    )


    print(
        "Log Loss change:",
        round(
            v2_logloss
            -
            v1_logloss,
            4
        )
    )


    # -----------------------------------------------------
    # SAVE RESULTS
    # -----------------------------------------------------

    results.append(
        {
            "season":
                test_season,

            "matches":
                len(test_df),

            "v1_accuracy":
                v1_accuracy,

            "v2_accuracy":
                v2_accuracy,

            "accuracy_change":
                v2_accuracy
                -
                v1_accuracy,

            "v1_logloss":
                v1_logloss,

            "v2_logloss":
                v2_logloss,

            "logloss_change":
                v2_logloss
                -
                v1_logloss
        }
    )


# ---------------------------------------------------------
# FINAL COMPARISON
# ---------------------------------------------------------

results_df = pd.DataFrame(
    results
)


print()
print("=" * 70)
print("FINAL COMPARISON")
print("=" * 70)


print(
    results_df.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# COMBINED RESULTS
# ---------------------------------------------------------

total_matches = (
    results_df["matches"].sum()
)


weighted_v1_accuracy = (
    (
        results_df["v1_accuracy"]
        *
        results_df["matches"]
    ).sum()
    /
    total_matches
)


weighted_v2_accuracy = (
    (
        results_df["v2_accuracy"]
        *
        results_df["matches"]
    ).sum()
    /
    total_matches
)


weighted_v1_logloss = (
    (
        results_df["v1_logloss"]
        *
        results_df["matches"]
    ).sum()
    /
    total_matches
)


weighted_v2_logloss = (
    (
        results_df["v2_logloss"]
        *
        results_df["matches"]
    ).sum()
    /
    total_matches
)


print()
print("=" * 70)
print("COMBINED WALK-FORWARD RESULTS")
print("=" * 70)


print(
    "Total test matches:",
    total_matches
)


print()

print(
    "V1 Accuracy:",
    round(
        weighted_v1_accuracy,
        4
    )
)


print(
    "V2 Accuracy:",
    round(
        weighted_v2_accuracy,
        4
    )
)


print(
    "Accuracy change:",
    round(
        weighted_v2_accuracy
        -
        weighted_v1_accuracy,
        4
    )
)


print()

print(
    "V1 Log Loss:",
    round(
        weighted_v1_logloss,
        4
    )
)


print(
    "V2 Log Loss:",
    round(
        weighted_v2_logloss,
        4
    )
)


print(
    "Log Loss change:",
    round(
        weighted_v2_logloss
        -
        weighted_v1_logloss,
        4
    )
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 18
# V1 vs V2 vs V3
#
# V1 = LOCKED BASELINE
# V2 = RAW INJURY COUNTS
# V3 = ABSENCE-TYPE FEATURES
# ---------------------------------------------------------

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    log_loss
)


# =========================================================
# LOAD EXISTING EXPERIMENTAL DATASET
# =========================================================

df = pd.read_csv(
    "../data/processed/"
    "pl_match_features_with_injuries_experimental.csv"
)


df["date"] = pd.to_datetime(
    df["date"],
    utc=True
)


# =========================================================
# LOAD ABSENCE-TYPE FEATURES
# =========================================================

absence_df = pd.read_csv(
    "../data/processed/"
    "api_football_absence_type_features.csv"
)


absence_df["date"] = pd.to_datetime(
    absence_df["date"],
    utc=True
)


# =========================================================
# THE SIX "OTHER" RECORDS WERE MEDICAL ABSENCES.
#
# Fold them into physical injury rather than creating
# a feature based on only six observations.
# =========================================================

for side in [
    "home",
    "away"
]:

    absence_df[
        f"physical_injury_{side}"
    ] = (
        absence_df[
            f"physical_injury_{side}"
        ]
        +
        absence_df[
            f"other_{side}"
        ]
    )


# Recalculate physical difference after adding "other"

absence_df[
    "physical_injury_difference"
] = (
    absence_df[
        "physical_injury_home"
    ]
    -
    absence_df[
        "physical_injury_away"
    ]
)


# =========================================================
# FEATURES WE WANT FROM ABSENCE DATA
# =========================================================

absence_type_features = [

    "physical_injury_home",
    "physical_injury_away",
    "physical_injury_difference",

    "disciplinary_home",
    "disciplinary_away",
    "disciplinary_difference",

    "illness_fitness_home",
    "illness_fitness_away",
    "illness_fitness_difference",

    "selection_unavailable_home",
    "selection_unavailable_away",
    "selection_unavailable_difference"
]


# =========================================================
# MERGE ABSENCE TYPES INTO MODEL DATA
#
# Both datasets contain the mapped model team names.
# Date + home + away uniquely identifies each PL match.
# =========================================================

df = df.merge(

    absence_df[
        [
            "date",
            "home_team",
            "away_team"
        ]
        +
        absence_type_features
    ],

    on=[
        "date",
        "home_team",
        "away_team"
    ],

    how="left",

    validate="one_to_one"
)


# =========================================================
# VALIDATE MERGE
# =========================================================

print("=" * 80)
print("MERGE CHECK")
print("=" * 80)


print(
    "Dataset shape:",
    df.shape
)


print(
    "Matches:",
    len(df)
)


print(
    "Missing absence-type cells:",
    int(
        df[
            absence_type_features
        ]
        .isna()
        .sum()
        .sum()
    )
)


print()


# =========================================================
# LOCKED V1 FEATURES
# =========================================================

baseline_features = [

    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "shots_on_target_difference_last5_all_home",

    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    "days_rest_home",
    "days_rest_away",

    "matches_last_7_days_home",
    "matches_last_7_days_away",

    "matches_last_14_days_home",
    "matches_last_14_days_away",

    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]


# =========================================================
# V2 RAW INJURY FEATURES
# =========================================================

raw_injury_features = [

    "missing_fixture_home",
    "questionable_home",

    "missing_fixture_away",
    "questionable_away",

    "missing_fixture_difference",
    "questionable_difference"
]


# =========================================================
# MODEL FEATURE SETS
# =========================================================

v1_features = (
    baseline_features
)


v2_features = (
    baseline_features
    +
    raw_injury_features
)


v3_features = (
    baseline_features
    +
    absence_type_features
)


print(
    "V1 feature count:",
    len(v1_features)
)


print(
    "V2 feature count:",
    len(v2_features)
)


print(
    "V3 feature count:",
    len(v3_features)
)


# =========================================================
# LOCKED RANDOM FOREST FUNCTION
#
# Each version gets its own imputer fit ONLY on training
# data. RF parameters remain identical.
# =========================================================

def train_and_predict(
    train_df,
    test_df,
    features
):

    imputer = SimpleImputer(
        strategy="median",
        add_indicator=True
    )


    X_train = imputer.fit_transform(
        train_df[
            features
        ]
    )


    X_test = imputer.transform(
        test_df[
            features
        ]
    )


    y_train = (
        train_df[
            "target"
        ]
    )


    model = RandomForestClassifier(

        n_estimators=500,

        max_depth=6,

        min_samples_leaf=10,

        max_features="sqrt",

        criterion="gini",

        class_weight=None,

        random_state=42,

        n_jobs=-1
    )


    model.fit(
        X_train,
        y_train
    )


    probabilities = (
        model.predict_proba(
            X_test
        )
    )


    predictions = (
        model.predict(
            X_test
        )
    )


    return (
        model,
        predictions,
        probabilities
    )


# =========================================================
# WALK-FORWARD TEST
# =========================================================

test_seasons = [
    "2023_24",
    "2024_25"
]


results = []


for test_season in test_seasons:

    print()
    print("=" * 80)
    print(
        "TEST SEASON:",
        test_season
    )
    print("=" * 80)


    # -----------------------------------------------------
    # TRAIN ONLY ON EARLIER API-FOOTBALL SEASONS
    # -----------------------------------------------------

    if test_season == "2023_24":

        train_seasons = [
            "2022_23"
        ]

    else:

        train_seasons = [
            "2022_23",
            "2023_24"
        ]


    train_df = (
        df[
            df[
                "season"
            ].isin(
                train_seasons
            )
        ]
        .copy()
    )


    test_df = (
        df[
            df[
                "season"
            ]
            ==
            test_season
        ]
        .copy()
    )


    y_test = (
        test_df[
            "target"
        ]
    )


    print(
        "Training matches:",
        len(train_df)
    )


    print(
        "Test matches:",
        len(test_df)
    )


    # =====================================================
    # V1
    # =====================================================

    (
        v1_model,
        v1_pred,
        v1_prob
    ) = train_and_predict(

        train_df,
        test_df,
        v1_features
    )


    # =====================================================
    # V2
    # =====================================================

    (
        v2_model,
        v2_pred,
        v2_prob
    ) = train_and_predict(

        train_df,
        test_df,
        v2_features
    )


    # =====================================================
    # V3
    # =====================================================

    (
        v3_model,
        v3_pred,
        v3_prob
    ) = train_and_predict(

        train_df,
        test_df,
        v3_features
    )


    # =====================================================
    # CLASS ORDER
    #
    # Should be A, D, H for all models.
    # =====================================================

    assert list(
        v1_model.classes_
    ) == list(
        v2_model.classes_
    ) == list(
        v3_model.classes_
    )


    class_order = (
        v1_model.classes_
    )


    # =====================================================
    # METRICS
    # =====================================================

    v1_accuracy = accuracy_score(
        y_test,
        v1_pred
    )


    v2_accuracy = accuracy_score(
        y_test,
        v2_pred
    )


    v3_accuracy = accuracy_score(
        y_test,
        v3_pred
    )


    v1_loss = log_loss(
        y_test,
        v1_prob,
        labels=class_order
    )


    v2_loss = log_loss(
        y_test,
        v2_prob,
        labels=class_order
    )


    v3_loss = log_loss(
        y_test,
        v3_prob,
        labels=class_order
    )


    # =====================================================
    # PRINT
    # =====================================================

    print()

    print(
        "V1 Accuracy:",
        round(
            v1_accuracy,
            4
        )
    )


    print(
        "V2 Accuracy:",
        round(
            v2_accuracy,
            4
        )
    )


    print(
        "V3 Accuracy:",
        round(
            v3_accuracy,
            4
        )
    )


    print()

    print(
        "V1 Log Loss:",
        round(
            v1_loss,
            4
        )
    )


    print(
        "V2 Log Loss:",
        round(
            v2_loss,
            4
        )
    )


    print(
        "V3 Log Loss:",
        round(
            v3_loss,
            4
        )
    )


    print()

    print(
        "V2 vs V1 Log Loss change:",
        round(
            v2_loss
            -
            v1_loss,
            4
        )
    )


    print(
        "V3 vs V1 Log Loss change:",
        round(
            v3_loss
            -
            v1_loss,
            4
        )
    )


    print(
        "V3 vs V2 Log Loss change:",
        round(
            v3_loss
            -
            v2_loss,
            4
        )
    )


    results.append(
        {
            "season":
                test_season,

            "matches":
                len(test_df),

            "v1_accuracy":
                v1_accuracy,

            "v2_accuracy":
                v2_accuracy,

            "v3_accuracy":
                v3_accuracy,

            "v1_logloss":
                v1_loss,

            "v2_logloss":
                v2_loss,

            "v3_logloss":
                v3_loss
        }
    )


# =========================================================
# FINAL SUMMARY
# =========================================================

results_df = pd.DataFrame(
    results
)


print()
print("=" * 80)
print("FINAL WALK-FORWARD SUMMARY")
print("=" * 80)


print(
    results_df
    .round(4)
    .to_string(
        index=False
    )
)


# =========================================================
# COMBINED / AVERAGE RESULTS
#
# Both seasons have 380 matches, so the simple mean gives
# the same weighting to each match here.
# =========================================================

print()
print("=" * 80)
print("AVERAGE PERFORMANCE")
print("=" * 80)


print(
    "V1 Accuracy:",
    round(
        results_df[
            "v1_accuracy"
        ].mean(),
        4
    )
)


print(
    "V2 Accuracy:",
    round(
        results_df[
            "v2_accuracy"
        ].mean(),
        4
    )
)


print(
    "V3 Accuracy:",
    round(
        results_df[
            "v3_accuracy"
        ].mean(),
        4
    )
)


print()


print(
    "V1 Log Loss:",
    round(
        results_df[
            "v1_logloss"
        ].mean(),
        4
    )
)


print(
    "V2 Log Loss:",
    round(
        results_df[
            "v2_logloss"
        ].mean(),
        4
    )
)


print(
    "V3 Log Loss:",
    round(
        results_df[
            "v3_logloss"
        ].mean(),
        4
    )
)


print()

print(
    "V2 improvement vs V1:",
    round(
        results_df[
            "v2_logloss"
        ].mean()
        -
        results_df[
            "v1_logloss"
        ].mean(),
        4
    )
)


print(
    "V3 improvement vs V1:",
    round(
        results_df[
            "v3_logloss"
        ].mean()
        -
        results_df[
            "v1_logloss"
        ].mean(),
        4
    )
)


print(
    "V3 improvement vs V2:",
    round(
        results_df[
            "v3_logloss"
        ].mean()
        -
        results_df[
            "v2_logloss"
        ].mean(),
        4
    )
)

In [ ]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("../data/processed")

csv_files = sorted(PROCESSED_DIR.glob("*.csv"))

print("Processed CSV files:")
for file_path in csv_files:
    print(file_path.name)

In [ ]:
baseline_features = [
    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "shots_on_target_difference_last5_all_home",
    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    "days_rest_home",
    "days_rest_away",

    "matches_last_7_days_home",
    "matches_last_7_days_away",
    "matches_last_14_days_home",
    "matches_last_14_days_away",

    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]

model_df = pd.read_csv(
    PROCESSED_DIR / "pl_match_features.csv"
)

missing_baseline_features = [
    col
    for col in baseline_features
    if col not in model_df.columns
]

print("Shape:", model_df.shape)

print("\nBaseline feature count:")
print(len(baseline_features))

print("\nRows by season:")
print(model_df.groupby("season").size())

print("\nMissing baseline features:")
print(missing_baseline_features)

print("\nTarget present:")
print("target" in model_df.columns)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, log_loss

train_2025 = model_df[
    model_df["season"].isin([
        "2020_21",
        "2021_22",
        "2022_23",
        "2023_24",
        "2024_25"
    ])
].copy()

test_2025 = model_df[
    model_df["season"] == "2025_26"
].copy()


X_train = train_2025[baseline_features].copy()
y_train = train_2025["target"].copy()

X_test = test_2025[baseline_features].copy()
y_test = test_2025["target"].copy()


v1_market_imputer = SimpleImputer(
    strategy="median",
    add_indicator=True
)

X_train_imputed = v1_market_imputer.fit_transform(X_train)
X_test_imputed = v1_market_imputer.transform(X_test)


v1_market_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=10,
    max_features="sqrt",
    criterion="gini",
    class_weight=None,
    random_state=42,
    n_jobs=-1
)

v1_market_model.fit(
    X_train_imputed,
    y_train
)


v1_predictions = v1_market_model.predict(
    X_test_imputed
)

v1_probabilities = v1_market_model.predict_proba(
    X_test_imputed
)


class_index = {
    label: i
    for i, label in enumerate(v1_market_model.classes_)
}


v1_2025_predictions = test_2025[
    [
        "match_id",
        "date",
        "season",
        "home_team",
        "away_team",
        "target"
    ]
].copy()

v1_2025_predictions["model_home_prob"] = (
    v1_probabilities[:, class_index["H"]]
)

v1_2025_predictions["model_draw_prob"] = (
    v1_probabilities[:, class_index["D"]]
)

v1_2025_predictions["model_away_prob"] = (
    v1_probabilities[:, class_index["A"]]
)

v1_2025_predictions["model_prediction"] = (
    v1_predictions
)


print("Training matches:", len(train_2025))
print("Test matches:", len(test_2025))

print("\nAccuracy:")
print(
    accuracy_score(
        y_test,
        v1_predictions
    )
)

print("\nLog loss:")
print(
    log_loss(
        y_test,
        v1_probabilities,
        labels=v1_market_model.classes_
    )
)

print("\nProbability sums:")
print(
    (
        v1_2025_predictions["model_home_prob"]
        + v1_2025_predictions["model_draw_prob"]
        + v1_2025_predictions["model_away_prob"]
    ).describe()
)

In [ ]:
# Load saved Bet365 fair market probabilities
bet365_market_df = pd.read_csv(
    PROCESSED_DIR / "bet365_pl_market_probabilities.csv"
)

# Keep only the 2025/26 market
market_2025 = bet365_market_df[
    bet365_market_df["season"] == "2025_26"
].copy()


# Merge model predictions with market probabilities
model_market_2025 = v1_2025_predictions.merge(
    market_2025,
    on=["match_id", "season"],
    how="left",
    validate="one_to_one"
)


# Calculate model vs market differences
model_market_2025["home_difference"] = (
    model_market_2025["model_home_prob"]
    - model_market_2025["market_home_fair"]
)

model_market_2025["draw_difference"] = (
    model_market_2025["model_draw_prob"]
    - model_market_2025["market_draw_fair"]
)

model_market_2025["away_difference"] = (
    model_market_2025["model_away_prob"]
    - model_market_2025["market_away_fair"]
)


print("Merged matches:", len(model_market_2025))

print("\nMissing market matches:")
print(
    model_market_2025[
        [
            "market_home_fair",
            "market_draw_fair",
            "market_away_fair"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)


print("\nAverage absolute model-market difference:")
print(
    model_market_2025[
        [
            "home_difference",
            "draw_difference",
            "away_difference"
        ]
    ]
    .abs()
    .mean()
)


print("\nSample model vs market comparison:")

display(
    model_market_2025[
        [
            "home_team",
            "away_team",
            "model_home_prob",
            "market_home_fair",
            "home_difference",
            "model_draw_prob",
            "market_draw_fair",
            "draw_difference",
            "model_away_prob",
            "market_away_fair",
            "away_difference"
        ]
    ].head(10)
)

In [ ]:
import numpy as np
import pandas as pd

# Find the outcome where the model is most optimistic vs the market
difference_cols = [
    "home_difference",
    "draw_difference",
    "away_difference"
]

model_market_2025["largest_positive_difference"] = (
    model_market_2025[difference_cols].max(axis=1)
)

model_market_2025["model_edge_outcome"] = (
    model_market_2025[difference_cols]
    .idxmax(axis=1)
    .map({
        "home_difference": "H",
        "draw_difference": "D",
        "away_difference": "A"
    })
)

# Did that outcome actually happen?
model_market_2025["edge_correct"] = (
    model_market_2025["model_edge_outcome"]
    == model_market_2025["target"]
).astype(int)


# Evaluate several disagreement thresholds
thresholds = [0.00, 0.05, 0.10, 0.15]

edge_results = []

for threshold in thresholds:

    subset = model_market_2025[
        model_market_2025["largest_positive_difference"] >= threshold
    ]

    edge_results.append({
        "Minimum Difference": threshold,
        "Matches": len(subset),
        "Correct": subset["edge_correct"].sum(),
        "Hit Rate": subset["edge_correct"].mean()
    })


edge_results_df = pd.DataFrame(edge_results)

print("MODEL VS MARKET DISAGREEMENT TEST")
print("--------------------------------")

display(edge_results_df)


print("\nAverage largest positive difference:")
print(
    model_market_2025[
        "largest_positive_difference"
    ].mean()
)

print("\nMaximum difference:")
print(
    model_market_2025[
        "largest_positive_difference"
    ].max()
)

In [ ]:
# Get the Bet365 odds corresponding to the model's selected outcome
model_market_2025["edge_odds"] = np.select(
    [
        model_market_2025["model_edge_outcome"] == "H",
        model_market_2025["model_edge_outcome"] == "D",
        model_market_2025["model_edge_outcome"] == "A"
    ],
    [
        model_market_2025["bet365_home_odds"],
        model_market_2025["bet365_draw_odds"],
        model_market_2025["bet365_away_odds"]
    ],
    default=np.nan
)


# Flat 1-unit profit
model_market_2025["edge_profit"] = np.where(
    model_market_2025["edge_correct"] == 1,
    model_market_2025["edge_odds"] - 1,
    -1
)


thresholds = [0.00, 0.05, 0.10, 0.15]

roi_results = []

for threshold in thresholds:

    subset = model_market_2025[
        model_market_2025["largest_positive_difference"] >= threshold
    ].copy()

    total_bets = len(subset)
    total_staked = total_bets
    total_profit = subset["edge_profit"].sum()

    roi = (
        total_profit / total_staked
        if total_staked > 0
        else np.nan
    )

    roi_results.append({
        "Minimum Difference": threshold,
        "Bets": total_bets,
        "Wins": subset["edge_correct"].sum(),
        "Hit Rate": subset["edge_correct"].mean(),
        "Average Odds": subset["edge_odds"].mean(),
        "Total Profit": total_profit,
        "ROI": roi
    })


roi_results_df = pd.DataFrame(roi_results)

print("MODEL-MARKET DIFFERENCE ROI TEST")
print("--------------------------------")

display(roi_results_df)

In [ ]:
edge_10 = model_market_2025[
    model_market_2025["largest_positive_difference"] >= 0.10
].copy()


outcome_breakdown = (
    edge_10
    .groupby("model_edge_outcome")
    .agg(
        Bets=("edge_correct", "size"),
        Wins=("edge_correct", "sum"),
        Hit_Rate=("edge_correct", "mean"),
        Average_Odds=("edge_odds", "mean"),
        Average_Difference=("largest_positive_difference", "mean"),
        Total_Profit=("edge_profit", "sum")
    )
    .reset_index()
)


outcome_breakdown["ROI"] = (
    outcome_breakdown["Total_Profit"]
    / outcome_breakdown["Bets"]
)


outcome_breakdown["Outcome"] = (
    outcome_breakdown["model_edge_outcome"]
    .map({
        "H": "Home",
        "D": "Draw",
        "A": "Away"
    })
)


outcome_breakdown = outcome_breakdown[
    [
        "Outcome",
        "Bets",
        "Wins",
        "Hit_Rate",
        "Average_Odds",
        "Average_Difference",
        "Total_Profit",
        "ROI"
    ]
]


print("10%+ MODEL-MARKET DIFFERENCE BY OUTCOME")
print("---------------------------------------")

display(outcome_breakdown)

In [ ]:
away_edge_10 = model_market_2025[
    (model_market_2025["largest_positive_difference"] >= 0.10)
    & (model_market_2025["model_edge_outcome"] == "A")
].copy()


away_edge_10 = away_edge_10.sort_values(
    "largest_positive_difference",
    ascending=False
)


away_edge_display = away_edge_10[
    [
        "date",
        "home_team",
        "away_team",
        "target",
        "model_away_prob",
        "market_away_fair",
        "largest_positive_difference",
        "bet365_away_odds",
        "edge_correct",
        "edge_profit"
    ]
].copy()


print("10%+ AWAY MODEL-MARKET DISAGREEMENTS")
print("------------------------------------")
print("Bets:", len(away_edge_display))
print("Wins:", away_edge_display["edge_correct"].sum())
print("Total profit:", away_edge_display["edge_profit"].sum())

display(away_edge_display)

In [ ]:
promoted_2025 = [
    "Leeds United",
    "Burnley",
    "Sunderland"
]


away_edge_10["promoted_away_team"] = (
    away_edge_10["away_team"].isin(promoted_2025)
)


promoted_breakdown = (
    away_edge_10
    .groupby("promoted_away_team")
    .agg(
        Bets=("edge_correct", "size"),
        Wins=("edge_correct", "sum"),
        Hit_Rate=("edge_correct", "mean"),
        Average_Odds=("edge_odds", "mean"),
        Average_Difference=("largest_positive_difference", "mean"),
        Total_Profit=("edge_profit", "sum")
    )
    .reset_index()
)


promoted_breakdown["ROI"] = (
    promoted_breakdown["Total_Profit"]
    / promoted_breakdown["Bets"]
)


promoted_breakdown["Group"] = (
    promoted_breakdown["promoted_away_team"]
    .map({
        True: "Promoted away teams",
        False: "Established PL away teams"
    })
)


promoted_breakdown = promoted_breakdown[
    [
        "Group",
        "Bets",
        "Wins",
        "Hit_Rate",
        "Average_Odds",
        "Average_Difference",
        "Total_Profit",
        "ROI"
    ]
]


print("10%+ AWAY DIFFERENCES: PROMOTED VS ESTABLISHED")
print("------------------------------------------------")

display(promoted_breakdown)

In [ ]:
# Convert the three probabilities per match into one long table
calibration_rows = []

for _, row in model_market_2025.iterrows():

    for outcome, model_col, market_col in [
        ("H", "model_home_prob", "market_home_fair"),
        ("D", "model_draw_prob", "market_draw_fair"),
        ("A", "model_away_prob", "market_away_fair")
    ]:

        calibration_rows.append({
            "outcome": outcome,
            "model_probability": row[model_col],
            "market_probability": row[market_col],
            "actual": int(row["target"] == outcome)
        })


calibration_df = pd.DataFrame(calibration_rows)


# Probability buckets
bins = [
    0.00,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    1.00
]

calibration_df["probability_bucket"] = pd.cut(
    calibration_df["model_probability"],
    bins=bins,
    include_lowest=True
)


calibration_summary = (
    calibration_df
    .groupby(
        "probability_bucket",
        observed=True
    )
    .agg(
        Predictions=("actual", "size"),
        Average_Model_Probability=("model_probability", "mean"),
        Average_Market_Probability=("market_probability", "mean"),
        Actual_Frequency=("actual", "mean")
    )
    .reset_index()
)


calibration_summary["Model_Calibration_Error"] = (
    calibration_summary["Average_Model_Probability"]
    - calibration_summary["Actual_Frequency"]
)


print("V1 PROBABILITY CALIBRATION — 2025/26")
print("------------------------------------")

display(calibration_summary)

In [ ]:
backtest_2026 = pd.read_csv(
    PROCESSED_DIR / "rf_v1_2026_27_backtest.csv"
)

print("Shape:")
print(backtest_2026.shape)

print("\nColumns:")
print(backtest_2026.columns.tolist())

print("\nFirst 5 rows:")
display(backtest_2026.head())

In [ ]:
# Get 2026/27 Bet365 market probabilities
market_2026 = bet365_market_df[
    bet365_market_df["season"] == "2026_27"
].copy()


# Merge saved V1 predictions with Bet365
model_market_2026 = backtest_2026.merge(
    market_2026,
    on="match_id",
    how="left",
    validate="one_to_one"
)


# Model vs market differences
model_market_2026["home_difference"] = (
    model_market_2026["prob_H"]
    - model_market_2026["market_home_fair"]
)

model_market_2026["draw_difference"] = (
    model_market_2026["prob_D"]
    - model_market_2026["market_draw_fair"]
)

model_market_2026["away_difference"] = (
    model_market_2026["prob_A"]
    - model_market_2026["market_away_fair"]
)


# Bet365's most likely outcome
market_probability_columns = [
    "market_home_fair",
    "market_draw_fair",
    "market_away_fair"
]

model_market_2026["market_prediction"] = (
    model_market_2026[market_probability_columns]
    .idxmax(axis=1)
    .map({
        "market_home_fair": "H",
        "market_draw_fair": "D",
        "market_away_fair": "A"
    })
)


# Bet365 accuracy
market_accuracy_2026 = (
    model_market_2026["market_prediction"]
    == model_market_2026["target"]
).mean()


# Bet365 log loss
market_probabilities_2026 = model_market_2026[
    [
        "market_away_fair",
        "market_draw_fair",
        "market_home_fair"
    ]
].to_numpy()

market_log_loss_2026 = log_loss(
    model_market_2026["target"],
    market_probabilities_2026,
    labels=["A", "D", "H"]
)


# V1 accuracy and log loss for the same matches
model_accuracy_2026 = (
    model_market_2026["prediction"]
    == model_market_2026["target"]
).mean()

model_probabilities_2026 = model_market_2026[
    [
        "prob_A",
        "prob_D",
        "prob_H"
    ]
].to_numpy()

model_log_loss_2026 = log_loss(
    model_market_2026["target"],
    model_probabilities_2026,
    labels=["A", "D", "H"]
)


print("Matched games:", len(model_market_2026))

print("\nMissing market probabilities:")
print(
    model_market_2026[
        [
            "market_home_fair",
            "market_draw_fair",
            "market_away_fair"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)


print("\n2026/27 V1")
print("Accuracy:", model_accuracy_2026)
print("Log loss:", model_log_loss_2026)


print("\n2026/27 Bet365 Fair Market")
print("Accuracy:", market_accuracy_2026)
print("Log loss:", market_log_loss_2026)


print("\nAverage absolute model-market difference:")
print(
    model_market_2026[
        [
            "home_difference",
            "draw_difference",
            "away_difference"
        ]
    ]
    .abs()
    .mean()
)

In [ ]:
difference_cols_2026 = [
    "home_difference",
    "draw_difference",
    "away_difference"
]


# Find V1's largest positive disagreement with Bet365
model_market_2026["largest_positive_difference"] = (
    model_market_2026[difference_cols_2026].max(axis=1)
)


model_market_2026["model_edge_outcome"] = (
    model_market_2026[difference_cols_2026]
    .idxmax(axis=1)
    .map({
        "home_difference": "H",
        "draw_difference": "D",
        "away_difference": "A"
    })
)


# Did that outcome happen?
model_market_2026["edge_correct"] = (
    model_market_2026["model_edge_outcome"]
    == model_market_2026["target"]
).astype(int)


# Get corresponding Bet365 decimal odds
model_market_2026["edge_odds"] = np.select(
    [
        model_market_2026["model_edge_outcome"] == "H",
        model_market_2026["model_edge_outcome"] == "D",
        model_market_2026["model_edge_outcome"] == "A"
    ],
    [
        model_market_2026["bet365_home_odds"],
        model_market_2026["bet365_draw_odds"],
        model_market_2026["bet365_away_odds"]
    ],
    default=np.nan
)


# Flat 1-unit profit
model_market_2026["edge_profit"] = np.where(
    model_market_2026["edge_correct"] == 1,
    model_market_2026["edge_odds"] - 1,
    -1
)


# Apply the previously examined 10% threshold
edge_2026_10 = model_market_2026[
    model_market_2026["largest_positive_difference"] >= 0.10
].copy()


print("2026/27 — 10%+ MODEL-MARKET DIFFERENCES")
print("----------------------------------------")

print("Bets:", len(edge_2026_10))
print("Wins:", edge_2026_10["edge_correct"].sum())

print(
    "Hit rate:",
    edge_2026_10["edge_correct"].mean()
)

print(
    "Average odds:",
    edge_2026_10["edge_odds"].mean()
)

print(
    "Average difference:",
    edge_2026_10["largest_positive_difference"].mean()
)

print(
    "Total profit:",
    edge_2026_10["edge_profit"].sum()
)

print(
    "ROI:",
    edge_2026_10["edge_profit"].sum()
    / len(edge_2026_10)
)


print("\nIndividual selections:")

display(
    edge_2026_10[
        [
            "date",
            "home_team",
            "away_team",
            "target",
            "model_edge_outcome",
            "largest_positive_difference",
            "edge_odds",
            "edge_correct",
            "edge_profit"
        ]
    ]
    .sort_values(
        "largest_positive_difference",
        ascending=False
    )
)

In [ ]:
edge_2026_display = edge_2026_10[
    [
        "home_team",
        "away_team",
        "target",
        "model_edge_outcome",
        "largest_positive_difference",
        "edge_odds",
        "edge_correct",
        "edge_profit"
    ]
].sort_values(
    "largest_positive_difference",
    ascending=False
)

print(
    edge_2026_display.to_string(
        index=False
    )
)

In [ ]:
promoted_by_season = {
    "2025_26": [
        "Leeds United",
        "Burnley",
        "Sunderland"
    ],
    "2026_27": [
        "Coventry City",
        "Hull City",
        "Ipswich Town"
    ]
}


# Add season to the 2026/27 comparison
model_market_2026["season"] = "2026_27"


# Combine both seasons
combined_market_analysis = pd.concat(
    [
        model_market_2025,
        model_market_2026
    ],
    ignore_index=True
)


# Flag whether either team was newly promoted that season
def involves_promoted_team(row):

    promoted = promoted_by_season.get(
        row["season"],
        []
    )

    return (
        row["home_team"] in promoted
        or row["away_team"] in promoted
    )


combined_market_analysis["promoted_team_involved"] = (
    combined_market_analysis.apply(
        involves_promoted_team,
        axis=1
    )
)


# Focus on the 10%+ disagreements
combined_edge_10 = combined_market_analysis[
    combined_market_analysis["largest_positive_difference"] >= 0.10
].copy()


promoted_diagnostic = (
    combined_edge_10
    .groupby("promoted_team_involved")
    .agg(
        Selections=("edge_correct", "size"),
        Correct=("edge_correct", "sum"),
        Hit_Rate=("edge_correct", "mean"),
        Average_Difference=("largest_positive_difference", "mean"),
        Average_Odds=("edge_odds", "mean"),
        Total_Profit=("edge_profit", "sum")
    )
    .reset_index()
)


promoted_diagnostic["ROI"] = (
    promoted_diagnostic["Total_Profit"]
    / promoted_diagnostic["Selections"]
)


promoted_diagnostic["Group"] = (
    promoted_diagnostic["promoted_team_involved"]
    .map({
        True: "Promoted team involved",
        False: "No promoted team involved"
    })
)


promoted_diagnostic = promoted_diagnostic[
    [
        "Group",
        "Selections",
        "Correct",
        "Hit_Rate",
        "Average_Difference",
        "Average_Odds",
        "Total_Profit",
        "ROI"
    ]
]


display(promoted_diagnostic)

In [ ]:
# Use both 2025/26 and 2026/27
difference_analysis = combined_market_analysis[
    combined_market_analysis["largest_positive_difference"] >= 0.05
].copy()


# Create disagreement-size buckets
difference_analysis["difference_bucket"] = pd.cut(
    difference_analysis["largest_positive_difference"],
    bins=[
        0.05,
        0.10,
        0.15,
        0.20,
        np.inf
    ],
    labels=[
        "5-10%",
        "10-15%",
        "15-20%",
        "20%+"
    ],
    include_lowest=True,
    right=False
)


difference_bucket_results = (
    difference_analysis
    .groupby(
        "difference_bucket",
        observed=True
    )
    .agg(
        Selections=("edge_correct", "size"),
        Correct=("edge_correct", "sum"),
        Hit_Rate=("edge_correct", "mean"),
        Average_Difference=("largest_positive_difference", "mean"),
        Average_Odds=("edge_odds", "mean"),
        Total_Profit=("edge_profit", "sum")
    )
    .reset_index()
)


difference_bucket_results["ROI"] = (
    difference_bucket_results["Total_Profit"]
    / difference_bucket_results["Selections"]
)


print("MODEL-MARKET DIFFERENCE SIZE ANALYSIS")
print("-------------------------------------")

display(difference_bucket_results)

In [ ]:
# Save 2025/26 model-market comparison
model_market_2025.to_csv(
    PROCESSED_DIR / "model_market_comparison_2025_26.csv",
    index=False
)


# Save untouched 2026/27 comparison
model_market_2026.to_csv(
    PROCESSED_DIR / "model_market_comparison_2026_27.csv",
    index=False
)


# Save combined analysis
combined_market_analysis.to_csv(
    PROCESSED_DIR / "model_market_comparison_combined.csv",
    index=False
)


print("Saved:")

for filename in [
    "model_market_comparison_2025_26.csv",
    "model_market_comparison_2026_27.csv",
    "model_market_comparison_combined.csv"
]:
    path = PROCESSED_DIR / filename

    print(
        filename,
        "| exists:",
        path.exists(),
        "| rows:",
        len(pd.read_csv(path))
    )

In [ ]:
# STEP 1: Inspect our trained Random Forest models

import os
from sklearn.ensemble import RandomForestClassifier

print("Current directory:", os.getcwd())

trained_models = []

# list() creates a snapshot so globals() cannot change during iteration
for name, obj in list(globals().items()):
    if isinstance(obj, RandomForestClassifier):
        trained_models.append(name)

print("\nTrained Random Forest models:")
print(trained_models)

In [ ]:
# STEP 2: Inspect each Random Forest model

for name in trained_models:
    model = globals()[name]

    print(f"\n{name}")
    print("-" * 50)

    print("Type:", type(model).__name__)

    if hasattr(model, "n_features_in_"):
        print("Number of features:", model.n_features_in_)

    if hasattr(model, "feature_names_in_"):
        print("Features:")
        print(list(model.feature_names_in_))

    print("Parameters:")
    print({
        "n_estimators": getattr(model, "n_estimators", None),
        "max_depth": getattr(model, "max_depth", None),
        "min_samples_leaf": getattr(model, "min_samples_leaf", None),
        "max_features": getattr(model, "max_features", None)
    })

In [ ]:
# STEP 3: Compact summary of all Random Forest models

import pandas as pd

model_summary = []

for name in trained_models:
    model = globals()[name]

    features = (
        list(model.feature_names_in_)
        if hasattr(model, "feature_names_in_")
        else []
    )

    model_summary.append({
        "model": name,
        "n_features": getattr(model, "n_features_in_", None),
        "n_estimators": getattr(model, "n_estimators", None),
        "max_depth": getattr(model, "max_depth", None),
        "min_samples_leaf": getattr(model, "min_samples_leaf", None),
        "has_elo": any("elo" in f.lower() for f in features),
        "has_injury": any(
            word in f.lower()
            for f in features
            for word in ["injury", "absence", "missing", "unavailable"]
        ),
    })

model_summary_df = pd.DataFrame(model_summary)

display(model_summary_df)

In [ ]:
# Look for evaluation/result objects already stored in the notebook

possible_results = []

keywords = [
    "result",
    "score",
    "metric",
    "accuracy",
    "logloss",
    "log_loss",
    "evaluation",
    "performance",
    "backtest",
    "walk"
]

for name, obj in list(globals().items()):
    if any(word in name.lower() for word in keywords):
        possible_results.append({
            "name": name,
            "type": type(obj).__name__
        })

pd.DataFrame(possible_results)

In [ ]:
# STEP 5: Compare model performance

performance = {
    "Original RF": {
        "Accuracy": rf_accuracy,
        "Log Loss": rf_logloss
    },
    "RF V2": {
        "Accuracy": rf_accuracy_v2,
        "Log Loss": rf_logloss_v2
    },
    "RF Weighted": {
        "Accuracy": rf_accuracy_weighted,
        "Log Loss": rf_logloss_weighted
    },
    "RF V3": {
        "Accuracy": rf_accuracy_v3,
        "Log Loss": rf_logloss_v3
    },
    "RF V4": {
        "Accuracy": rf_accuracy_v4,
        "Log Loss": rf_logloss_v4
    },
    "RF V4b": {
        "Accuracy": rf_accuracy_v4b,
        "Log Loss": rf_logloss_v4b
    },
    "RF V4c": {
        "Accuracy": rf_accuracy_v4c,
        "Log Loss": rf_logloss_v4c
    },
    "Walk-Forward RF": {
        "Accuracy": backtest_accuracy,
        "Log Loss": backtest_logloss
    },
    "Calibrated Walk-Forward RF": {
        "Accuracy": cal_backtest_accuracy,
        "Log Loss": cal_backtest_logloss
    },
    "Bet365 Market": {
        "Accuracy": market_backtest_accuracy,
        "Log Loss": market_backtest_logloss
    }
}

performance_df = (
    pd.DataFrame(performance)
    .T
    .sort_values("Log Loss")
)

# Display without requiring jinja2
print(
    performance_df.round({
        "Accuracy": 4,
        "Log Loss": 4
    }).to_string()
)

In [ ]:
print("Base calibrated log loss:", round(calibrated_logloss, 4))
print("Expanded uncalibrated log loss:", round(expanded_uncalibrated_logloss, 4))
print("Expanded calibrated log loss:", round(expanded_calibrated_logloss, 4))

In [ ]:
# STEP 6: Find calibrated model objects

from sklearn.calibration import CalibratedClassifierCV

calibrated_models = []

for name, obj in list(globals().items()):
    if isinstance(obj, CalibratedClassifierCV):
        calibrated_models.append({
            "name": name,
            "type": type(obj).__name__,
            "n_features": getattr(obj, "n_features_in_", None)
        })

print(pd.DataFrame(calibrated_models).to_string(index=False))

In [ ]:
print("Base calibrated log loss:", calibrated_logloss)
print("Expanded calibrated log loss:", expanded_calibrated_logloss)

print("\nPossible calibrated model variables:")
for item in calibrated_models:
    print(item)

In [ ]:
# STEP 7: Inspect evaluation dataset sizes and periods

objects_to_check = [
    "y_test",
    "y_backtest",
    "y_cal_backtest",
    "test_df",
    "backtest_2526",
    "backtest_results_df",
    "calibrated_backtest_df"
]

for name in objects_to_check:
    if name in globals():
        obj = globals()[name]

        print(f"\n{name}")
        print("-" * 50)

        try:
            print("Length:", len(obj))
        except:
            pass

        if isinstance(obj, pd.DataFrame):
            print("Shape:", obj.shape)

            date_cols = [
                c for c in obj.columns
                if "date" in c.lower()
            ]

            for col in date_cols:
                try:
                    dates = pd.to_datetime(obj[col])
                    print(
                        f"{col}:",
                        dates.min(),
                        "to",
                        dates.max()
                    )
                except:
                    pass

        elif isinstance(obj, pd.Series):
            print("Index range:", obj.index.min(), "to", obj.index.max())

In [ ]:
# STEP 8: Save production model and metadata

from pathlib import Path
import joblib
import json
import pandas as pd

# Project folders
project_root = Path.cwd().parent
model_dir = project_root / "models"
model_dir.mkdir(exist_ok=True)

# Production model
production_model = calibrated_rf

# Exact feature order expected by the model
feature_names = list(production_model.feature_names_in_)

# Class order is CRITICAL when interpreting predict_proba()
class_order = [
    int(x) if hasattr(x, "item") else x
    for x in production_model.classes_
]

# Save model
joblib.dump(
    production_model,
    model_dir / "premier_league_model.joblib"
)

# Save feature schema
with open(model_dir / "feature_schema.json", "w") as f:
    json.dump(feature_names, f, indent=2)

# Save metadata
metadata = {
    "model_name": "Base Calibrated Random Forest",
    "model_type": type(production_model).__name__,
    "n_features": len(feature_names),
    "classes": class_order,
    "test_matches": 357,
    "test_start": "2025-08-15",
    "test_end": "2026-05-24",
    "log_loss": round(float(calibrated_logloss), 4),
    "market_log_loss": 1.0161,
    "purpose": "Premier League pre-match H/D/A probability prediction"
}

with open(model_dir / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Production model saved successfully.")
print("\nModel:", model_dir / "premier_league_model.joblib")
print("Features:", model_dir / "feature_schema.json")
print("Metadata:", model_dir / "model_metadata.json")

print("\nNumber of features:", len(feature_names))
print("Class order:", class_order)
print("Log loss:", calibrated_logloss)

In [ ]:
# STEP 9: Verify saved production model

loaded_model = joblib.load(
    model_dir / "premier_league_model.joblib"
)

with open(model_dir / "feature_schema.json", "r") as f:
    loaded_features = json.load(f)

with open(model_dir / "model_metadata.json", "r") as f:
    loaded_metadata = json.load(f)

print("Model loaded:", type(loaded_model).__name__)
print("Features loaded:", len(loaded_features))
print("Expected features:", loaded_model.n_features_in_)
print("Classes:", loaded_model.classes_)
print("\nMetadata:")
print(loaded_metadata)

assert len(loaded_features) == loaded_model.n_features_in_

print("\n✓ MODEL PACKAGE VERIFIED")

In [ ]:
# STEP 10: Inspect exact production feature schema

print(f"Production model expects {len(loaded_features)} features:\n")

for i, feature in enumerate(loaded_features, start=1):
    print(f"{i:02d}. {feature}")

In [ ]:
# Check what historical DataFrames are currently available

candidate_dfs = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        candidate_dfs.append({
            "name": name,
            "rows": len(obj),
            "columns": len(obj.columns)
        })

candidate_dfs = (
    pd.DataFrame(candidate_dfs)
    .sort_values("rows", ascending=False)
)

print(candidate_dfs.head(20).to_string(index=False))

In [ ]:
# STEP 11: Inspect the V4b modeling dataset

print("Shape:", model_ready_df_v4b.shape)

print("\nAll columns:")
for i, col in enumerate(model_ready_df_v4b.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nSample identifying information:")

id_cols = [
    col for col in [
        "date",
        "season",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "target"
    ]
    if col in model_ready_df_v4b.columns
]

print(model_ready_df_v4b[id_cols].tail(10).to_string(index=False))

In [ ]:
# STEP 12: Check whether all production features exist in V4b

missing_features = [
    feature
    for feature in loaded_features
    if feature not in model_ready_df_v4b.columns
]

print("Production features:", len(loaded_features))
print("Features found in V4b:", len(loaded_features) - len(missing_features))
print("Missing features:", missing_features)

In [ ]:
# STEP 13: Find historical match DataFrames we can use
# to generate features for future fixtures

candidates = []

required_raw_cols = [
    "date",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals"
]

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):

        cols = list(obj.columns)

        has_required = all(
            col in cols for col in required_raw_cols
        )

        if has_required:
            candidates.append({
                "name": name,
                "rows": len(obj),
                "columns": len(cols),
                "has_xg": any("xg" in c.lower() for c in cols),
                "has_shots": any("shot" in c.lower() for c in cols),
                "has_possession": any("possession" in c.lower() for c in cols),
                "has_passes": any("pass" in c.lower() for c in cols),
                "has_corners": any("corner" in c.lower() for c in cols),
            })

candidate_source_dfs = (
    pd.DataFrame(candidates)
    .sort_values("rows", ascending=False)
)

print(candidate_source_dfs.to_string(index=False))

In [ ]:
# STEP 14: Understand what is inside model_ready_df_v4b

cols = list(model_ready_df_v4b.columns)

production_set = set(loaded_features)

non_model_cols = [
    col for col in cols
    if col not in production_set
]

print("TOTAL COLUMNS:", len(cols))
print("MODEL FEATURES:", len(loaded_features))
print("OTHER COLUMNS:", len(non_model_cols))

print("\n--- OTHER / SOURCE COLUMNS ---")
for col in non_model_cols:
    print(col)

In [ ]:
# STEP 15: Inspect the latest matches available in V4b

preview_cols = [
    col for col in [
        "date",
        "season",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "target"
    ]
    if col in model_ready_df_v4b.columns
]

latest_matches = (
    model_ready_df_v4b
    .sort_values("date")
    [preview_cols]
    .tail(20)
)

print(latest_matches.to_string(index=False))

In [ ]:
print("calibrated_rf:", calibrated_rf)

print("\nRelevant dataset sizes:")
for name in [
    "X_train",
    "X_train_v2",
    "X_train_v4b",
    "X_rf_train",
    "training_pool",
    "training_pool_cal",
    "calibration_window_df",
    "test_df"
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {len(obj)}")

In [ ]:
# HISTORICAL DATA AUDIT
# Understand how many matches are available and how they are distributed

import pandas as pd

df = model_ready_df_v4b.copy()

df["date"] = pd.to_datetime(df["date"], utc=True)

print("=" * 55)
print("HISTORICAL DATA AUDIT")
print("=" * 55)

print("\n1. TOTAL MODEL-READY MATCHES")
print(len(df))

print("\n2. DATE RANGE")
print("Earliest:", df["date"].min())
print("Latest:", df["date"].max())

print("\n3. MATCHES BY SEASON")
print(
    df.groupby("season")
    .size()
    .to_string()
)

print("\n4. MATCHES BY YEAR")
print(
    df.groupby(df["date"].dt.year)
    .size()
    .to_string()
)

print("\n5. MISSING PRODUCTION FEATURES")

missing = (
    df[loaded_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing[missing > 0].to_string())

print("\n6. MATCHES WITH ALL 51 FEATURES AVAILABLE")

complete_matches = df[loaded_features].notna().all(axis=1)

print("Complete:", complete_matches.sum())
print("Incomplete:", (~complete_matches).sum())

In [ ]:
# Find every DataFrame that contains match dates and inspect its history

results = []

for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame):
        continue

    # Find likely date column
    date_col = None

    for candidate in ["date", "date_home", "match_date", "datetime"]:
        if candidate in obj.columns:
            date_col = candidate
            break

    if date_col is None:
        continue

    dates = pd.to_datetime(obj[date_col], errors="coerce", utc=True)

    if dates.notna().sum() == 0:
        continue

    results.append({
        "dataframe": name,
        "rows": len(obj),
        "date_column": date_col,
        "earliest": dates.min(),
        "latest": dates.max(),
        "has_season": "season" in obj.columns,
        "has_home_team": "home_team" in obj.columns,
        "has_xg": any("xg" in str(c).lower() for c in obj.columns),
    })

history_audit = (
    pd.DataFrame(results)
    .sort_values(["earliest", "rows"])
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)

print(history_audit.to_string(index=False))

In [ ]:
# AUDIT: Where did the 2,300 PL matches disappear?

base = model_df.copy()
rich = model_ready_df_v4b.copy()

base["date"] = pd.to_datetime(base["date"], utc=True)
rich["date"] = pd.to_datetime(rich["date"], utc=True)

print("=" * 65)
print("BASE MODEL_DF VS MODEL_READY_DF_V4B")
print("=" * 65)

print("\nMODEL_DF:")
print("Rows:", len(base))
print("Range:", base["date"].min(), "→", base["date"].max())

print("\nMODEL_READY_DF_V4B:")
print("Rows:", len(rich))
print("Range:", rich["date"].min(), "→", rich["date"].max())

print("\n--- MATCHES BY SEASON ---")

base_seasons = base.groupby("season").size().rename("model_df")
rich_seasons = rich.groupby("season").size().rename("model_ready_v4b")

season_compare = pd.concat(
    [base_seasons, rich_seasons],
    axis=1
).fillna(0).astype(int)

season_compare["lost"] = (
    season_compare["model_df"] -
    season_compare["model_ready_v4b"]
)

season_compare["retained_pct"] = (
    season_compare["model_ready_v4b"] /
    season_compare["model_df"]
    * 100
).round(1)

print(season_compare.to_string())

print("\n--- MODEL_DF COLUMNS ---")
for col in base.columns:
    print(col)

In [ ]:
# Find DataFrames containing RAW rich match statistics

rich_raw_cols = [
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "home_possession",
    "away_possession",
    "home_passes",
    "away_passes",
    "home_corners",
    "away_corners",
]

results = []

for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame):
        continue

    found = [c for c in rich_raw_cols if c in obj.columns]

    if len(found) >= 4:

        date_col = "date" if "date" in obj.columns else None

        if date_col:
            dates = pd.to_datetime(
                obj[date_col],
                errors="coerce",
                utc=True
            )

            earliest = dates.min()
            latest = dates.max()
        else:
            earliest = None
            latest = None

        results.append({
            "dataframe": name,
            "rows": len(obj),
            "columns": len(obj.columns),
            "rich_cols_found": len(found),
            "earliest": earliest,
            "latest": latest
        })

rich_source_audit = (
    pd.DataFrame(results)
    .sort_values(
        ["earliest", "rows"],
        ascending=[True, False]
    )
)

print(rich_source_audit.to_string(index=False))